In [1]:
# ## 무엇을 줄였나
# 기존 캐시에는 저자 ID만 있고 소속 국가가 없습니다. 국가는 어딘가에서 한 번은 받아야 하므로 "0회"는 불가능하고, 대신 아래 네 가지로 요청을 줄였습니다.
# | 이전 | 이번 |
# |---|---|
# | 전체 DOI 약 10만 건 수집 | **① 표본 점검(probe)**: 연도×저널 칸마다 10편만 받아 커버리지를 추정하고 분석 창을 먼저 정합니다. 약 100회 안팎(추정) |
# | 창과 무관하게 전체 수집 | **② 창 안만, 칸당 상한**: 분석 창 안의 논문만 받고, 연도×저널 칸마다 최대 100편만 받습니다. 표본 가중치로 보정합니다 |
# | DOI 없는 논문 전수 제목 검색(수천 회) | **③ 제목 검색 제거**: DOI 없는 논문은 unknown 칸으로 두고, 회수율은 100편만 시험(`fuzzy_probe`)합니다 |
# | 제한 없음 | **④ 요청 상한 `MAX_CALLS`**: 한 번 실행에 최대 300회에서 멈추고, 다시 실행하면 이어서 합니다 |
# 응답에서도 `authorships`와 `doi`만 요청해 데이터량을 줄였습니다.

# ## 설계상 알아 둘 점
# - **창이 먼저 정해지는 구조**입니다. 표본 점검 → 창 결정 → 창 안만 수집 순서입니다.
# - **DOI 없는 논문은 국가 미상**으로 센 보수적 보유율입니다. 구 연도 보유율이 낮게 나올 수 있고, 그 사실이 창 시작을 늦출 수 있습니다. 이 규칙은 결과를 보기 전에 정했다는 점을 CHANGELOG에 남기세요(코드가 첫 실행 때 자동으로 한 줄 기록합니다).
# - **칸당 상한(100편)을 두면 저자 단위 분석은 약해집니다.** 국가·소득군 몫(RQ4)은 가중 추정으로 충분하지만, 저자 단위로 이어 붙이는 분석이 필요하면 `FETCH_CAP_PER_CELL = None`으로 바꿔 전체를 받으세요. 상한이 있으면 집계할 때 `w × samp_w`를 써야 합니다.
# - 요청 수는 데이터를 못 봐서 **추정**입니다. `fetch`가 실행 전에 "받을 논문 수와 예상 요청 수"를 출력하니 확인하고 진행하세요.

In [ ]:
# OA_KEY = "HUq4YJ4SUrqcakXV20WhUL"

In [2]:
# -*- coding: utf-8 -*-
# =====================================================================================
# 논문 1 / S1. 국가 층 구축과 커버리지 감사 (단일 실행 파일 v3, OpenAlex 요청 최소화, Lens 미사용)
# 실행 순서(셀을 STAGES만 바꿔 여러 번 실행):
#   1) ["inspect", "probe", "audit"]            표본 점검 -> 분석 창 후보 결정
#   2) ["fetch", "build", "audit", "samples"]   창 안만 수집(칸당 상한) -> 구축 -> 감사 -> 표본 시트
#      (MAX_CALLS에 걸려 멈추면 같은 셀을 다시 실행: 이어서 수집)
#   3) 수작업(manual_verify.csv, unknown_check.csv 채움) 후 ["evaluate"]
#   선택: ["fuzzy_probe"]  DOI 없는 논문의 제목 매칭 회수율을 100편으로만 시험
# =====================================================================================
import re, json, math, time, hashlib, difflib, warnings
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, requests
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)

# ---------------- 설정 (여기만 수정) ----------------
OA_KEY = "HUq4"
MAILTO = "yj11021@korea.ac.kr"
STAGES = ["inspect", "probe", "audit"]
PROBE_PER_CELL = 10          # 표본 점검: 연도x저널 칸마다 받을 논문 수
FETCH_CAP_PER_CELL = 100     # 본 수집: 창 안 칸당 상한(None이면 전체 수집, 요청 수 크게 증가)
MAX_CALLS = 300              # 한 번 실행에서 보낼 최대 요청 수(초과 시 멈추고 재실행하면 이어서)
FALLBACK_START = 2010        # 창 후보가 없을 때 쓸 시작 연도
FUZZY_N = 100                # fuzzy_probe 표본 수
CFG = dict(seed=2026, cov_min=0.80, acc_min=0.95, wilson_low=0.90, manual_n=200, unknown_n=100,
           batch=50, fuzzy_accept=0.92, min_papers_venue_year=10, min_att_venue_year=3, max_consec_err=5)
# ----------------------------------------------------
assert OA_KEY and not OA_KEY.startswith("여기에"), "OA_KEY에 OpenAlex API 키를 붙여 넣으세요"
RNG = np.random.default_rng(CFG["seed"])
hdr = lambda t: print("\n" + "=" * 100 + f"\n{t}\n" + "=" * 100)
sha = lambda b: hashlib.sha256(b).hexdigest()
now = lambda: pd.Timestamp.now().isoformat(timespec="seconds")

# ---------------- 경로 ----------------
CANDS = [Path("../analysis_data/final"), Path("analysis_data/final"), Path("./final"),
         Path.home() / "Research/Minnesota_Transport/analysis_data/final"]
DATA = next(p for p in CANDS if (p / "F1_papers.csv").exists())
ROOT = DATA.resolve().parent.parent
S0 = ROOT / "paper1" / "00_S0"; S1 = ROOT / "paper1" / "01_S1"; RAW = S1 / "oa_raw"
RAW.mkdir(parents=True, exist_ok=True)
PREREG = json.loads((S0 / "PREREG.json").read_text(encoding="utf-8"))
FIXED = PREREG["event"]["fixed_venues"]

def changelog(line):
    cl = S1 / "CHANGELOG_S1.md"
    if not cl.exists(): cl.write_text("# CHANGELOG S1 (날짜 | 변경 | 사유 | 영향)\n\n", encoding="utf-8")
    with cl.open("a", encoding="utf-8") as f: f.write(line + "\n")
if not (S1 / "CHANGELOG_S1.md").exists():
    changelog(f"- {now()} | (1) Lens 교차 확인(계획서 1.6) 생략, 대체로 소속 있는데 국가 누락된 비율 수작업 100건 | 접근 곤란 | 검증 강도(단일 출처 의존을 한계로 명시)")
    changelog(f"- {now()} | (2) OpenAlex 요청 최소화: 표본 점검으로 창을 먼저 정하고 창 안만 칸당 상한({FETCH_CAP_PER_CELL})으로 수집, 표본 가중 보정. DOI 없는 논문 전수 제목 검색은 하지 않고 국가 미상으로 계산(보수적), 회수율은 {FUZZY_N}편 시험 | 요청량·사용량 절감 | 보유율 추정이 표본 기반, 구 연도 보유율은 낮게 나올 수 있음 | 결과 열람 전에 결정")

# ---------------- F1X ----------------
def nd(s):
    if not isinstance(s, str) or not s.strip(): return None
    s = s.strip().lower(); s = re.sub(r"^(https?://(dx\.)?doi\.org/|doi:\s*)", "", s); return s or None

F1 = pd.read_csv(DATA / "F1_papers.csv", low_memory=False); F1["pid"] = F1.pid.astype(str)
exf = S0 / "EXCLUDE_PIDS_FROZEN.csv"
log = pd.read_csv(S0 / "FREEZE_LOG.csv"); last = log[log.file == exf.name].iloc[-1]
assert sha(exf.read_bytes()) == last.sha256, "EXCLUDE_PIDS_FROZEN.csv 해시가 동결 기록과 다릅니다"
EXC = set(pd.read_csv(exf, dtype=str, encoding="utf-8-sig").pid)
F1X = F1[~F1.pid.isin(EXC)].copy(); F1X["doi_use"] = F1X.doi_norm.map(nd)
pf = S0 / "doi_fixes.csv"
if pf.exists():
    fx = pd.read_csv(pf, dtype=str, encoding="utf-8-sig"); m = dict(zip(fx.pid.astype(str), fx.doi_fixed.map(nd)))
    F1X["doi_use"] = [m.get(p, d) for p, d in zip(F1X.pid, F1X.doi_use)]
print(f"F1 {len(F1):,} -> F1X {len(F1X):,} | DOI 보유 {F1X.doi_use.notna().mean():.1%} | STAGES={STAGES}")
SEL = "doi,authorships"                      # 필요한 필드만 요청

def read_tbl(name):
    p = S1 / f"{name}.parquet"
    return pd.read_parquet(p) if p.exists() else pd.read_csv(S1 / f"{name}.csv.gz")
def save_tbl(df, name):
    try: df.to_parquet(S1 / f"{name}.parquet", index=False); return f"{name}.parquet"
    except Exception: df.to_csv(S1 / f"{name}.csv.gz", index=False, compression="gzip"); return f"{name}.csv.gz"

# ============================================================
# OpenAlex 호출 (요청 상한, 인증 오류·연속 실패 시 중단)
# ============================================================
class OAStop(Exception): pass
ERR = {"n": 0}; CALLS = {"n": 0}
def oa_get(url, params, tries=5):
    if CALLS["n"] >= MAX_CALLS: raise OAStop(f"이번 실행 요청 상한 MAX_CALLS={MAX_CALLS} 도달. 같은 셀을 다시 실행하면 이어서 수집합니다")
    CALLS["n"] += 1
    p = dict(params); p["mailto"] = MAILTO; p["api_key"] = OA_KEY; last = None
    for i in range(tries):
        try:
            r = requests.get(url, params=p, timeout=60); last = r.status_code
            if r.status_code == 200: ERR["n"] = 0; return r.json()
            if r.status_code in (401, 403): raise OAStop(f"인증 오류 {r.status_code}: API 키를 확인하세요")
            if r.status_code in (429, 500, 502, 503, 504): time.sleep(2 ** i); continue
            break
        except requests.RequestException: time.sleep(2 ** i)
    ERR["n"] += 1
    if ERR["n"] >= CFG["max_consec_err"]: raise OAStop(f"연속 {ERR['n']}회 실패(마지막 상태 {last}). 일일 사용량 소진 가능성: 다음 날 다시 실행하세요")
    return {"error": last}

def load_raw():
    W, p = {}, RAW / "works_by_doi.jsonl"
    if p.exists():
        for l in p.open(encoding="utf-8"):
            w = json.loads(l); W[nd(w.get("doi"))] = w
    mp = RAW / "missing_dois.txt"; miss = set(mp.read_text().split()) if mp.exists() else set()
    return W, miss

def fetch_dois(dois):
    """DOI 목록을 50개씩 묶어 조회. 기록은 raw/missing에 즉시 저장(재시작 가능). 상한·오류 시 False"""
    ps = set(d for d in dois if "," not in d and "|" not in d); plain = [d for d in dois if d in ps]; odd = [d for d in dois if d not in ps]
    nb = CFG["batch"]; t0 = time.time()
    try:
        with (RAW / "works_by_doi.jsonl").open("a", encoding="utf-8") as fo, (RAW / "missing_dois.txt").open("a") as fm:
            for i in range(0, len(plain), nb):
                b = plain[i:i + nb]
                j = oa_get("https://api.openalex.org/works", {"filter": "doi:" + "|".join(b), "per-page": nb, "select": SEL})
                if "error" in j: continue
                got = set()
                for w in j.get("results", []): fo.write(json.dumps(w, ensure_ascii=False) + "\n"); got.add(nd(w.get("doi")))
                for d in b:
                    if d not in got: fm.write(d + "\n")
                fo.flush(); fm.flush(); time.sleep(0.1)
                if (i // nb) % 20 == 0: print(f"  {i + len(b):,}/{len(plain):,} | 요청 {CALLS['n']} | {time.time() - t0:.0f}s")
            for d in odd:
                w = oa_get("https://api.openalex.org/works/doi:" + requests.utils.quote(d, safe="/"), {"select": SEL})
                if "error" not in w: fo.write(json.dumps(w, ensure_ascii=False) + "\n")
        return True
    except OAStop as e:
        print("⚠ 중단:", e); return False

# ============================================================
# 국가 규칙
# ============================================================
TERRITORY = {"HK", "MO", "TW", "PR", "GU", "VI", "AS", "MP", "GI", "FO", "GL", "AW", "CW", "SX", "NC", "PF", "RE", "GP", "MQ", "GF", "YT", "KY", "BM", "XK", "PS", "EH"}
FORMER = {"SU", "YU", "CS", "DD", "AN"}
def cn(c): return "ZZ_former" if c in FORMER else c
def countries_of(a):
    cs = {i.get("country_code") for i in (a.get("institutions") or []) if i.get("country_code")}
    if not cs: cs = set(a.get("countries") or [])
    return sorted(cn(c) for c in cs if c)

# ============================================================
# [inspect]
# ============================================================
def st_inspect():
    hdr("[inspect] 입력 점검")
    c = DATA.resolve().parent / "openalex" / "works_by_doi.jsonl"
    if c.exists():
        r = json.loads(c.open(encoding="utf-8").readline())
        print("기존 캐시 첫 레코드 키:", list(r.keys()))
        if "authorships" in r: print("→ 기존 캐시에 authorships가 있습니다. 소속 국가가 들어 있으면 알려 주세요(재수집 없이 가져오는 코드로 바꿀 수 있음)")
        else: print("→ 국가 필드 없음: 저자 ID만 있어 국가는 새로 받아야 합니다(요청 최소화 설정 적용)")
    d = F1X[F1X.doi_use.notna()]
    print(f"제외 pid {len(EXC)} | DOI 보유 논문 {len(d):,}")
    print(F1X.groupby(pd.cut(F1X.year, [1966, 1989, 1999, 2009, 2019, 2025])).agg(n=("pid", "size"), doi=("doi_use", lambda s: s.notna().mean())).round(3))
    cells = d[d.year >= 1995].groupby(["year", "venue_short"]).size()
    est = int(np.minimum(cells, PROBE_PER_CELL).sum())
    print(f"표본 점검(probe) 예상: 칸 {len(cells):,}개, 논문 약 {est:,}편, 요청 약 {math.ceil(est / CFG['batch'])}회 (이미 받은 것은 제외)")
    try:
        j = oa_get("https://api.openalex.org/works", {"filter": "doi:" + d.doi_use.iloc[0], "select": SEL})
        print("키 시험 호출:", "성공" if "results" in j else j)
        return "results" in j
    except OAStop as e: print("⚠", e); return False

# ============================================================
# [probe] 표본 점검: 연도 x 저널 칸마다 PROBE_PER_CELL편만
# ============================================================
def st_probe():
    hdr("[probe] 표본 수집")
    W, miss = load_raw(); att = set(W) | miss
    pool = F1X[F1X.doi_use.notna() & (F1X.year >= 1995)]
    samp = pool.groupby(["year", "venue_short"], group_keys=False).apply(lambda g: g.sample(min(len(g), PROBE_PER_CELL), random_state=CFG["seed"]))
    todo = sorted(set(samp.doi_use) - att)
    print(f"표본 {len(samp):,}편 중 새로 받을 것 {len(todo):,}편 (요청 약 {math.ceil(len(todo) / CFG['batch'])}회)")
    ok = fetch_dois(todo)
    if ok: print("표본 수집 완료 -> audit로 분석 창 후보를 정하세요")
    return ok

# ============================================================
# [audit] 커버리지(표본 기반 추정 가능), 분석 창 시작 연도
#   칸(연도x저널) 유효 보유율 = (DOI 보유 저자 비중) x (수집된 논문에서 국가가 있는 저자 비중)
#   DOI 없는 논문은 국가 미상으로 간주(보수적)
# ============================================================
def cov_cells():
    W, miss = load_raw(); att = set(W) | miss
    ks = {}
    for d, w in W.items():
        au = w.get("authorships") or []; ks[d] = (sum(1 for a in au if countries_of(a)) / len(au)) if au else 0.0
    D = F1X[["pid", "year", "venue_short", "doi_use", "n_authors"]].copy()
    D = D[D.year >= 1995]; D["has_doi"] = D.doi_use.notna(); D["att"] = D.doi_use.isin(att)
    D["kw"] = D.doi_use.map(ks).fillna(0.0) * D.n_authors
    k = ["year", "venue_short"]; A = D[D.att]
    T = pd.DataFrame({"n_papers": D.groupby(k).size(), "A_all": D.groupby(k).n_authors.sum(), "A_doi": D[D.has_doi].groupby(k).n_authors.sum(),
                      "n_att": A.groupby(k).size(), "A_att": A.groupby(k).n_authors.sum(), "kw_att": A.groupby(k).kw.sum()}).fillna(0).reset_index()
    T["cov_doi"] = np.where(T.A_att > 0, T.kw_att / T.A_att, np.nan)
    T["eff_cov"] = np.where(T.A_doi == 0, 0.0, (T.A_doi / T.A_all) * T.cov_doi)
    return T

def st_audit():
    hdr("[audit] 연도 x 저널 커버리지 (수집된 논문 기준 추정)")
    T = cov_cells(); T["fixed"] = T.venue_short.isin(FIXED); T.to_csv(S1 / "coverage_audit.csv", index=False, encoding="utf-8-sig")
    print(f"표본에 없는 칸(DOI 보유 논문은 있는데 수집 0): {int(((T.A_doi > 0) & (T.n_att == 0)).sum())}개 (보유율 계산에서 제외)")
    pooled = lambda t: (lambda t: (t.A_all * t.eff_cov).sum() / t.A_all.sum() if len(t) else np.nan)(t.dropna(subset=["eff_cov"]))
    cov = pd.DataFrame({"전체": T.groupby("year").apply(pooled), "고정저널(합산)": T[T.fixed].groupby("year").apply(pooled),
                        "고정저널 최소(저널-연도)": T[T.fixed & (T.n_papers >= CFG["min_papers_venue_year"]) & (T.n_att >= CFG["min_att_venue_year"])].groupby("year").eff_cov.min()})
    cov.to_csv(S1 / "coverage_by_year.csv", encoding="utf-8-sig"); print(cov.round(3).to_string())
    def start(rule):
        ok = rule.reindex(range(1995, 2026)) >= CFG["cov_min"]
        for y in range(1995, 2026):
            if ok.loc[y:].all(): return y
        return None
    S = {"합산(계획서 문언)": start(np.minimum(cov["전체"], cov["고정저널(합산)"])),
         "저널별 최소(엄격)": start(np.minimum(cov["전체"], cov["고정저널 최소(저널-연도)"].fillna(1)))}
    print("\n분석 창 시작 연도 후보:", S); main = S["합산(계획서 문언)"]
    (S1 / "WINDOW_DRAFT.json").write_text(json.dumps(dict(window_start=main, rules=S, threshold=CFG["cov_min"], at=now()), ensure_ascii=False, indent=2), encoding="utf-8")
    if main is None or main > 2010: print("⚠ 창이 2010년 이후로 줄거나 없음 -> 계획서 규칙: RQ4, RQ5를 부록으로 내리고 RQ1~RQ3 중심으로 전환")
    print("※ 표본 기반 추정입니다(칸 표본이 작으면 최소값이 흔들림). 어느 규칙이 주 규칙인지 결과를 보기 전에 CHANGELOG_S1.md에 기록하세요.")
    return True

def window_start():
    p = S1 / "WINDOW_DRAFT.json"
    ws = json.loads(p.read_text()).get("window_start") if p.exists() else None
    return ws or FALLBACK_START

# ============================================================
# [fetch] 창 안 논문만, 칸당 상한으로 수집 (이미 받은 논문 우선 포함)
# ============================================================
def st_fetch():
    hdr("[fetch] 본 수집 (창 안, 칸당 상한)")
    y0 = window_start(); W, miss = load_raw(); att = set(W) | miss
    pool = F1X[F1X.doi_use.notna() & (F1X.year >= y0)].copy(); pool["att"] = pool.doi_use.isin(att); pool["u"] = RNG.random(len(pool))
    if FETCH_CAP_PER_CELL:
        pool = pool.sort_values(["att", "u"], ascending=[False, True]).groupby(["year", "venue_short"]).head(FETCH_CAP_PER_CELL)
    todo = sorted(set(pool.doi_use) - att)
    print(f"창 시작 {y0} | 대상 논문 {len(pool):,}편 (이미 받음 {int(pool.att.sum()):,}) | 새로 받을 것 {len(todo):,}편 | 예상 요청 약 {math.ceil(len(todo) / CFG['batch'])}회 | 이번 실행 상한 {MAX_CALLS}회")
    ok = fetch_dois(todo)
    if ok: changelog(f"- {now()} | 본 수집 완료: 창 {y0}~, 칸당 상한 {FETCH_CAP_PER_CELL} | S1")
    return ok

# ============================================================
# [build] 저자 정렬 + 소속 처리 (수집된 논문만, 표본 가중치 samp_w 부여)
#   samp_w = (창 안 칸의 DOI 보유 논문 수) / (그 칸에서 수집된 논문 수). 집계는 w * samp_w
# ============================================================
def wb_income():
    p = S1 / "wb_income_current.csv"
    if not p.exists():
        j = requests.get("https://api.worldbank.org/v2/country", params={"format": "json", "per_page": 400}, timeout=60).json()[1]
        pd.DataFrame([dict(iso2=c["iso2Code"], name=c["name"], income=c["incomeLevel"]["id"], region=c["region"]["id"]) for c in j]).to_csv(p, index=False)
    Wb = pd.read_csv(p, keep_default_na=False)
    return Wb[(Wb.income != "INX") & (Wb.region != "NA")]

def st_build():
    hdr("[build] 저자 정렬과 국가 분배")
    y0 = window_start(); W, miss = load_raw(); att = set(W) | miss
    D = F1X[F1X.year >= y0].copy(); D["has_doi"] = D.doi_use.notna(); D["att"] = D.doi_use.isin(att)
    n_doi = D[D.has_doi].groupby(["year", "venue_short"]).size(); n_att = D[D.att].groupby(["year", "venue_short"]).size()
    cw = (n_doi / n_att).rename("samp_w"); D = D.join(cw, on=["year", "venue_short"])
    D["samp_w"] = np.where(D.has_doi, D.samp_w, 1.0)                    # DOI 없는 논문은 전부 unknown으로 포함(가중 1)
    D = D[D.att | ~D.has_doi]                                           # 수집된 논문 + DOI 없는 논문
    P, A, ST = [], [], []
    for r in D.itertuples():
        w = W.get(r.doi_use) if r.has_doi else None
        au = (w or {}).get("authorships") or []; n = len(au)
        if n == 0:
            P.append((r.pid, "UNK", 1.0)); ST.append((r.pid, w is not None, "doi" if w else "", 0, False, 0.0, r.samp_w)); continue
        cnt, known = Counter(), 0
        for k, a in enumerate(au):
            cs = countries_of(a); aff = " || ".join((a.get("raw_affiliation_strings") or [])[:2])[:300]
            A.append((r.pid, k, (a.get("author") or {}).get("id"), a.get("author_position"), "|".join(cs), aff, n == r.n_authors))
            if cs:
                known += 1
                for c in cs: cnt[c] += 1.0 / (n * len(cs))
            else: cnt["UNK"] += 1.0 / n
        P += [(r.pid, c, v) for c, v in cnt.items()]; ST.append((r.pid, True, "doi", n, n == r.n_authors, known / n, r.samp_w))
    P = pd.DataFrame(P, columns=["pid", "country", "w"])
    A = pd.DataFrame(A, columns=["pid", "rank", "oa_author_id", "author_position", "countries", "raw_aff", "aligned"])
    ST = pd.DataFrame(ST, columns=["pid", "matched", "how", "n_oa", "aligned", "known_share", "samp_w"])
    inc = wb_income().set_index("iso2").income
    P["is_territory"] = P.country.isin(TERRITORY); P["income_fixed"] = P.country.map(inc).fillna("UNK")
    P = P.merge(F1X[["pid", "year", "venue_short"]], on="pid").merge(ST[["pid", "samp_w"]], on="pid")
    hist = S1 / "wb_income_history.csv"
    if hist.exists():
        H = pd.read_csv(hist, keep_default_na=False); P = P.merge(H.rename(columns={"income": "income_tv"}), left_on=["country", "year"], right_on=["iso2", "year"], how="left").drop(columns="iso2")
    else: print("※ wb_income_history.csv 없음: 소득군은 현재 분류(고정)만 부착")
    print("저장:", save_tbl(P, "country_authorship"), save_tbl(A, "authorship_level")); ST.to_csv(S1 / "paper_status.csv", index=False)
    wm = np.average(ST.known_share, weights=ST.samp_w)
    print(f"창 {y0}~ | 수집·포함 논문 {len(ST):,} (DOI 없는 {int((~D.has_doi).sum()):,}) | 가중 평균 국가 보유율 {wm:.1%}")
    print("국가 몫 상위(w*samp_w, unknown 포함):"); ps = (P.w * P.samp_w).groupby(P.country).sum(); print((ps / ps.sum()).sort_values(ascending=False).head(10).round(4))
    changelog(f"- {now()} | build: 창 {y0}~, 표본 가중치 samp_w, 다중 소속 균등 분할, unknown 칸 유지 | S7")
    return True

# ============================================================
# [samples] 1.7 수작업 시트, 국가 누락 점검, 1.8 대상 렌즈, 1.9 저널 메타
# ============================================================
def strat_sample(D, n, keys):
    g = D.groupby(keys); size = g.size(); alloc = (size / size.sum() * n).round().clip(lower=1).astype(int)
    return pd.concat([x.sample(min(len(x), alloc[k]), random_state=CFG["seed"]) for k, x in g])

def st_samples():
    hdr("[samples] 검증 표본 생성")
    y0 = window_start()
    Au0 = read_tbl("authorship_level").merge(F1X[["pid", "year", "venue_short", "title", "doi_use"]], on="pid"); Au0 = Au0[Au0.year >= y0].copy()
    Au0["countries"] = Au0.countries.fillna(""); Au0["raw_aff"] = Au0.raw_aff.fillna("")
    Au = Au0[Au0.countries != ""].copy()
    Au["per"] = pd.cut(Au.year, [y0 - 1, 2004, 2014, 2025]).astype(str); Au["c1"] = Au.countries.str.split("|").str[0]
    top = Au.c1.value_counts().head(5).index; Au["cg"] = np.where(Au.c1.isin(top), Au.c1, "other")
    M = strat_sample(Au, CFG["manual_n"], ["per", "cg"]).sample(frac=1, random_state=CFG["seed"]).head(CFG["manual_n"])
    M = M.assign(true_country="", ok="")[["pid", "rank", "year", "venue_short", "doi_use", "title", "raw_aff", "countries", "true_country", "ok"]]
    M.to_csv(S1 / "manual_verify.csv", index=False, encoding="utf-8-sig")
    print(f"manual_verify.csv {len(M)}행: raw_aff와 countries를 원문과 대조해 ok에 1(맞음)/0(틀림), 틀리면 true_country도 기입")
    U = Au0[(Au0.countries == "") & (Au0.raw_aff != "")]
    print(f"창 안 authorship 중 소속 문자열은 있으나 국가 없음: {len(U):,} ({len(U) / max(len(Au0), 1):.1%})")
    if len(U):
        U = U.sample(min(CFG["unknown_n"], len(U)), random_state=CFG["seed"]).assign(true_country="", missed="")
        U[["pid", "rank", "year", "venue_short", "raw_aff", "true_country", "missed"]].to_csv(S1 / "unknown_check.csv", index=False, encoding="utf-8-sig")
        print("unknown_check.csv: raw_aff에서 국가를 판단할 수 있으면 missed=1(누락), 판단 불가면 0")
    try:
        import pycountry; names = {c.name: c.alpha_2 for c in pycountry.countries} | {getattr(c, "common_name", c.name): c.alpha_2 for c in pycountry.countries}
    except Exception: names = {}; print("※ pycountry 없음: pip install pycountry (없으면 대상 렌즈 생략)")
    amb = {"Georgia", "Turkey", "Jordan", "Chad", "Niger", "Guinea", "Mali", "Malta", "Cuba", "Peru", "Oman", "Togo", "Samoa"}
    gz = S1 / "gazetteer_cities.csv"
    if gz.exists(): names |= dict(pd.read_csv(gz, dtype=str, keep_default_na=False).values)
    names = {k: v for k, v in names.items() if k not in amb and len(k) >= 4}
    if names:
        rx = re.compile(r"\b(" + "|".join(sorted(map(re.escape, names), key=len, reverse=True)) + r")\b")
        hit = F1X.title.fillna("").map(lambda s: sorted({names[m] for m in rx.findall(s)}))
        TP = F1X[["pid", "year", "venue_short", "title"]].assign(target_countries=hit.map("|".join)); TP.to_csv(S1 / "title_places.csv", index=False, encoding="utf-8-sig")
        has = TP.target_countries != ""; print("대상 렌즈 보유율(5년 구간):", has.groupby(TP.year // 5 * 5).mean().round(3).to_dict())
        TP[has].sample(min(200, int(has.sum())), random_state=CFG["seed"]).assign(precision_ok="").to_csv(S1 / "target_lens_review.csv", index=False, encoding="utf-8-sig")
    ev = pd.read_csv(S0 / "EVENTS_FROZEN.csv")[["venue_short", "E1_corpus_entry", "E1_founding", "entry_ruling"]]
    vm = F1X.groupby("venue_short").agg(n_papers=("pid", "size")).reset_index().merge(ev, on="venue_short", how="left"); vm["fixed"] = vm.venue_short.isin(FIXED)
    vm.to_csv(S1 / "venue_meta.csv", index=False, encoding="utf-8-sig")
    pd.MultiIndex.from_product([sorted(F1X.venue_short.unique()), range(2000, 2026)], names=["venue_short", "year"]).to_frame(index=False).assign(jif="", source="").to_csv(S1 / "jif_template.csv", index=False)
    print("venue_meta.csv, jif_template.csv 저장 (JIF는 맥락 변수)")
    return True

# ============================================================
# [fuzzy_probe] DOI 없는 논문의 제목 매칭 회수율을 FUZZY_N편으로만 시험(결과는 분석에 반영하지 않음)
# ============================================================
def st_fuzzy_probe():
    hdr("[fuzzy_probe] 제목 매칭 회수율 시험")
    y0 = window_start(); q = F1X[F1X.doi_use.isna() & (F1X.year >= y0)]
    if len(q) == 0: print("창 안에 DOI 없는 논문이 없습니다"); return True
    q = q.sample(min(FUZZY_N, len(q)), random_state=CFG["seed"]); nm = lambda s: re.sub(r"[^a-z0-9]+", " ", str(s).lower()).strip(); rows = []
    try:
        for r in q.itertuples():
            title = re.sub(r"[,|:;]", " ", str(r.title))[:250]
            j = oa_get("https://api.openalex.org/works", {"filter": f"title.search:{title},publication_year:{int(r.year)}", "per-page": 5, "select": "display_name,doi"})
            sc = max([difflib.SequenceMatcher(None, nm(r.title), nm(w.get("display_name"))).ratio() for w in j.get("results", [])] + [0.0]) if "error" not in j else np.nan
            rows.append(dict(pid=r.pid, year=r.year, venue=r.venue_short, score=sc)); time.sleep(0.1)
    except OAStop as e: print("⚠ 중단:", e)
    F = pd.DataFrame(rows); F.to_csv(S1 / "fuzzy_probe.csv", index=False)
    if len(F): print(f"표본 {len(F)}편 | 회수(점수≥{CFG['fuzzy_accept']}) {(F.score >= CFG['fuzzy_accept']).mean():.1%} | 창 안 DOI 없는 논문 {len(q) if len(q) < FUZZY_N else int((F1X.doi_use.isna() & (F1X.year >= y0)).sum()):,}편")
    print("회수율이 높으면 그때 전수 매칭 여부를 결정하세요. 낮으면 DOI 없는 논문은 국가 미상으로 두는 것이 합리적입니다.")
    return True

# ============================================================
# [evaluate]
# ============================================================
def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n); h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)); return (c - h) / d, (c + h) / d

def st_evaluate():
    hdr("[evaluate] 통과 판정")
    R = []; ws = json.loads((S1 / "WINDOW_DRAFT.json").read_text())
    R.append(("1.5 분석 창 시작 연도(보유율 0.80 이상 유지, 2010 이하면 통과)", ws["window_start"], bool(ws["window_start"]) and ws["window_start"] <= 2010))
    mp = S1 / "manual_verify.csv"
    if mp.exists():
        Mv = pd.read_csv(mp, dtype=str, encoding="utf-8-sig").fillna(""); done = Mv[Mv.ok.isin(["0", "1"])]
        if len(done) >= 150:
            k, n = int((done.ok == "1").sum()), len(done); lo, hi = wilson(k, n)
            R.append(("1.7 수작업 정확도(점추정≥0.95, Wilson 하한≥0.90)", f"{k / n:.3f} [{lo:.3f}, {hi:.3f}] n={n}", k / n >= CFG["acc_min"] and lo >= CFG["wilson_low"]))
        else: R.append(("1.7 수작업 정확도", f"판정 {len(done)}건(<150): 더 채우세요", None))
    else: R.append(("1.7 수작업 정확도", "manual_verify.csv 없음", None))
    up = S1 / "unknown_check.csv"
    if up.exists():
        U = pd.read_csv(up, dtype=str, encoding="utf-8-sig").fillna(""); U = U[U.missed.isin(["0", "1"])]
        if len(U) >= 80:
            k, n = int((U.missed == "1").sum()), len(U); lo, hi = wilson(k, n)
            R.append(("1.6 대체: 소속 문자열 있는데 국가 누락된 비율(참고)", f"{k / n:.3f} [{lo:.3f}, {hi:.3f}] n={n}", "참고"))
        else: R.append(("1.6 대체: 국가 누락 점검", f"판정 {len(U)}건(<80)", None))
    else: R.append(("1.6 대체: 국가 누락 점검", "unknown_check.csv 없음", None))
    T = pd.DataFrame(R, columns=["검사", "값", "통과"]); print(T.to_string(index=False))
    fail = any(x is False for x in T.통과); skip = any(x is None for x in T.통과)
    msg = ("# S1 검증 보고서\n\n" + T.to_string(index=False) + f"\n\n작성 {now()}\n\n"
           + ("**일부 검사 미실시(SKIP)**: 완료로 간주하지 말 것.\n" if skip else "")
           + ("**FAIL 존재**: FALLBACK 적용(창을 2010년 이후로 축소 / 고정 저널만 / 비DOI는 부록). 창이 2010년 이후면 RQ4·RQ5를 부록으로, RQ1~RQ3 중심 논문으로 전환.\n" if fail else "")
           + "\n한계: 소속 국가는 국적이 아님. 구 자료는 소속 기재가 성김. 국가는 단일 출처(OpenAlex)에 의존(Lens 교차 확인 안 함). 수집은 칸당 표본이며 집계는 samp_w 가중, DOI 없는 논문은 국가 미상으로 계산.\n")
    (S1 / "validation_report.md").write_text(msg, encoding="utf-8"); print("저장: validation_report.md")
    return True

FUNCS = {"inspect": st_inspect, "probe": st_probe, "audit": st_audit, "fetch": st_fetch, "build": st_build,
         "samples": st_samples, "fuzzy_probe": st_fuzzy_probe, "evaluate": st_evaluate}
for s in STAGES:
    if FUNCS[s]() is False: break

F1 111,685 -> F1X 111,416 | DOI 보유 89.6% | STAGES=['inspect', 'probe', 'audit']

[inspect] 입력 점검
기존 캐시 첫 레코드 키: ['doi_norm', 'oa_id', 'oy', 'cby', 'subfield', 'field', 'authors', 'n_oa_auth']
→ 국가 필드 없음: 저자 ID만 있어 국가는 새로 받아야 합니다(요청 최소화 설정 적용)
제외 pid 269 | DOI 보유 논문 99,842
                  n    doi
year                      
(1966, 1989]   9778  0.229
(1989, 1999]   9844  0.621
(1999, 2009]  15372  0.986
(2009, 2019]  33615  0.998
(2019, 2025]  42807  1.000
표본 점검(probe) 예상: 칸 660개, 논문 약 6,568편, 요청 약 132회 (이미 받은 것은 제외)
키 시험 호출: 성공

[probe] 표본 수집
표본 6,568편 중 새로 받을 것 6,568편 (요청 약 132회)
  50/6,568 | 요청 2 | 1s
  1,050/6,568 | 요청 22 | 22s
  2,050/6,568 | 요청 42 | 43s
  3,050/6,568 | 요청 62 | 63s
  4,050/6,568 | 요청 82 | 83s
  5,050/6,568 | 요청 102 | 103s
  6,050/6,568 | 요청 122 | 124s
표본 수집 완료 -> audit로 분석 창 후보를 정하세요

[audit] 연도 x 저널 커버리지 (수집된 논문 기준 추정)
표본에 없는 칸(DOI 보유 논문은 있는데 수집 0): 0개 (보유율 계산에서 제외)
         전체  고정저널(합산)  고정저널 최소(저널-연도)
year                                 
1995  0.331     0.331

In [3]:
#

In [4]:
   T = pd.read_csv(S1/"coverage_audit.csv")
   x = T[(T.year>=2007)&T.fixed&(T.n_att>=3)&(T.eff_cov<0.80)].sort_values("eff_cov")
   print(x[["year","venue_short","n_papers","n_att","eff_cov"]].head(20).to_string(index=False))
   print(T[(T.year.between(1999,2002))&T.fixed].sort_values("eff_cov")[["year","venue_short","eff_cov"]].head(8))

 year venue_short  n_papers  n_att  eff_cov
 2009         TRR       786   10.0 0.701422
 2024         JTG       249   10.0 0.750000
 2008   TranspPol        32   10.0 0.765112
 2012         TRR       942   10.0 0.765661
 2024        TR-F       316   10.0 0.789474
 2019         TRR       835   10.0 0.793268
     year venue_short   eff_cov
77   2000   TranspRev  0.077112
62   1999   TranspRev  0.083333
92   2001   TranspRev  0.133333
107  2002   TranspRev  0.189076
74   2000         TRR  0.565557
63   2000         AAP  0.694834
72   2000        TR-E  0.727273
59   1999         TRR  0.741274


In [ ]:
# **판독: 걱정할 수준이 아닙니다. 2007년 창은 유지하고 fetch로 진행하세요.**
# ## 2007년 이후 낮은 칸
# - 0.80 미만인 칸은 6개뿐입니다. 고정 저널 14개 × 19년 = 약 266칸 중 2% 정도이고, 가장 낮은 값도 0.70입니다.
# - 6칸 모두 표본이 10편입니다(`n_att=10`). TRR은 연 800~900편 중 10편만 본 값이라 오차가 큽니다. 이 칸들의 낮은 값은 표본 잡음일 가능성이 높고, 체계적 결함이라는 증거는 아직 없습니다.
# - 엄격 규칙이 2025년으로 밀린 원인은 단 한 칸(JTG 2024, 0.750)입니다. 10편 표본의 한 칸이 창 시작을 18년 늦추는 규칙은 지나치게 민감하다는 뜻이므로, 이 규칙을 주 규칙으로 삼으면 안 됩니다.
# - 칸당 상한 100편으로 fetch한 뒤 audit를 다시 돌리면 TRR·JTG 값이 안정됩니다. 그때도 0.80 아래에 남는 칸만 실제 문제입니다.
# ## 1999~2002년의 TranspRev
# - TranspRev(Transport Reviews)의 국가 보유율이 0.08~0.19로 매우 낮습니다. 이 기간 논문의 소속 정보가 OpenAlex에 거의 없는 것으로 보입니다. 제 추측이고, 원인은 확인하지 못했습니다.
# - 분석 창이 2007년부터라서 현재 계획에는 영향이 없습니다. 다만 2000~2006년으로 창을 넓히는 시도를 할 때는 이 저널이 걸림돌이 됩니다.
# - TRR 2000년(0.566)과 AAP 2000년(0.695)도 같은 이유로 창 확장 시 문제가 됩니다.
# ## 기록할 것 (CHANGELOG_S1.md)
# - 주 규칙은 계획서 문언(합산 규칙, 2007년)이라고 적으세요. 엄격 규칙(2025년)은 한 칸에 좌우되는 표본 잡음 때문에 주 규칙으로 쓰지 않는다고 사유를 쓰되, 사후 판단이라는 점도 함께 적으세요.
# - fetch 후 audit를 다시 실행해 결과가 바뀌면 그 값도 그대로 기록하세요.
# - 창 안에서 0.80 미만으로 남는 칸 목록을 부록 표로 보고하세요. 심사자의 "저널별로는 어떤가"라는 질문에 바로 답할 수 있습니다.
# ## 다음 실행
# `STAGES = ["fetch", "build", "audit", "samples"]`를 실행하세요. fetch 후 audit에서 확인할 것은 두 가지입니다. 하나는 창 시작이 2007년으로 유지되는지, 다른 하나는 위 6칸이 0.80 이상으로 올라가는지입니다. 결과를 붙여 주시면 다음 판단을 같이 하겠습니다.

In [ ]:
# **아니요, S1은 아직 끝나지 않았습니다.** 지금까지 한 것은 S1의 앞부분(표본 점검과 창 후보 결정)뿐입니다.
# ## 완료한 것
# - 입력 점검(`inspect`): 제외 pid 해시 검증, DOI 보유율 확인, API 키 시험.
# - 표본 수집(`probe`): 연도×저널 칸당 10편, 6,568편.
# - 커버리지 감사(`audit`): 표본 기반 추정으로 분석 창 후보 2007년 도출(계획서 1.5의 초안).
# ## 남은 것
# | 계획서 항목 | 상태 | 할 일 |
# |---|---|---|
# | 1.1 본 수집 | 미실행 | `fetch` (창 2007~, 칸당 상한 100편). MAX_CALLS에 걸리면 같은 셀을 반복 실행 |
# | 1.2, 1.3 저자 정렬·국가 분배·소득군 | 미실행 | `build`. 정렬 일치/불일치 두 방식을 모두 보고하려면 결과 표에서 따로 집계해야 함 |
# | 1.3 시변 소득군 | 미실시 | `wb_income_history.csv`를 직접 만들어야 함. 없으면 현재 분류(고정)만 붙음 |
# | 1.5 본 수집 후 감사 재실행 | 미실행 | `audit`를 다시 돌려 창이 2007년으로 유지되는지, 낮은 6칸이 올라가는지 확인 |
# | 1.4 DOI 없는 논문 | 미실시 | `fuzzy_probe`(100편 회수율 시험)로 판단 |
# | 1.6 Lens 교차 확인 | **생략 결정** | CHANGELOG_S1에 사유 기록됨. 대체로 국가 누락 점검 100건 수행 |
# | 1.7 수작업 검증 200건 | 미실시 | `samples`로 시트 생성, 직접 판정 후 `evaluate` |
# | 1.8 대상 렌즈 | 미실시 | `samples`가 제목 지명 매칭을 만들고, 수작업 200편 정밀도 확인 필요 |
# | 1.9 저널 메타·JIF | 미실시 | 창간 연도 출판사 확인, JIF 수집(맥락 변수라 본 분석을 막지는 않음) |
# | 창 규칙 확정 기록 | 미기록 | 주 규칙(계획서 문언, 2007년)과 엄격 규칙은 민감도라는 사유를 CHANGELOG_S1에 기록 |
# | 통과 판정 | 미실시 | `evaluate`가 `validation_report.md` 작성 |
# ## 통과 여부는 아직 모릅니다
# 현재 확인된 것은 보유율 기준의 창 후보뿐이고, 수작업 정확도(점추정 0.95 이상, Wilson 하한 0.90 이상)는 아직 측정 전입니다. 이 값이 PASS여야 S1 통과이고 RQ4·RQ5를 본 논문에 둘 수 있습니다. 실패하면 계획서의 FALLBACK대로 갑니다.
# ## 다음 실행 순서
# 1. `STAGES = ["fetch", "build", "audit", "samples"]` 실행(fetch가 상한에 걸리면 반복).
# 2. 같은 때 `fuzzy_probe`도 한 번 실행.
# 3. 생성된 `manual_verify.csv`(200건)와 `unknown_check.csv`(100건)를 직접 채우기.
# 4. `STAGES = ["evaluate"]`로 판정하고, 결과를 붙여 주시면 S1 종료 여부를 같이 판단하겠습니다.
# 수작업(3번)이 가장 시간이 걸리는 구간이니, 1~2번을 돌려 두고 바로 시작하시는 것이 좋습니다.

In [2]:
# -*- coding: utf-8 -*-
# =====================================================================================
# 논문 1 / S1. 국가 층 구축과 커버리지 감사 (단일 실행 파일 v3.2, OpenAlex 무료 요금제 예산 기준, Lens 미사용)
# v3.1 -> v3.2 변경
#   (1) API 키 읽기 수정: 환경변수 OPENALEX_API_KEY 또는 실행 시 입력 (v3.1 사본의 os.environ.get("HU")는 항상 None이라 assert 실패)
#   (2) 무료 요금제($1/일 = 10,000크레딧) 기준 전수 수집: FETCH_CAP_PER_CELL=None, MAX_CALLS=2000
#   (3) 크레딧 장부: 요청마다 크레딧을 세고(필터 1, 제목 검색은 보수적으로 10), UTC 날짜별로 credit_ledger.csv에 누적.
#       하루 한도 DAILY_CREDIT_CAP(기본 8,000 = $0.80) 도달 시 중단 -> 다음 날(UTC 자정 = 한국 09:00 이후) 같은 셀 재실행
#       (장부는 이 스크립트의 요청만 센다. openalex.org 웹 탐색 사용분은 반영되지 않음)
#   (4) 메모리 절감: 읽을 때 필요한 필드만 남김(원본 jsonl은 그대로 보관)
#   (5) 실행 끝에 이번 실행 요청 수·크레딧·누적 출력
# 실행 순서
#   1회차  STAGES = ["fetch", "build", "audit", "samples"]
#          (fetch가 상한에 걸리면 뒤 단계는 실행되지 않음 -> 같은 셀을 다시 실행. 수집이 끝나야 build 이후 진행)
#   2회차  STAGES = ["fuzzy_probe"]   (요청 상한이 실행마다 초기화되므로 별도 실행)
#   수작업 manual_verify.csv, unknown_check.csv 채운 뒤
#   3회차  STAGES = ["evaluate"]
# =====================================================================================
import os, re, json, math, time, hashlib, difflib, getpass, warnings
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, requests
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)

# ---------------- 설정 (여기만 수정) ----------------
OA_KEY = os.environ.get("OPENALEX_API_KEY") or getpass.getpass("OpenAlex API key: ")
MAILTO = "yj11021@korea.ac.kr"
STAGES = ["fetch", "build", "audit", "samples"]
WINDOW_OVERRIDE = 2007       # 확정 분석 창 시작 연도. None이면 WINDOW_DRAFT.json 사용
PROBE_PER_CELL = 10
FETCH_CAP_PER_CELL = None    # None = 창 안 전수 수집(무료 예산으로 가능). 숫자면 칸당 상한
MAX_CALLS = 2000             # 한 번 실행에서 보낼 최대 요청 수
DAILY_CREDIT_CAP = 8000      # 하루 크레딧 한도(무료 플랜 10,000 중 웹 탐색·여유분 2,000을 남김)
CREDIT_FILTER = 1            # 필터 목록 요청 1크레딧
CREDIT_TITLE_SEARCH = 10     # 제목 검색은 검색(10크레딧)으로 과금될 수 있어 보수적으로 계산
FALLBACK_START = 2010
FUZZY_N = 100
CFG = dict(seed=2026, cov_min=0.80, acc_min=0.95, wilson_low=0.90, manual_n=200, unknown_n=100,
           batch=50, fuzzy_accept=0.92, min_papers_venue_year=10, min_att_venue_year=3, max_consec_err=5)
# ----------------------------------------------------
assert OA_KEY and len(OA_KEY.strip()) > 8, "OpenAlex API 키가 비어 있거나 너무 짧습니다"
OA_KEY = OA_KEY.strip()
RNG = np.random.default_rng(CFG["seed"])
hdr = lambda t: print("\n" + "=" * 100 + f"\n{t}\n" + "=" * 100)
sha = lambda b: hashlib.sha256(b).hexdigest()
now = lambda: pd.Timestamp.now().isoformat(timespec="seconds")
utc_today = lambda: pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%d")

# ---------------- 경로 ----------------
CANDS = [Path("../analysis_data/final"), Path("analysis_data/final"), Path("./final"),
         Path.home() / "Research/Minnesota_Transport/analysis_data/final"]
DATA = next(p for p in CANDS if (p / "F1_papers.csv").exists())
ROOT = DATA.resolve().parent.parent
S0 = ROOT / "paper1" / "00_S0"; S1 = ROOT / "paper1" / "01_S1"; RAW = S1 / "oa_raw"
RAW.mkdir(parents=True, exist_ok=True)
PREREG = json.loads((S0 / "PREREG.json").read_text(encoding="utf-8"))
FIXED = PREREG["event"]["fixed_venues"]

def changelog(line):
    cl = S1 / "CHANGELOG_S1.md"
    if not cl.exists(): cl.write_text("# CHANGELOG S1 (날짜 | 변경 | 사유 | 영향)\n\n", encoding="utf-8")
    with cl.open("a", encoding="utf-8") as f: f.write(line + "\n")
def changelog_once(key, line):
    cl = S1 / "CHANGELOG_S1.md"
    if cl.exists() and key in cl.read_text(encoding="utf-8"): return
    changelog(line)

changelog_once("(1) Lens", f"- {now()} | (1) Lens 교차 확인(계획서 1.6) 생략, 대체로 소속 있는데 국가 누락된 비율 수작업 100건 | 접근 곤란 | 검증 강도(단일 출처 의존을 한계로 명시)")
changelog_once("(2) OpenAlex", f"- {now()} | (2) OpenAlex 요청 최소화: 표본 점검으로 창을 먼저 정하고 창 안만 수집, DOI 없는 논문 전수 제목 검색은 하지 않고 국가 미상으로 계산(보수적), 회수율은 {FUZZY_N}편 시험 | 요청량·사용량 절감 | 결과 열람 전에 결정")
if WINDOW_OVERRIDE:
    changelog_once("(3) 분석 창", f"- {now()} | (3) 분석 창 시작 {WINDOW_OVERRIDE} 확정: 계획서 문언 규칙(전체·고정 저널 합산 보유율 0.80 이상 유지)을 probe 표본(칸당 10편) audit에 적용한 결과. 엄격 규칙(저널x연도 최솟값)은 2025가 나왔으나 JTG 2024 한 칸(n=10, 0.750)에 좌우되는 표본 잡음으로 판단해 민감도로만 보고. probe audit 결과를 본 뒤 내린 사후 판단임 | 분석 창 | S5.1(c), S7, S8. 본 수집 후 audit 재계산 값과 비교해 기록")
if FETCH_CAP_PER_CELL is None:
    changelog_once("(4) 전수 수집", f"- {now()} | (4) 칸당 상한 100 -> 없음(창 안 전수 수집), MAX_CALLS 2000, 하루 크레딧 한도 {DAILY_CREDIT_CAP} | 무료 요금제 일일 예산($1)으로 전수 수집이 가능함을 확인 | 표본 잡음 감소, samp_w가 사실상 1 | 본 수집 결과 열람 전에 결정")

# ---------------- F1X ----------------
def nd(s):
    if not isinstance(s, str) or not s.strip(): return None
    s = s.strip().lower(); s = re.sub(r"^(https?://(dx\.)?doi\.org/|doi:\s*)", "", s); return s or None

F1 = pd.read_csv(DATA / "F1_papers.csv", low_memory=False); F1["pid"] = F1.pid.astype(str)
exf = S0 / "EXCLUDE_PIDS_FROZEN.csv"
log = pd.read_csv(S0 / "FREEZE_LOG.csv"); last = log[log.file == exf.name].iloc[-1]
assert sha(exf.read_bytes()) == last.sha256, "EXCLUDE_PIDS_FROZEN.csv 해시가 동결 기록과 다릅니다"
EXC = set(pd.read_csv(exf, dtype=str, encoding="utf-8-sig").pid)
F1X = F1[~F1.pid.isin(EXC)].copy(); F1X["doi_use"] = F1X.doi_norm.map(nd)
pf = S0 / "doi_fixes.csv"
if pf.exists():
    fx = pd.read_csv(pf, dtype=str, encoding="utf-8-sig"); m = dict(zip(fx.pid.astype(str), fx.doi_fixed.map(nd)))
    F1X["doi_use"] = [m.get(p, d) for p, d in zip(F1X.pid, F1X.doi_use)]
print(f"F1 {len(F1):,} -> F1X {len(F1X):,} | DOI 보유 {F1X.doi_use.notna().mean():.1%} | STAGES={STAGES}")
SEL = "doi,authorships"

def read_tbl(name):
    p = S1 / f"{name}.parquet"
    return pd.read_parquet(p) if p.exists() else pd.read_csv(S1 / f"{name}.csv.gz")
def save_tbl(df, name):
    try: df.to_parquet(S1 / f"{name}.parquet", index=False); return f"{name}.parquet"
    except Exception: df.to_csv(S1 / f"{name}.csv.gz", index=False, compression="gzip"); return f"{name}.csv.gz"

def window_start():
    if WINDOW_OVERRIDE: return WINDOW_OVERRIDE
    p = S1 / "WINDOW_DRAFT.json"
    ws = json.loads(p.read_text()).get("window_start") if p.exists() else None
    return ws or FALLBACK_START

# ============================================================
# 크레딧 장부 (UTC 날짜별 누적)
# ============================================================
LEDGER = S1 / "credit_ledger.csv"
def ledger_today():
    if not LEDGER.exists(): return 0
    L = pd.read_csv(LEDGER); return int(L[L.date_utc == utc_today()].credits.sum())
USED_TODAY = ledger_today(); CREDITS = {"n": 0}
def ledger_write():
    if CREDITS["n"] <= 0: return
    new = not LEDGER.exists()
    with LEDGER.open("a", encoding="utf-8") as f:
        if new: f.write("date_utc,credits,calls,written_at\n")
        f.write(f"{utc_today()},{CREDITS['n']},{CALLS['n']},{now()}\n")
    CREDITS["n"] = 0

# ============================================================
# OpenAlex 호출 (요청 상한, 크레딧 상한, 인증 오류·연속 실패 시 중단)
# ============================================================
class OAStop(Exception): pass
ERR = {"n": 0}; CALLS = {"n": 0}
def oa_get(url, params, tries=5, cost=CREDIT_FILTER):
    if CALLS["n"] >= MAX_CALLS: raise OAStop(f"이번 실행 요청 상한 MAX_CALLS={MAX_CALLS} 도달. 같은 셀을 다시 실행하면 이어서 수집합니다")
    if USED_TODAY + CREDITS["n"] + cost > DAILY_CREDIT_CAP:
        raise OAStop(f"하루 크레딧 한도 {DAILY_CREDIT_CAP} 도달(오늘 UTC 누적 {USED_TODAY + CREDITS['n']}). 다음 날 UTC 자정(한국 09:00) 이후 같은 셀을 다시 실행하세요")
    CALLS["n"] += 1; CREDITS["n"] += cost
    p = dict(params); p["mailto"] = MAILTO; p["api_key"] = OA_KEY; last = None
    for i in range(tries):
        try:
            r = requests.get(url, params=p, timeout=60); last = r.status_code
            if r.status_code == 200: ERR["n"] = 0; return r.json()
            if r.status_code in (401, 403): raise OAStop(f"인증 오류 {r.status_code}: API 키를 확인하세요")
            if r.status_code == 429: raise OAStop("429: 일일 예산 소진 또는 속도 제한 가능성. 잠시 뒤 또는 다음 날 같은 셀을 다시 실행하세요")
            if r.status_code in (500, 502, 503, 504): time.sleep(2 ** i); continue
            break
        except requests.RequestException: time.sleep(2 ** i)
    ERR["n"] += 1
    if ERR["n"] >= CFG["max_consec_err"]: raise OAStop(f"연속 {ERR['n']}회 실패(마지막 상태 {last}). 잠시 뒤 다시 실행하세요")
    return {"error": last}

def slim(w):
    return {"doi": w.get("doi"),
            "authorships": [{"author": {"id": (a.get("author") or {}).get("id")}, "author_position": a.get("author_position"),
                             "institutions": [{"country_code": i.get("country_code")} for i in (a.get("institutions") or [])],
                             "countries": a.get("countries"), "raw_affiliation_strings": (a.get("raw_affiliation_strings") or [])[:2]}
                            for a in (w.get("authorships") or [])]}

def load_raw():
    W, p = {}, RAW / "works_by_doi.jsonl"
    if p.exists():
        for l in p.open(encoding="utf-8"):
            w = json.loads(l); W[nd(w.get("doi"))] = slim(w)
    mp = RAW / "missing_dois.txt"; miss = set(mp.read_text().split()) if mp.exists() else set()
    return W, miss

def fetch_dois(dois):
    """DOI 목록을 50개씩 묶어 조회. 기록은 raw/missing에 즉시 저장(재시작 가능). 상한·오류 시 False"""
    ps = set(d for d in dois if "," not in d and "|" not in d); plain = [d for d in dois if d in ps]; odd = [d for d in dois if d not in ps]
    nb = CFG["batch"]; t0 = time.time()
    try:
        with (RAW / "works_by_doi.jsonl").open("a", encoding="utf-8") as fo, (RAW / "missing_dois.txt").open("a") as fm:
            for i in range(0, len(plain), nb):
                b = plain[i:i + nb]
                j = oa_get("https://api.openalex.org/works", {"filter": "doi:" + "|".join(b), "per-page": nb, "select": SEL})
                if "error" in j: continue
                got = set()
                for w in j.get("results", []): fo.write(json.dumps(w, ensure_ascii=False) + "\n"); got.add(nd(w.get("doi")))
                for d in b:
                    if d not in got: fm.write(d + "\n")
                fo.flush(); fm.flush(); time.sleep(0.1)
                if (i // nb) % 20 == 0: print(f"  {i + len(b):,}/{len(plain):,} | 요청 {CALLS['n']} | 크레딧(오늘) {USED_TODAY + CREDITS['n']} | {time.time() - t0:.0f}s")
            for d in odd:
                w = oa_get("https://api.openalex.org/works/doi:" + requests.utils.quote(d, safe="/"), {"select": SEL}, cost=0)
                if "error" not in w: fo.write(json.dumps(w, ensure_ascii=False) + "\n")
        return True
    except OAStop as e:
        print("⚠ 중단:", e); return False

# ============================================================
# 국가 규칙
# ============================================================
TERRITORY = {"HK", "MO", "TW", "PR", "GU", "VI", "AS", "MP", "GI", "FO", "GL", "AW", "CW", "SX", "NC", "PF", "RE", "GP", "MQ", "GF", "YT", "KY", "BM", "XK", "PS", "EH"}
FORMER = {"SU", "YU", "CS", "DD", "AN"}
def cn(c): return "ZZ_former" if c in FORMER else c
def countries_of(a):
    cs = {i.get("country_code") for i in (a.get("institutions") or []) if i.get("country_code")}
    if not cs: cs = set(a.get("countries") or [])
    return sorted(cn(c) for c in cs if c)

# ============================================================
# [inspect] / [probe]  (이미 실행함. 필요 시 STAGES에 넣어 재실행 가능)
# ============================================================
def st_inspect():
    hdr("[inspect] 입력 점검")
    d = F1X[F1X.doi_use.notna()]
    print(f"제외 pid {len(EXC)} | DOI 보유 논문 {len(d):,} | 오늘(UTC) 누적 크레딧 {USED_TODAY}")
    try:
        j = oa_get("https://api.openalex.org/works", {"filter": "doi:" + d.doi_use.iloc[0], "select": SEL})
        print("키 시험 호출:", "성공" if "results" in j else j); return "results" in j
    except OAStop as e: print("⚠", e); return False

def st_probe():
    hdr("[probe] 표본 수집")
    W, miss = load_raw(); att = set(W) | miss
    pool = F1X[F1X.doi_use.notna() & (F1X.year >= 1995)]
    samp = pool.groupby(["year", "venue_short"], group_keys=False).apply(lambda g: g.sample(min(len(g), PROBE_PER_CELL), random_state=CFG["seed"]))
    todo = sorted(set(samp.doi_use) - att)
    print(f"표본 {len(samp):,}편 중 새로 받을 것 {len(todo):,}편")
    return fetch_dois(todo)

# ============================================================
# [audit]
#   칸(연도x저널) 유효 보유율 = (DOI 보유 저자 비중) x (수집된 논문에서 국가가 있는 저자 비중)
#   DOI 없는 논문은 국가 미상으로 간주(보수적)
# ============================================================
def cov_cells():
    W, miss = load_raw(); att = set(W) | miss
    ks = {}
    for d, w in W.items():
        au = w.get("authorships") or []; ks[d] = (sum(1 for a in au if countries_of(a)) / len(au)) if au else 0.0
    D = F1X[["pid", "year", "venue_short", "doi_use", "n_authors"]].copy()
    D = D[D.year >= 1995]; D["has_doi"] = D.doi_use.notna(); D["att"] = D.doi_use.isin(att)
    D["kw"] = D.doi_use.map(ks).fillna(0.0) * D.n_authors
    k = ["year", "venue_short"]; A = D[D.att]
    T = pd.DataFrame({"n_papers": D.groupby(k).size(), "A_all": D.groupby(k).n_authors.sum(), "A_doi": D[D.has_doi].groupby(k).n_authors.sum(),
                      "n_att": A.groupby(k).size(), "A_att": A.groupby(k).n_authors.sum(), "kw_att": A.groupby(k).kw.sum()}).fillna(0).reset_index()
    T["cov_doi"] = np.where(T.A_att > 0, T.kw_att / T.A_att, np.nan)
    T["eff_cov"] = np.where(T.A_doi == 0, 0.0, (T.A_doi / T.A_all) * T.cov_doi)
    return T

def st_audit():
    hdr("[audit] 연도 x 저널 커버리지 (수집된 논문 기준 추정)")
    T = cov_cells(); T["fixed"] = T.venue_short.isin(FIXED); T.to_csv(S1 / "coverage_audit.csv", index=False, encoding="utf-8-sig")
    print(f"표본에 없는 칸(DOI 보유 논문은 있는데 수집 0): {int(((T.A_doi > 0) & (T.n_att == 0)).sum())}개 (보유율 계산에서 제외)")
    pooled = lambda t: (lambda t: (t.A_all * t.eff_cov).sum() / t.A_all.sum() if len(t) else np.nan)(t.dropna(subset=["eff_cov"]))
    cov = pd.DataFrame({"전체": T.groupby("year").apply(pooled), "고정저널(합산)": T[T.fixed].groupby("year").apply(pooled),
                        "고정저널 최소(저널-연도)": T[T.fixed & (T.n_papers >= CFG["min_papers_venue_year"]) & (T.n_att >= CFG["min_att_venue_year"])].groupby("year").eff_cov.min()})
    cov.to_csv(S1 / "coverage_by_year.csv", encoding="utf-8-sig"); print(cov.round(3).to_string())
    def start(rule):
        ok = rule.reindex(range(1995, 2026)) >= CFG["cov_min"]
        for y in range(1995, 2026):
            if ok.loc[y:].all(): return y
        return None
    S = {"합산(계획서 문언)": start(np.minimum(cov["전체"], cov["고정저널(합산)"])),
         "저널별 최소(엄격)": start(np.minimum(cov["전체"], cov["고정저널 최소(저널-연도)"].fillna(1)))}
    main = S["합산(계획서 문언)"]; used = window_start()
    print("\n재계산한 창 시작 연도 후보:", S, "| 확정 창:", used)
    if main != used: print(f"⚠ 재계산 값({main})이 확정 창({used})과 다릅니다. 확정 창은 바꾸지 않았습니다. 값과 사유를 CHANGELOG_S1.md에 기록하고 민감도로 보고하세요")
    (S1 / "WINDOW_DRAFT.json").write_text(json.dumps(dict(window_start=main, window_used=used, rules=S, threshold=CFG["cov_min"], at=now()), ensure_ascii=False, indent=2), encoding="utf-8")
    low = T[(T.year >= used) & T.fixed & (T.n_att >= CFG["min_att_venue_year"]) & (T.eff_cov < CFG["cov_min"])].sort_values("eff_cov")
    low[["year", "venue_short", "n_papers", "n_att", "eff_cov"]].to_csv(S1 / "coverage_low_cells.csv", index=False, encoding="utf-8-sig")
    tot = len(T[(T.year >= used) & T.fixed & (T.n_att >= CFG["min_att_venue_year"])])
    print(f"\n창 안 고정 저널 칸 {tot}개 중 보유율 {CFG['cov_min']} 미만 {len(low)}개 -> coverage_low_cells.csv (부록 표로 보고)")
    if len(low): print(low[["year", "venue_short", "n_papers", "n_att", "eff_cov"]].head(15).to_string(index=False))
    w = T[T.year >= used]; print(f"창 안 저자 기준 DOI 없는 비중(국가 미상으로 계산됨): {1 - w.A_doi.sum() / w.A_all.sum():.1%}")
    if used > 2010: print("⚠ 창이 2010년 이후 -> 계획서 규칙: RQ4, RQ5를 부록으로 내리고 RQ1~RQ3 중심으로 전환")
    return True

# ============================================================
# [fetch] 창 안 논문 수집 (FETCH_CAP_PER_CELL=None이면 전수). 이미 받은 논문은 건너뜀
# ============================================================
def st_fetch():
    hdr("[fetch] 본 수집 (창 안)")
    y0 = window_start(); W, miss = load_raw(); att = set(W) | miss
    pool = F1X[F1X.doi_use.notna() & (F1X.year >= y0)].copy(); pool["att"] = pool.doi_use.isin(att); pool["u"] = RNG.random(len(pool))
    if FETCH_CAP_PER_CELL:
        pool = pool.sort_values(["att", "u"], ascending=[False, True]).groupby(["year", "venue_short"]).head(FETCH_CAP_PER_CELL)
    todo = sorted(set(pool.doi_use) - att); est = math.ceil(len(todo) / CFG["batch"])
    print(f"창 시작 {y0} | 대상 논문 {len(pool):,}편 (이미 받음 {int(pool.att.sum()):,}) | 새로 받을 것 {len(todo):,}편 | 예상 요청 약 {est}회(약 {est}크레딧) | "
          f"이번 실행 상한 {MAX_CALLS}회 | 오늘 누적 {USED_TODAY}/{DAILY_CREDIT_CAP}")
    ok = fetch_dois(todo)
    if ok: changelog_once("본 수집 완료", f"- {now()} | 본 수집 완료: 창 {y0}~, 칸당 상한 {FETCH_CAP_PER_CELL} | S1")
    else: print("→ 수집이 끝나지 않아 이후 단계는 실행하지 않았습니다. 같은 셀을 다시 실행하세요")
    return ok

# ============================================================
# [build] 저자 정렬 + 소속 처리 (수집된 논문만, 표본 가중치 samp_w 부여)
#   samp_w = (창 안 칸의 DOI 보유 논문 수) / (그 칸에서 수집된 논문 수). 전수 수집이면 1에 가까움. 집계는 w * samp_w
# ============================================================
def wb_income():
    p = S1 / "wb_income_current.csv"
    if not p.exists():
        j = requests.get("https://api.worldbank.org/v2/country", params={"format": "json", "per_page": 400}, timeout=60).json()[1]
        pd.DataFrame([dict(iso2=c["iso2Code"], name=c["name"], income=c["incomeLevel"]["id"], region=c["region"]["id"]) for c in j]).to_csv(p, index=False)
    Wb = pd.read_csv(p, keep_default_na=False)
    return Wb[(Wb.income != "INX") & (Wb.region != "NA")]

def st_build():
    hdr("[build] 저자 정렬과 국가 분배")
    y0 = window_start(); W, miss = load_raw(); att = set(W) | miss
    D = F1X[F1X.year >= y0].copy(); D["has_doi"] = D.doi_use.notna(); D["att"] = D.doi_use.isin(att)
    n_doi = D[D.has_doi].groupby(["year", "venue_short"]).size(); n_att = D[D.att].groupby(["year", "venue_short"]).size()
    cw = (n_doi / n_att).rename("samp_w"); D = D.join(cw, on=["year", "venue_short"])
    D["samp_w"] = np.where(D.has_doi, D.samp_w, 1.0)
    D = D[D.att | ~D.has_doi]
    P, A, ST = [], [], []
    for r in D.itertuples():
        w = W.get(r.doi_use) if r.has_doi else None
        au = (w or {}).get("authorships") or []; n = len(au)
        if n == 0:
            P.append((r.pid, "UNK", 1.0)); ST.append((r.pid, w is not None, "doi" if w else "", 0, False, 0.0, r.samp_w)); continue
        cnt, known = Counter(), 0
        for k, a in enumerate(au):
            cs = countries_of(a); aff = " || ".join((a.get("raw_affiliation_strings") or [])[:2])[:300]
            A.append((r.pid, k, (a.get("author") or {}).get("id"), a.get("author_position"), "|".join(cs), aff, n == r.n_authors))
            if cs:
                known += 1
                for c in cs: cnt[c] += 1.0 / (n * len(cs))
            else: cnt["UNK"] += 1.0 / n
        P += [(r.pid, c, v) for c, v in cnt.items()]; ST.append((r.pid, True, "doi", n, n == r.n_authors, known / n, r.samp_w))
    P = pd.DataFrame(P, columns=["pid", "country", "w"])
    A = pd.DataFrame(A, columns=["pid", "rank", "oa_author_id", "author_position", "countries", "raw_aff", "aligned"])
    ST = pd.DataFrame(ST, columns=["pid", "matched", "how", "n_oa", "aligned", "known_share", "samp_w"])
    inc = wb_income().set_index("iso2").income
    P["is_territory"] = P.country.isin(TERRITORY); P["income_fixed"] = P.country.map(inc).fillna("UNK")
    P = P.merge(F1X[["pid", "year", "venue_short"]], on="pid").merge(ST[["pid", "samp_w", "aligned"]], on="pid")
    hist = S1 / "wb_income_history.csv"
    if hist.exists():
        H = pd.read_csv(hist, keep_default_na=False); P = P.merge(H.rename(columns={"income": "income_tv"}), left_on=["country", "year"], right_on=["iso2", "year"], how="left").drop(columns="iso2")
    else: print("※ wb_income_history.csv 없음: 소득군은 현재 분류(고정)만 부착. 시변 분류 파일을 만든 뒤 build만 다시 실행하면 됩니다(API 호출 없음)")
    print("저장:", save_tbl(P, "country_authorship"), save_tbl(A, "authorship_level")); ST.to_csv(S1 / "paper_status.csv", index=False)
    wm = np.average(ST.known_share, weights=ST.samp_w); mt = ST[ST.n_oa > 0]
    print(f"창 {y0}~ | 수집·포함 논문 {len(ST):,} (DOI 없는 {int((~D.has_doi).sum()):,}) | 가중 평균 국가 보유율 {wm:.1%} | samp_w 범위 {ST.samp_w.min():.2f}~{ST.samp_w.max():.2f}")
    print(f"저자 수 정렬 일치(OpenAlex 저자 수 = F1 저자 수): {mt.aligned.mean():.1%} (수집 논문 {len(mt):,}편 중). 불일치 논문도 논문 단위 몫으로 포함됨, aligned 열로 민감도 확인 가능")
    print("국가 몫 상위(w*samp_w, unknown 포함):"); ps = (P.w * P.samp_w).groupby(P.country).sum(); print((ps / ps.sum()).sort_values(ascending=False).head(10).round(4))
    changelog_once("build: 창", f"- {now()} | build: 창 {y0}~, 표본 가중치 samp_w, 다중 소속 균등 분할, unknown 칸 유지, aligned 열 추가 | S7")
    return True

# ============================================================
# [samples] 1.7 수작업 시트, 국가 누락 점검, 1.8 대상 렌즈, 1.9 저널 메타
# ============================================================
def strat_sample(D, n, keys):
    g = D.groupby(keys); size = g.size(); alloc = (size / size.sum() * n).round().clip(lower=1).astype(int)
    return pd.concat([x.sample(min(len(x), alloc[k]), random_state=CFG["seed"]) for k, x in g])

def st_samples():
    hdr("[samples] 검증 표본 생성")
    y0 = window_start()
    Au0 = read_tbl("authorship_level").merge(F1X[["pid", "year", "venue_short", "title", "doi_use"]], on="pid"); Au0 = Au0[Au0.year >= y0].copy()
    Au0["countries"] = Au0.countries.fillna(""); Au0["raw_aff"] = Au0.raw_aff.fillna("")
    Au = Au0[Au0.countries != ""].copy()
    Au["per"] = pd.cut(Au.year, [y0 - 1, 2013, 2019, 2025]).astype(str); Au["c1"] = Au.countries.str.split("|").str[0]
    top = Au.c1.value_counts().head(5).index; Au["cg"] = np.where(Au.c1.isin(top), Au.c1, "other")
    M = strat_sample(Au, CFG["manual_n"], ["per", "cg"]).sample(frac=1, random_state=CFG["seed"]).head(CFG["manual_n"])
    M = M.assign(true_country="", ok="")[["pid", "rank", "year", "venue_short", "doi_use", "title", "raw_aff", "countries", "true_country", "ok"]]
    M.to_csv(S1 / "manual_verify.csv", index=False, encoding="utf-8-sig")
    print(f"manual_verify.csv {len(M)}행: raw_aff와 countries를 원문과 대조해 ok에 1(맞음)/0(틀림), 틀리면 true_country도 기입")
    U = Au0[(Au0.countries == "") & (Au0.raw_aff != "")]
    print(f"창 안 authorship 중 소속 문자열은 있으나 국가 없음: {len(U):,} ({len(U) / max(len(Au0), 1):.1%})")
    if len(U):
        U = U.sample(min(CFG["unknown_n"], len(U)), random_state=CFG["seed"]).assign(true_country="", missed="")
        U[["pid", "rank", "year", "venue_short", "raw_aff", "true_country", "missed"]].to_csv(S1 / "unknown_check.csv", index=False, encoding="utf-8-sig")
        print("unknown_check.csv: raw_aff에서 국가를 판단할 수 있으면 missed=1(누락), 판단 불가면 0")
    try:
        import pycountry; names = {c.name: c.alpha_2 for c in pycountry.countries} | {getattr(c, "common_name", c.name): c.alpha_2 for c in pycountry.countries}
    except Exception: names = {}; print("※ pycountry 없음: pip install pycountry (없으면 대상 렌즈 생략)")
    amb = {"Georgia", "Turkey", "Jordan", "Chad", "Niger", "Guinea", "Mali", "Malta", "Cuba", "Peru", "Oman", "Togo", "Samoa"}
    gz = S1 / "gazetteer_cities.csv"
    if gz.exists(): names |= dict(pd.read_csv(gz, dtype=str, keep_default_na=False).values)
    names = {k: v for k, v in names.items() if k not in amb and len(k) >= 4}
    if names:
        rx = re.compile(r"\b(" + "|".join(sorted(map(re.escape, names), key=len, reverse=True)) + r")\b")
        hit = F1X.title.fillna("").map(lambda s: sorted({names[m] for m in rx.findall(s)}))
        TP = F1X[["pid", "year", "venue_short", "title"]].assign(target_countries=hit.map("|".join)); TP.to_csv(S1 / "title_places.csv", index=False, encoding="utf-8-sig")
        has = TP.target_countries != ""; print("대상 렌즈 보유율(5년 구간):", has.groupby(TP.year // 5 * 5).mean().round(3).to_dict())
        TP[has].sample(min(200, int(has.sum())), random_state=CFG["seed"]).assign(precision_ok="").to_csv(S1 / "target_lens_review.csv", index=False, encoding="utf-8-sig")
    ev = pd.read_csv(S0 / "EVENTS_FROZEN.csv")[["venue_short", "E1_corpus_entry", "E1_founding", "entry_ruling"]]
    vm = F1X.groupby("venue_short").agg(n_papers=("pid", "size")).reset_index().merge(ev, on="venue_short", how="left"); vm["fixed"] = vm.venue_short.isin(FIXED)
    vm.to_csv(S1 / "venue_meta.csv", index=False, encoding="utf-8-sig")
    pd.MultiIndex.from_product([sorted(F1X.venue_short.unique()), range(2000, 2026)], names=["venue_short", "year"]).to_frame(index=False).assign(jif="", source="").to_csv(S1 / "jif_template.csv", index=False)
    print("venue_meta.csv, jif_template.csv 저장 (JIF는 맥락 변수)")
    return True

# ============================================================
# [fuzzy_probe] DOI 없는 논문의 제목 매칭 회수율을 FUZZY_N편으로만 시험(결과는 분석에 반영하지 않음)
#   제목 검색은 보수적으로 요청당 10크레딧으로 계산 (100편 = 1,000크레딧)
# ============================================================
def st_fuzzy_probe():
    hdr("[fuzzy_probe] 제목 매칭 회수율 시험")
    y0 = window_start(); q = F1X[F1X.doi_use.isna() & (F1X.year >= y0)]
    if len(q) == 0: print("창 안에 DOI 없는 논문이 없습니다"); return True
    n_all = len(q); q = q.sample(min(FUZZY_N, len(q)), random_state=CFG["seed"]); nm = lambda s: re.sub(r"[^a-z0-9]+", " ", str(s).lower()).strip(); rows = []
    try:
        for r in q.itertuples():
            title = re.sub(r"[,|:;]", " ", str(r.title))[:250]
            j = oa_get("https://api.openalex.org/works", {"filter": f"title.search:{title},publication_year:{int(r.year)}", "per-page": 5, "select": "display_name,doi"}, cost=CREDIT_TITLE_SEARCH)
            sc = max([difflib.SequenceMatcher(None, nm(r.title), nm(w.get("display_name"))).ratio() for w in j.get("results", [])] + [0.0]) if "error" not in j else np.nan
            rows.append(dict(pid=r.pid, year=r.year, venue=r.venue_short, score=sc)); time.sleep(0.1)
    except OAStop as e: print("⚠ 중단:", e)
    F = pd.DataFrame(rows); F.to_csv(S1 / "fuzzy_probe.csv", index=False)
    if len(F): print(f"표본 {len(F)}편 | 회수(점수≥{CFG['fuzzy_accept']}) {(F.score >= CFG['fuzzy_accept']).mean():.1%} | 창 안 DOI 없는 논문 {n_all:,}편")
    print("회수율이 높으면 그때 전수 매칭 여부를 결정하세요. 낮으면 DOI 없는 논문은 국가 미상으로 두는 것이 합리적입니다.")
    return True

# ============================================================
# [evaluate]
# ============================================================
def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n); h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)); return (c - h) / d, (c + h) / d

def st_evaluate():
    hdr("[evaluate] 통과 판정")
    R = []; ws = json.loads((S1 / "WINDOW_DRAFT.json").read_text()); used = window_start()
    R.append((f"1.5 분석 창 시작 연도(확정 {used}, 재계산 {ws['window_start']}; 2010 이하면 통과)", used, used <= 2010))
    mp = S1 / "manual_verify.csv"
    if mp.exists():
        Mv = pd.read_csv(mp, dtype=str, encoding="utf-8-sig").fillna(""); done = Mv[Mv.ok.isin(["0", "1"])]
        if len(done) >= 150:
            k, n = int((done.ok == "1").sum()), len(done); lo, hi = wilson(k, n)
            R.append(("1.7 수작업 정확도(점추정≥0.95, Wilson 하한≥0.90)", f"{k / n:.3f} [{lo:.3f}, {hi:.3f}] n={n}", k / n >= CFG["acc_min"] and lo >= CFG["wilson_low"]))
        else: R.append(("1.7 수작업 정확도", f"판정 {len(done)}건(<150): 더 채우세요", None))
    else: R.append(("1.7 수작업 정확도", "manual_verify.csv 없음", None))
    up = S1 / "unknown_check.csv"
    if up.exists():
        U = pd.read_csv(up, dtype=str, encoding="utf-8-sig").fillna(""); U = U[U.missed.isin(["0", "1"])]
        if len(U) >= 80:
            k, n = int((U.missed == "1").sum()), len(U); lo, hi = wilson(k, n)
            R.append(("1.6 대체: 소속 문자열 있는데 국가 누락된 비율(참고)", f"{k / n:.3f} [{lo:.3f}, {hi:.3f}] n={n}", "참고"))
        else: R.append(("1.6 대체: 국가 누락 점검", f"판정 {len(U)}건(<80)", None))
    else: R.append(("1.6 대체: 국가 누락 점검", "unknown_check.csv 없음", None))
    T = pd.DataFrame(R, columns=["검사", "값", "통과"]); print(T.to_string(index=False))
    fail = any(x is False for x in T.통과); skip = any(x is None for x in T.통과)
    msg = ("# S1 검증 보고서\n\n" + T.to_string(index=False) + f"\n\n작성 {now()}\n\n"
           + ("**일부 검사 미실시(SKIP)**: 완료로 간주하지 말 것.\n" if skip else "")
           + ("**FAIL 존재**: FALLBACK 적용(창을 2010년 이후로 축소 / 고정 저널만 / 비DOI는 부록). 창이 2010년 이후면 RQ4·RQ5를 부록으로, RQ1~RQ3 중심 논문으로 전환.\n" if fail else "")
           + "\n한계: 소속 국가는 국적이 아님. 구 자료는 소속 기재가 성김. 국가는 단일 출처(OpenAlex)에 의존(Lens 교차 확인 안 함). 창 안 DOI 보유 논문은 전수 수집(상한 설정 시 samp_w 가중), DOI 없는 논문은 국가 미상으로 계산.\n")
    (S1 / "validation_report.md").write_text(msg, encoding="utf-8"); print("저장: validation_report.md")
    return True

FUNCS = {"inspect": st_inspect, "probe": st_probe, "audit": st_audit, "fetch": st_fetch, "build": st_build,
         "samples": st_samples, "fuzzy_probe": st_fuzzy_probe, "evaluate": st_evaluate}
try:
    for s in STAGES:
        if FUNCS[s]() is False: break
finally:
    c, n = CREDITS["n"], CALLS["n"]; ledger_write()
    print(f"\n[이번 실행] 요청 {n}회, 추정 크레딧 {c} (약 ${c / 10000:.3f}) | 오늘(UTC) 누적 {ledger_today()}/{DAILY_CREDIT_CAP} (일일 총 10,000 = $1)")

OpenAlex API key:  ········


F1 111,685 -> F1X 111,416 | DOI 보유 89.6% | STAGES=['fetch', 'build', 'audit', 'samples']

[fetch] 본 수집 (창 안)
창 시작 2007 | 대상 논문 81,485편 (이미 받음 34,957) | 새로 받을 것 46,523편 | 예상 요청 약 931회(약 931크레딧) | 이번 실행 상한 2000회 | 오늘 누적 601/8000
  50/46,523 | 요청 1 | 크레딧(오늘) 602 | 1s
  1,050/46,523 | 요청 21 | 크레딧(오늘) 622 | 22s
  2,050/46,523 | 요청 41 | 크레딧(오늘) 642 | 43s
  3,050/46,523 | 요청 61 | 크레딧(오늘) 662 | 65s
  4,050/46,523 | 요청 81 | 크레딧(오늘) 682 | 86s
  5,050/46,523 | 요청 101 | 크레딧(오늘) 702 | 107s
  6,050/46,523 | 요청 121 | 크레딧(오늘) 722 | 132s
  7,050/46,523 | 요청 141 | 크레딧(오늘) 742 | 152s
  8,050/46,523 | 요청 161 | 크레딧(오늘) 762 | 173s
  9,050/46,523 | 요청 181 | 크레딧(오늘) 782 | 193s
  10,050/46,523 | 요청 201 | 크레딧(오늘) 802 | 213s
  11,050/46,523 | 요청 221 | 크레딧(오늘) 822 | 235s
  12,050/46,523 | 요청 241 | 크레딧(오늘) 842 | 257s
  13,050/46,523 | 요청 261 | 크레딧(오늘) 862 | 279s
  14,050/46,523 | 요청 281 | 크레딧(오늘) 882 | 302s
  15,050/46,523 | 요청 301 | 크레딧(오늘) 902 | 325s
  16,050/46,523 | 요청 321 | 크레딧(오늘) 922 | 348s
  17,050/46,523 

In [4]:
changelog(f"- {now()} | 본 수집 후 audit 재계산: 합산 규칙 2007(확정 창과 동일), 엄격 규칙 2009(TranspPol 2008 한 칸 0.800 경계) | 전수 수집으로 표본 잡음 제거 | 창은 2007 유지, 엄격 규칙은 민감도로 보고")
changelog(f"- {now()} | fuzzy_probe 생략: 창 안 DOI 없는 논문 160편(0.2%), 국가 미상으로 계산 | 영향 무시 가능 | 계획서 1.4 회수율 시험은 규모 때문에 생략")

In [ ]:
# **수집과 구축은 성공했고, S1의 자동 단계는 끝났습니다.** 남은 것은 수작업 검증과 판정입니다.
# ## 결과 판독
# - **수집:** 931회 요청, 추정 $0.093으로 예상(931회)과 정확히 일치했습니다. 오늘 누적은 1,532크레딧입니다.
# - **창 2007년 유지:** 계획서 규칙(합산)으로 다시 계산해도 2007년입니다. 고정 저널 266칸 중 0.80 미만은 TranspPol 2008 한 칸뿐이고, 값이 0.800에 가까워 사실상 경계선입니다.
# - **엄격 규칙은 2025에서 2009로 바뀌었습니다.** 표본 잡음이던 2025가 사라졌고, 이제 이 한 칸(논문 32편) 때문에 2009가 됩니다. 어느 쪽이든 2010년 이하라서 RQ4·RQ5를 부록으로 내리는 조건은 발동하지 않습니다.
# - **오래된 연도 행(1995~2006)은 본 수집 대상이 아니라 probe 표본 값 그대로입니다.** 해석하지 마세요.
# - **DOI 없는 논문은 창 안에 160편(0.2%)뿐입니다.** OpenAlex 매칭 실패도 약 44편으로 무시할 수준입니다.
# - **국가 미상(UNK)은 5.3%입니다.** 상위 몫은 US 25.5%, CN 21.7%, GB 4.3%, CA 4.1%, AU 3.9%이고, HK는 2.0%로 따로 잡힙니다.
# ## 짚을 점
# 1. **정렬 일치율 100%는 독립적인 점검이 아닙니다.** F1의 저자 수가 OpenAlex에서 온 것이라면 같은 출처끼리 비교한 것이라 당연히 100%가 나옵니다. 그렇다면 계획서 1.2의 "두 방식을 모두 보고"는 사실상 한 방식으로 합쳐집니다. 보고서에는 "저자 수가 같은 출처라 정렬 불일치가 없었다"고만 쓰세요(F1의 저자 수가 어디서 왔는지는 확인이 필요합니다).
# 2. **국가 미상의 대부분은 복구 가능성이 있습니다.** 소속 문자열은 있는데 국가가 없는 저자가 13,362명(4.4%)으로, UNK 5.3%의 대부분을 차지하는 것으로 보입니다. 소속 텍스트에 나라 이름이 들어 있는지는 `unknown_check.csv` 100건을 판정하면 비율이 나옵니다. 누락 비율이 높으면 규칙 기반 복구를 민감도 분석으로 추가하는 것을 검토할 수 있지만, 그 결정은 판정 결과를 본 뒤에 하세요.
# 3. **pycountry가 없어서 대상 렌즈(제목 지명)가 건너뛰어졌습니다.**
# ## 다음 순서
# 1. **`pip install pycountry` 후 `STAGES = ["samples"]`를 다시 실행하세요.** API 호출 없이 로컬에서만 돕니다. 단, `manual_verify.csv`와 `unknown_check.csv`가 덮어쓰이므로 수작업 입력을 시작하기 전에 하세요.
# 2. **수작업 판정:** `manual_verify.csv` 200건(ok에 1 또는 0), `unknown_check.csv` 100건(missed에 1 또는 0), 대상 렌즈 정밀도 200편입니다.
# 3. **`STAGES = ["evaluate"]`로 판정하세요.** 통과 기준은 정확도 점추정 0.95 이상, Wilson 하한 0.90 이상입니다.
# ## CHANGELOG_S1에 기록할 것
# ```python
# changelog(f"- {now()} | 본 수집 후 audit 재계산: 합산 규칙 2007(확정 창과 동일), 엄격 규칙 2009(TranspPol 2008 한 칸 0.800 경계) | 전수 수집으로 표본 잡음 제거 | 창은 2007 유지, 엄격 규칙은 민감도로 보고")
# changelog(f"- {now()} | fuzzy_probe 생략: 창 안 DOI 없는 논문 160편(0.2%), 국가 미상으로 계산 | 영향 무시 가능 | 계획서 1.4 회수율 시험은 규모 때문에 생략")
# ``

# 또 계획서 1.3의 영토 표기 규칙(HK, TW 등을 OpenAlex 코드대로 별도 표기하고 합산은 민감도로만 한다는 식의 내용)을 S7 결과를 보기 전에 문서로 남기세요. fuzzy_probe는 160편 규모라 실행하지 않아도 됩니다.
# 시변 소득군 파일(`wb_income_history.csv`), 창간 연도 확인, JIF 수집은 아직 남아 있습니다. 판정 결과를 붙여 주시면 S1 종료 여부를 같이 판단하겠습니다.

In [7]:
import shutil
from pathlib import Path
S1 = Path.home() / "Research/Minnesota_Transport/paper1/01_S1"
for f in ["manual_verify.csv", "unknown_check.csv"]:
    p = S1 / f
    if p.exists():
        shutil.copy(p, S1 / f.replace(".csv", "_backup.csv"))
        print("백업:", f)

백업: manual_verify.csv
백업: unknown_check.csv


In [8]:
import os, re, json, math, time, hashlib, difflib, getpass, warnings
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, requests
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)

# ---------------- 설정 (여기만 수정) ----------------
OA_KEY = os.environ.get("OPENALEX_API_KEY") or getpass.getpass("OpenAlex API key: ")
MAILTO = "yj11021@korea.ac.kr"
STAGES = ["samples"]
WINDOW_OVERRIDE = 2007       # 확정 분석 창 시작 연도. None이면 WINDOW_DRAFT.json 사용
PROBE_PER_CELL = 10
FETCH_CAP_PER_CELL = None    # None = 창 안 전수 수집(무료 예산으로 가능). 숫자면 칸당 상한
MAX_CALLS = 2000             # 한 번 실행에서 보낼 최대 요청 수
DAILY_CREDIT_CAP = 8000      # 하루 크레딧 한도(무료 플랜 10,000 중 웹 탐색·여유분 2,000을 남김)
CREDIT_FILTER = 1            # 필터 목록 요청 1크레딧
CREDIT_TITLE_SEARCH = 10     # 제목 검색은 검색(10크레딧)으로 과금될 수 있어 보수적으로 계산
FALLBACK_START = 2010
FUZZY_N = 100
CFG = dict(seed=2026, cov_min=0.80, acc_min=0.95, wilson_low=0.90, manual_n=200, unknown_n=100,
           batch=50, fuzzy_accept=0.92, min_papers_venue_year=10, min_att_venue_year=3, max_consec_err=5)
# ----------------------------------------------------
assert OA_KEY and len(OA_KEY.strip()) > 8, "OpenAlex API 키가 비어 있거나 너무 짧습니다"
OA_KEY = OA_KEY.strip()
RNG = np.random.default_rng(CFG["seed"])
hdr = lambda t: print("\n" + "=" * 100 + f"\n{t}\n" + "=" * 100)
sha = lambda b: hashlib.sha256(b).hexdigest()
now = lambda: pd.Timestamp.now().isoformat(timespec="seconds")
utc_today = lambda: pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%d")

# ---------------- 경로 ----------------
CANDS = [Path("../analysis_data/final"), Path("analysis_data/final"), Path("./final"),
         Path.home() / "Research/Minnesota_Transport/analysis_data/final"]
DATA = next(p for p in CANDS if (p / "F1_papers.csv").exists())
ROOT = DATA.resolve().parent.parent
S0 = ROOT / "paper1" / "00_S0"; S1 = ROOT / "paper1" / "01_S1"; RAW = S1 / "oa_raw"
RAW.mkdir(parents=True, exist_ok=True)
PREREG = json.loads((S0 / "PREREG.json").read_text(encoding="utf-8"))
FIXED = PREREG["event"]["fixed_venues"]

def changelog(line):
    cl = S1 / "CHANGELOG_S1.md"
    if not cl.exists(): cl.write_text("# CHANGELOG S1 (날짜 | 변경 | 사유 | 영향)\n\n", encoding="utf-8")
    with cl.open("a", encoding="utf-8") as f: f.write(line + "\n")
def changelog_once(key, line):
    cl = S1 / "CHANGELOG_S1.md"
    if cl.exists() and key in cl.read_text(encoding="utf-8"): return
    changelog(line)

changelog_once("(1) Lens", f"- {now()} | (1) Lens 교차 확인(계획서 1.6) 생략, 대체로 소속 있는데 국가 누락된 비율 수작업 100건 | 접근 곤란 | 검증 강도(단일 출처 의존을 한계로 명시)")
changelog_once("(2) OpenAlex", f"- {now()} | (2) OpenAlex 요청 최소화: 표본 점검으로 창을 먼저 정하고 창 안만 수집, DOI 없는 논문 전수 제목 검색은 하지 않고 국가 미상으로 계산(보수적), 회수율은 {FUZZY_N}편 시험 | 요청량·사용량 절감 | 결과 열람 전에 결정")
if WINDOW_OVERRIDE:
    changelog_once("(3) 분석 창", f"- {now()} | (3) 분석 창 시작 {WINDOW_OVERRIDE} 확정: 계획서 문언 규칙(전체·고정 저널 합산 보유율 0.80 이상 유지)을 probe 표본(칸당 10편) audit에 적용한 결과. 엄격 규칙(저널x연도 최솟값)은 2025가 나왔으나 JTG 2024 한 칸(n=10, 0.750)에 좌우되는 표본 잡음으로 판단해 민감도로만 보고. probe audit 결과를 본 뒤 내린 사후 판단임 | 분석 창 | S5.1(c), S7, S8. 본 수집 후 audit 재계산 값과 비교해 기록")
if FETCH_CAP_PER_CELL is None:
    changelog_once("(4) 전수 수집", f"- {now()} | (4) 칸당 상한 100 -> 없음(창 안 전수 수집), MAX_CALLS 2000, 하루 크레딧 한도 {DAILY_CREDIT_CAP} | 무료 요금제 일일 예산($1)으로 전수 수집이 가능함을 확인 | 표본 잡음 감소, samp_w가 사실상 1 | 본 수집 결과 열람 전에 결정")

# ---------------- F1X ----------------
def nd(s):
    if not isinstance(s, str) or not s.strip(): return None
    s = s.strip().lower(); s = re.sub(r"^(https?://(dx\.)?doi\.org/|doi:\s*)", "", s); return s or None

F1 = pd.read_csv(DATA / "F1_papers.csv", low_memory=False); F1["pid"] = F1.pid.astype(str)
exf = S0 / "EXCLUDE_PIDS_FROZEN.csv"
log = pd.read_csv(S0 / "FREEZE_LOG.csv"); last = log[log.file == exf.name].iloc[-1]
assert sha(exf.read_bytes()) == last.sha256, "EXCLUDE_PIDS_FROZEN.csv 해시가 동결 기록과 다릅니다"
EXC = set(pd.read_csv(exf, dtype=str, encoding="utf-8-sig").pid)
F1X = F1[~F1.pid.isin(EXC)].copy(); F1X["doi_use"] = F1X.doi_norm.map(nd)
pf = S0 / "doi_fixes.csv"
if pf.exists():
    fx = pd.read_csv(pf, dtype=str, encoding="utf-8-sig"); m = dict(zip(fx.pid.astype(str), fx.doi_fixed.map(nd)))
    F1X["doi_use"] = [m.get(p, d) for p, d in zip(F1X.pid, F1X.doi_use)]
print(f"F1 {len(F1):,} -> F1X {len(F1X):,} | DOI 보유 {F1X.doi_use.notna().mean():.1%} | STAGES={STAGES}")
SEL = "doi,authorships"

def read_tbl(name):
    p = S1 / f"{name}.parquet"
    return pd.read_parquet(p) if p.exists() else pd.read_csv(S1 / f"{name}.csv.gz")
def save_tbl(df, name):
    try: df.to_parquet(S1 / f"{name}.parquet", index=False); return f"{name}.parquet"
    except Exception: df.to_csv(S1 / f"{name}.csv.gz", index=False, compression="gzip"); return f"{name}.csv.gz"

def window_start():
    if WINDOW_OVERRIDE: return WINDOW_OVERRIDE
    p = S1 / "WINDOW_DRAFT.json"
    ws = json.loads(p.read_text()).get("window_start") if p.exists() else None
    return ws or FALLBACK_START

# ============================================================
# 크레딧 장부 (UTC 날짜별 누적)
# ============================================================
LEDGER = S1 / "credit_ledger.csv"
def ledger_today():
    if not LEDGER.exists(): return 0
    L = pd.read_csv(LEDGER); return int(L[L.date_utc == utc_today()].credits.sum())
USED_TODAY = ledger_today(); CREDITS = {"n": 0}
def ledger_write():
    if CREDITS["n"] <= 0: return
    new = not LEDGER.exists()
    with LEDGER.open("a", encoding="utf-8") as f:
        if new: f.write("date_utc,credits,calls,written_at\n")
        f.write(f"{utc_today()},{CREDITS['n']},{CALLS['n']},{now()}\n")
    CREDITS["n"] = 0

# ============================================================
# OpenAlex 호출 (요청 상한, 크레딧 상한, 인증 오류·연속 실패 시 중단)
# ============================================================
class OAStop(Exception): pass
ERR = {"n": 0}; CALLS = {"n": 0}
def oa_get(url, params, tries=5, cost=CREDIT_FILTER):
    if CALLS["n"] >= MAX_CALLS: raise OAStop(f"이번 실행 요청 상한 MAX_CALLS={MAX_CALLS} 도달. 같은 셀을 다시 실행하면 이어서 수집합니다")
    if USED_TODAY + CREDITS["n"] + cost > DAILY_CREDIT_CAP:
        raise OAStop(f"하루 크레딧 한도 {DAILY_CREDIT_CAP} 도달(오늘 UTC 누적 {USED_TODAY + CREDITS['n']}). 다음 날 UTC 자정(한국 09:00) 이후 같은 셀을 다시 실행하세요")
    CALLS["n"] += 1; CREDITS["n"] += cost
    p = dict(params); p["mailto"] = MAILTO; p["api_key"] = OA_KEY; last = None
    for i in range(tries):
        try:
            r = requests.get(url, params=p, timeout=60); last = r.status_code
            if r.status_code == 200: ERR["n"] = 0; return r.json()
            if r.status_code in (401, 403): raise OAStop(f"인증 오류 {r.status_code}: API 키를 확인하세요")
            if r.status_code == 429: raise OAStop("429: 일일 예산 소진 또는 속도 제한 가능성. 잠시 뒤 또는 다음 날 같은 셀을 다시 실행하세요")
            if r.status_code in (500, 502, 503, 504): time.sleep(2 ** i); continue
            break
        except requests.RequestException: time.sleep(2 ** i)
    ERR["n"] += 1
    if ERR["n"] >= CFG["max_consec_err"]: raise OAStop(f"연속 {ERR['n']}회 실패(마지막 상태 {last}). 잠시 뒤 다시 실행하세요")
    return {"error": last}

def slim(w):
    return {"doi": w.get("doi"),
            "authorships": [{"author": {"id": (a.get("author") or {}).get("id")}, "author_position": a.get("author_position"),
                             "institutions": [{"country_code": i.get("country_code")} for i in (a.get("institutions") or [])],
                             "countries": a.get("countries"), "raw_affiliation_strings": (a.get("raw_affiliation_strings") or [])[:2]}
                            for a in (w.get("authorships") or [])]}

def load_raw():
    W, p = {}, RAW / "works_by_doi.jsonl"
    if p.exists():
        for l in p.open(encoding="utf-8"):
            w = json.loads(l); W[nd(w.get("doi"))] = slim(w)
    mp = RAW / "missing_dois.txt"; miss = set(mp.read_text().split()) if mp.exists() else set()
    return W, miss

def fetch_dois(dois):
    """DOI 목록을 50개씩 묶어 조회. 기록은 raw/missing에 즉시 저장(재시작 가능). 상한·오류 시 False"""
    ps = set(d for d in dois if "," not in d and "|" not in d); plain = [d for d in dois if d in ps]; odd = [d for d in dois if d not in ps]
    nb = CFG["batch"]; t0 = time.time()
    try:
        with (RAW / "works_by_doi.jsonl").open("a", encoding="utf-8") as fo, (RAW / "missing_dois.txt").open("a") as fm:
            for i in range(0, len(plain), nb):
                b = plain[i:i + nb]
                j = oa_get("https://api.openalex.org/works", {"filter": "doi:" + "|".join(b), "per-page": nb, "select": SEL})
                if "error" in j: continue
                got = set()
                for w in j.get("results", []): fo.write(json.dumps(w, ensure_ascii=False) + "\n"); got.add(nd(w.get("doi")))
                for d in b:
                    if d not in got: fm.write(d + "\n")
                fo.flush(); fm.flush(); time.sleep(0.1)
                if (i // nb) % 20 == 0: print(f"  {i + len(b):,}/{len(plain):,} | 요청 {CALLS['n']} | 크레딧(오늘) {USED_TODAY + CREDITS['n']} | {time.time() - t0:.0f}s")
            for d in odd:
                w = oa_get("https://api.openalex.org/works/doi:" + requests.utils.quote(d, safe="/"), {"select": SEL}, cost=0)
                if "error" not in w: fo.write(json.dumps(w, ensure_ascii=False) + "\n")
        return True
    except OAStop as e:
        print("⚠ 중단:", e); return False

# ============================================================
# 국가 규칙
# ============================================================
TERRITORY = {"HK", "MO", "TW", "PR", "GU", "VI", "AS", "MP", "GI", "FO", "GL", "AW", "CW", "SX", "NC", "PF", "RE", "GP", "MQ", "GF", "YT", "KY", "BM", "XK", "PS", "EH"}
FORMER = {"SU", "YU", "CS", "DD", "AN"}
def cn(c): return "ZZ_former" if c in FORMER else c
def countries_of(a):
    cs = {i.get("country_code") for i in (a.get("institutions") or []) if i.get("country_code")}
    if not cs: cs = set(a.get("countries") or [])
    return sorted(cn(c) for c in cs if c)

# ============================================================
# [inspect] / [probe]  (이미 실행함. 필요 시 STAGES에 넣어 재실행 가능)
# ============================================================
def st_inspect():
    hdr("[inspect] 입력 점검")
    d = F1X[F1X.doi_use.notna()]
    print(f"제외 pid {len(EXC)} | DOI 보유 논문 {len(d):,} | 오늘(UTC) 누적 크레딧 {USED_TODAY}")
    try:
        j = oa_get("https://api.openalex.org/works", {"filter": "doi:" + d.doi_use.iloc[0], "select": SEL})
        print("키 시험 호출:", "성공" if "results" in j else j); return "results" in j
    except OAStop as e: print("⚠", e); return False

def st_probe():
    hdr("[probe] 표본 수집")
    W, miss = load_raw(); att = set(W) | miss
    pool = F1X[F1X.doi_use.notna() & (F1X.year >= 1995)]
    samp = pool.groupby(["year", "venue_short"], group_keys=False).apply(lambda g: g.sample(min(len(g), PROBE_PER_CELL), random_state=CFG["seed"]))
    todo = sorted(set(samp.doi_use) - att)
    print(f"표본 {len(samp):,}편 중 새로 받을 것 {len(todo):,}편")
    return fetch_dois(todo)

# ============================================================
# [audit]
#   칸(연도x저널) 유효 보유율 = (DOI 보유 저자 비중) x (수집된 논문에서 국가가 있는 저자 비중)
#   DOI 없는 논문은 국가 미상으로 간주(보수적)
# ============================================================
def cov_cells():
    W, miss = load_raw(); att = set(W) | miss
    ks = {}
    for d, w in W.items():
        au = w.get("authorships") or []; ks[d] = (sum(1 for a in au if countries_of(a)) / len(au)) if au else 0.0
    D = F1X[["pid", "year", "venue_short", "doi_use", "n_authors"]].copy()
    D = D[D.year >= 1995]; D["has_doi"] = D.doi_use.notna(); D["att"] = D.doi_use.isin(att)
    D["kw"] = D.doi_use.map(ks).fillna(0.0) * D.n_authors
    k = ["year", "venue_short"]; A = D[D.att]
    T = pd.DataFrame({"n_papers": D.groupby(k).size(), "A_all": D.groupby(k).n_authors.sum(), "A_doi": D[D.has_doi].groupby(k).n_authors.sum(),
                      "n_att": A.groupby(k).size(), "A_att": A.groupby(k).n_authors.sum(), "kw_att": A.groupby(k).kw.sum()}).fillna(0).reset_index()
    T["cov_doi"] = np.where(T.A_att > 0, T.kw_att / T.A_att, np.nan)
    T["eff_cov"] = np.where(T.A_doi == 0, 0.0, (T.A_doi / T.A_all) * T.cov_doi)
    return T

def st_audit():
    hdr("[audit] 연도 x 저널 커버리지 (수집된 논문 기준 추정)")
    T = cov_cells(); T["fixed"] = T.venue_short.isin(FIXED); T.to_csv(S1 / "coverage_audit.csv", index=False, encoding="utf-8-sig")
    print(f"표본에 없는 칸(DOI 보유 논문은 있는데 수집 0): {int(((T.A_doi > 0) & (T.n_att == 0)).sum())}개 (보유율 계산에서 제외)")
    pooled = lambda t: (lambda t: (t.A_all * t.eff_cov).sum() / t.A_all.sum() if len(t) else np.nan)(t.dropna(subset=["eff_cov"]))
    cov = pd.DataFrame({"전체": T.groupby("year").apply(pooled), "고정저널(합산)": T[T.fixed].groupby("year").apply(pooled),
                        "고정저널 최소(저널-연도)": T[T.fixed & (T.n_papers >= CFG["min_papers_venue_year"]) & (T.n_att >= CFG["min_att_venue_year"])].groupby("year").eff_cov.min()})
    cov.to_csv(S1 / "coverage_by_year.csv", encoding="utf-8-sig"); print(cov.round(3).to_string())
    def start(rule):
        ok = rule.reindex(range(1995, 2026)) >= CFG["cov_min"]
        for y in range(1995, 2026):
            if ok.loc[y:].all(): return y
        return None
    S = {"합산(계획서 문언)": start(np.minimum(cov["전체"], cov["고정저널(합산)"])),
         "저널별 최소(엄격)": start(np.minimum(cov["전체"], cov["고정저널 최소(저널-연도)"].fillna(1)))}
    main = S["합산(계획서 문언)"]; used = window_start()
    print("\n재계산한 창 시작 연도 후보:", S, "| 확정 창:", used)
    if main != used: print(f"⚠ 재계산 값({main})이 확정 창({used})과 다릅니다. 확정 창은 바꾸지 않았습니다. 값과 사유를 CHANGELOG_S1.md에 기록하고 민감도로 보고하세요")
    (S1 / "WINDOW_DRAFT.json").write_text(json.dumps(dict(window_start=main, window_used=used, rules=S, threshold=CFG["cov_min"], at=now()), ensure_ascii=False, indent=2), encoding="utf-8")
    low = T[(T.year >= used) & T.fixed & (T.n_att >= CFG["min_att_venue_year"]) & (T.eff_cov < CFG["cov_min"])].sort_values("eff_cov")
    low[["year", "venue_short", "n_papers", "n_att", "eff_cov"]].to_csv(S1 / "coverage_low_cells.csv", index=False, encoding="utf-8-sig")
    tot = len(T[(T.year >= used) & T.fixed & (T.n_att >= CFG["min_att_venue_year"])])
    print(f"\n창 안 고정 저널 칸 {tot}개 중 보유율 {CFG['cov_min']} 미만 {len(low)}개 -> coverage_low_cells.csv (부록 표로 보고)")
    if len(low): print(low[["year", "venue_short", "n_papers", "n_att", "eff_cov"]].head(15).to_string(index=False))
    w = T[T.year >= used]; print(f"창 안 저자 기준 DOI 없는 비중(국가 미상으로 계산됨): {1 - w.A_doi.sum() / w.A_all.sum():.1%}")
    if used > 2010: print("⚠ 창이 2010년 이후 -> 계획서 규칙: RQ4, RQ5를 부록으로 내리고 RQ1~RQ3 중심으로 전환")
    return True

# ============================================================
# [fetch] 창 안 논문 수집 (FETCH_CAP_PER_CELL=None이면 전수). 이미 받은 논문은 건너뜀
# ============================================================
def st_fetch():
    hdr("[fetch] 본 수집 (창 안)")
    y0 = window_start(); W, miss = load_raw(); att = set(W) | miss
    pool = F1X[F1X.doi_use.notna() & (F1X.year >= y0)].copy(); pool["att"] = pool.doi_use.isin(att); pool["u"] = RNG.random(len(pool))
    if FETCH_CAP_PER_CELL:
        pool = pool.sort_values(["att", "u"], ascending=[False, True]).groupby(["year", "venue_short"]).head(FETCH_CAP_PER_CELL)
    todo = sorted(set(pool.doi_use) - att); est = math.ceil(len(todo) / CFG["batch"])
    print(f"창 시작 {y0} | 대상 논문 {len(pool):,}편 (이미 받음 {int(pool.att.sum()):,}) | 새로 받을 것 {len(todo):,}편 | 예상 요청 약 {est}회(약 {est}크레딧) | "
          f"이번 실행 상한 {MAX_CALLS}회 | 오늘 누적 {USED_TODAY}/{DAILY_CREDIT_CAP}")
    ok = fetch_dois(todo)
    if ok: changelog_once("본 수집 완료", f"- {now()} | 본 수집 완료: 창 {y0}~, 칸당 상한 {FETCH_CAP_PER_CELL} | S1")
    else: print("→ 수집이 끝나지 않아 이후 단계는 실행하지 않았습니다. 같은 셀을 다시 실행하세요")
    return ok

# ============================================================
# [build] 저자 정렬 + 소속 처리 (수집된 논문만, 표본 가중치 samp_w 부여)
#   samp_w = (창 안 칸의 DOI 보유 논문 수) / (그 칸에서 수집된 논문 수). 전수 수집이면 1에 가까움. 집계는 w * samp_w
# ============================================================
def wb_income():
    p = S1 / "wb_income_current.csv"
    if not p.exists():
        j = requests.get("https://api.worldbank.org/v2/country", params={"format": "json", "per_page": 400}, timeout=60).json()[1]
        pd.DataFrame([dict(iso2=c["iso2Code"], name=c["name"], income=c["incomeLevel"]["id"], region=c["region"]["id"]) for c in j]).to_csv(p, index=False)
    Wb = pd.read_csv(p, keep_default_na=False)
    return Wb[(Wb.income != "INX") & (Wb.region != "NA")]

def st_build():
    hdr("[build] 저자 정렬과 국가 분배")
    y0 = window_start(); W, miss = load_raw(); att = set(W) | miss
    D = F1X[F1X.year >= y0].copy(); D["has_doi"] = D.doi_use.notna(); D["att"] = D.doi_use.isin(att)
    n_doi = D[D.has_doi].groupby(["year", "venue_short"]).size(); n_att = D[D.att].groupby(["year", "venue_short"]).size()
    cw = (n_doi / n_att).rename("samp_w"); D = D.join(cw, on=["year", "venue_short"])
    D["samp_w"] = np.where(D.has_doi, D.samp_w, 1.0)
    D = D[D.att | ~D.has_doi]
    P, A, ST = [], [], []
    for r in D.itertuples():
        w = W.get(r.doi_use) if r.has_doi else None
        au = (w or {}).get("authorships") or []; n = len(au)
        if n == 0:
            P.append((r.pid, "UNK", 1.0)); ST.append((r.pid, w is not None, "doi" if w else "", 0, False, 0.0, r.samp_w)); continue
        cnt, known = Counter(), 0
        for k, a in enumerate(au):
            cs = countries_of(a); aff = " || ".join((a.get("raw_affiliation_strings") or [])[:2])[:300]
            A.append((r.pid, k, (a.get("author") or {}).get("id"), a.get("author_position"), "|".join(cs), aff, n == r.n_authors))
            if cs:
                known += 1
                for c in cs: cnt[c] += 1.0 / (n * len(cs))
            else: cnt["UNK"] += 1.0 / n
        P += [(r.pid, c, v) for c, v in cnt.items()]; ST.append((r.pid, True, "doi", n, n == r.n_authors, known / n, r.samp_w))
    P = pd.DataFrame(P, columns=["pid", "country", "w"])
    A = pd.DataFrame(A, columns=["pid", "rank", "oa_author_id", "author_position", "countries", "raw_aff", "aligned"])
    ST = pd.DataFrame(ST, columns=["pid", "matched", "how", "n_oa", "aligned", "known_share", "samp_w"])
    inc = wb_income().set_index("iso2").income
    P["is_territory"] = P.country.isin(TERRITORY); P["income_fixed"] = P.country.map(inc).fillna("UNK")
    P = P.merge(F1X[["pid", "year", "venue_short"]], on="pid").merge(ST[["pid", "samp_w", "aligned"]], on="pid")
    hist = S1 / "wb_income_history.csv"
    if hist.exists():
        H = pd.read_csv(hist, keep_default_na=False); P = P.merge(H.rename(columns={"income": "income_tv"}), left_on=["country", "year"], right_on=["iso2", "year"], how="left").drop(columns="iso2")
    else: print("※ wb_income_history.csv 없음: 소득군은 현재 분류(고정)만 부착. 시변 분류 파일을 만든 뒤 build만 다시 실행하면 됩니다(API 호출 없음)")
    print("저장:", save_tbl(P, "country_authorship"), save_tbl(A, "authorship_level")); ST.to_csv(S1 / "paper_status.csv", index=False)
    wm = np.average(ST.known_share, weights=ST.samp_w); mt = ST[ST.n_oa > 0]
    print(f"창 {y0}~ | 수집·포함 논문 {len(ST):,} (DOI 없는 {int((~D.has_doi).sum()):,}) | 가중 평균 국가 보유율 {wm:.1%} | samp_w 범위 {ST.samp_w.min():.2f}~{ST.samp_w.max():.2f}")
    print(f"저자 수 정렬 일치(OpenAlex 저자 수 = F1 저자 수): {mt.aligned.mean():.1%} (수집 논문 {len(mt):,}편 중). 불일치 논문도 논문 단위 몫으로 포함됨, aligned 열로 민감도 확인 가능")
    print("국가 몫 상위(w*samp_w, unknown 포함):"); ps = (P.w * P.samp_w).groupby(P.country).sum(); print((ps / ps.sum()).sort_values(ascending=False).head(10).round(4))
    changelog_once("build: 창", f"- {now()} | build: 창 {y0}~, 표본 가중치 samp_w, 다중 소속 균등 분할, unknown 칸 유지, aligned 열 추가 | S7")
    return True

# ============================================================
# [samples] 1.7 수작업 시트, 국가 누락 점검, 1.8 대상 렌즈, 1.9 저널 메타
# ============================================================
def strat_sample(D, n, keys):
    g = D.groupby(keys); size = g.size(); alloc = (size / size.sum() * n).round().clip(lower=1).astype(int)
    return pd.concat([x.sample(min(len(x), alloc[k]), random_state=CFG["seed"]) for k, x in g])

def st_samples():
    hdr("[samples] 검증 표본 생성")
    y0 = window_start()
    Au0 = read_tbl("authorship_level").merge(F1X[["pid", "year", "venue_short", "title", "doi_use"]], on="pid"); Au0 = Au0[Au0.year >= y0].copy()
    Au0["countries"] = Au0.countries.fillna(""); Au0["raw_aff"] = Au0.raw_aff.fillna("")
    Au = Au0[Au0.countries != ""].copy()
    Au["per"] = pd.cut(Au.year, [y0 - 1, 2013, 2019, 2025]).astype(str); Au["c1"] = Au.countries.str.split("|").str[0]
    top = Au.c1.value_counts().head(5).index; Au["cg"] = np.where(Au.c1.isin(top), Au.c1, "other")
    M = strat_sample(Au, CFG["manual_n"], ["per", "cg"]).sample(frac=1, random_state=CFG["seed"]).head(CFG["manual_n"])
    M = M.assign(true_country="", ok="")[["pid", "rank", "year", "venue_short", "doi_use", "title", "raw_aff", "countries", "true_country", "ok"]]
    M.to_csv(S1 / "manual_verify.csv", index=False, encoding="utf-8-sig")
    print(f"manual_verify.csv {len(M)}행: raw_aff와 countries를 원문과 대조해 ok에 1(맞음)/0(틀림), 틀리면 true_country도 기입")
    U = Au0[(Au0.countries == "") & (Au0.raw_aff != "")]
    print(f"창 안 authorship 중 소속 문자열은 있으나 국가 없음: {len(U):,} ({len(U) / max(len(Au0), 1):.1%})")
    if len(U):
        U = U.sample(min(CFG["unknown_n"], len(U)), random_state=CFG["seed"]).assign(true_country="", missed="")
        U[["pid", "rank", "year", "venue_short", "raw_aff", "true_country", "missed"]].to_csv(S1 / "unknown_check.csv", index=False, encoding="utf-8-sig")
        print("unknown_check.csv: raw_aff에서 국가를 판단할 수 있으면 missed=1(누락), 판단 불가면 0")
    try:
        import pycountry; names = {c.name: c.alpha_2 for c in pycountry.countries} | {getattr(c, "common_name", c.name): c.alpha_2 for c in pycountry.countries}
    except Exception: names = {}; print("※ pycountry 없음: pip install pycountry (없으면 대상 렌즈 생략)")
    amb = {"Georgia", "Turkey", "Jordan", "Chad", "Niger", "Guinea", "Mali", "Malta", "Cuba", "Peru", "Oman", "Togo", "Samoa"}
    gz = S1 / "gazetteer_cities.csv"
    if gz.exists(): names |= dict(pd.read_csv(gz, dtype=str, keep_default_na=False).values)
    names = {k: v for k, v in names.items() if k not in amb and len(k) >= 4}
    if names:
        rx = re.compile(r"\b(" + "|".join(sorted(map(re.escape, names), key=len, reverse=True)) + r")\b")
        hit = F1X.title.fillna("").map(lambda s: sorted({names[m] for m in rx.findall(s)}))
        TP = F1X[["pid", "year", "venue_short", "title"]].assign(target_countries=hit.map("|".join)); TP.to_csv(S1 / "title_places.csv", index=False, encoding="utf-8-sig")
        has = TP.target_countries != ""; print("대상 렌즈 보유율(5년 구간):", has.groupby(TP.year // 5 * 5).mean().round(3).to_dict())
        TP[has].sample(min(200, int(has.sum())), random_state=CFG["seed"]).assign(precision_ok="").to_csv(S1 / "target_lens_review.csv", index=False, encoding="utf-8-sig")
    ev = pd.read_csv(S0 / "EVENTS_FROZEN.csv")[["venue_short", "E1_corpus_entry", "E1_founding", "entry_ruling"]]
    vm = F1X.groupby("venue_short").agg(n_papers=("pid", "size")).reset_index().merge(ev, on="venue_short", how="left"); vm["fixed"] = vm.venue_short.isin(FIXED)
    vm.to_csv(S1 / "venue_meta.csv", index=False, encoding="utf-8-sig")
    pd.MultiIndex.from_product([sorted(F1X.venue_short.unique()), range(2000, 2026)], names=["venue_short", "year"]).to_frame(index=False).assign(jif="", source="").to_csv(S1 / "jif_template.csv", index=False)
    print("venue_meta.csv, jif_template.csv 저장 (JIF는 맥락 변수)")
    return True

# ============================================================
# [fuzzy_probe] DOI 없는 논문의 제목 매칭 회수율을 FUZZY_N편으로만 시험(결과는 분석에 반영하지 않음)
#   제목 검색은 보수적으로 요청당 10크레딧으로 계산 (100편 = 1,000크레딧)
# ============================================================
def st_fuzzy_probe():
    hdr("[fuzzy_probe] 제목 매칭 회수율 시험")
    y0 = window_start(); q = F1X[F1X.doi_use.isna() & (F1X.year >= y0)]
    if len(q) == 0: print("창 안에 DOI 없는 논문이 없습니다"); return True
    n_all = len(q); q = q.sample(min(FUZZY_N, len(q)), random_state=CFG["seed"]); nm = lambda s: re.sub(r"[^a-z0-9]+", " ", str(s).lower()).strip(); rows = []
    try:
        for r in q.itertuples():
            title = re.sub(r"[,|:;]", " ", str(r.title))[:250]
            j = oa_get("https://api.openalex.org/works", {"filter": f"title.search:{title},publication_year:{int(r.year)}", "per-page": 5, "select": "display_name,doi"}, cost=CREDIT_TITLE_SEARCH)
            sc = max([difflib.SequenceMatcher(None, nm(r.title), nm(w.get("display_name"))).ratio() for w in j.get("results", [])] + [0.0]) if "error" not in j else np.nan
            rows.append(dict(pid=r.pid, year=r.year, venue=r.venue_short, score=sc)); time.sleep(0.1)
    except OAStop as e: print("⚠ 중단:", e)
    F = pd.DataFrame(rows); F.to_csv(S1 / "fuzzy_probe.csv", index=False)
    if len(F): print(f"표본 {len(F)}편 | 회수(점수≥{CFG['fuzzy_accept']}) {(F.score >= CFG['fuzzy_accept']).mean():.1%} | 창 안 DOI 없는 논문 {n_all:,}편")
    print("회수율이 높으면 그때 전수 매칭 여부를 결정하세요. 낮으면 DOI 없는 논문은 국가 미상으로 두는 것이 합리적입니다.")
    return True

# ============================================================
# [evaluate]
# ============================================================
def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n); h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)); return (c - h) / d, (c + h) / d

def st_evaluate():
    hdr("[evaluate] 통과 판정")
    R = []; ws = json.loads((S1 / "WINDOW_DRAFT.json").read_text()); used = window_start()
    R.append((f"1.5 분석 창 시작 연도(확정 {used}, 재계산 {ws['window_start']}; 2010 이하면 통과)", used, used <= 2010))
    mp = S1 / "manual_verify.csv"
    if mp.exists():
        Mv = pd.read_csv(mp, dtype=str, encoding="utf-8-sig").fillna(""); done = Mv[Mv.ok.isin(["0", "1"])]
        if len(done) >= 150:
            k, n = int((done.ok == "1").sum()), len(done); lo, hi = wilson(k, n)
            R.append(("1.7 수작업 정확도(점추정≥0.95, Wilson 하한≥0.90)", f"{k / n:.3f} [{lo:.3f}, {hi:.3f}] n={n}", k / n >= CFG["acc_min"] and lo >= CFG["wilson_low"]))
        else: R.append(("1.7 수작업 정확도", f"판정 {len(done)}건(<150): 더 채우세요", None))
    else: R.append(("1.7 수작업 정확도", "manual_verify.csv 없음", None))
    up = S1 / "unknown_check.csv"
    if up.exists():
        U = pd.read_csv(up, dtype=str, encoding="utf-8-sig").fillna(""); U = U[U.missed.isin(["0", "1"])]
        if len(U) >= 80:
            k, n = int((U.missed == "1").sum()), len(U); lo, hi = wilson(k, n)
            R.append(("1.6 대체: 소속 문자열 있는데 국가 누락된 비율(참고)", f"{k / n:.3f} [{lo:.3f}, {hi:.3f}] n={n}", "참고"))
        else: R.append(("1.6 대체: 국가 누락 점검", f"판정 {len(U)}건(<80)", None))
    else: R.append(("1.6 대체: 국가 누락 점검", "unknown_check.csv 없음", None))
    T = pd.DataFrame(R, columns=["검사", "값", "통과"]); print(T.to_string(index=False))
    fail = any(x is False for x in T.통과); skip = any(x is None for x in T.통과)
    msg = ("# S1 검증 보고서\n\n" + T.to_string(index=False) + f"\n\n작성 {now()}\n\n"
           + ("**일부 검사 미실시(SKIP)**: 완료로 간주하지 말 것.\n" if skip else "")
           + ("**FAIL 존재**: FALLBACK 적용(창을 2010년 이후로 축소 / 고정 저널만 / 비DOI는 부록). 창이 2010년 이후면 RQ4·RQ5를 부록으로, RQ1~RQ3 중심 논문으로 전환.\n" if fail else "")
           + "\n한계: 소속 국가는 국적이 아님. 구 자료는 소속 기재가 성김. 국가는 단일 출처(OpenAlex)에 의존(Lens 교차 확인 안 함). 창 안 DOI 보유 논문은 전수 수집(상한 설정 시 samp_w 가중), DOI 없는 논문은 국가 미상으로 계산.\n")
    (S1 / "validation_report.md").write_text(msg, encoding="utf-8"); print("저장: validation_report.md")
    return True

FUNCS = {"inspect": st_inspect, "probe": st_probe, "audit": st_audit, "fetch": st_fetch, "build": st_build,
         "samples": st_samples, "fuzzy_probe": st_fuzzy_probe, "evaluate": st_evaluate}
try:
    for s in STAGES:
        if FUNCS[s]() is False: break
finally:
    c, n = CREDITS["n"], CALLS["n"]; ledger_write()
    print(f"\n[이번 실행] 요청 {n}회, 추정 크레딧 {c} (약 ${c / 10000:.3f}) | 오늘(UTC) 누적 {ledger_today()}/{DAILY_CREDIT_CAP} (일일 총 10,000 = $1)")

OpenAlex API key:  ········


F1 111,685 -> F1X 111,416 | DOI 보유 89.6% | STAGES=['samples']

[samples] 검증 표본 생성
manual_verify.csv 200행: raw_aff와 countries를 원문과 대조해 ok에 1(맞음)/0(틀림), 틀리면 true_country도 기입
창 안 authorship 중 소속 문자열은 있으나 국가 없음: 13,362 (4.4%)
unknown_check.csv: raw_aff에서 국가를 판단할 수 있으면 missed=1(누락), 판단 불가면 0
대상 렌즈 보유율(5년 구간): {1965: 0.006, 1970: 0.018, 1975: 0.004, 1980: 0.012, 1985: 0.014, 1990: 0.02, 1995: 0.046, 2000: 0.056, 2005: 0.058, 2010: 0.07, 2015: 0.071, 2020: 0.059, 2025: 0.057}
venue_meta.csv, jif_template.csv 저장 (JIF는 맥락 변수)

[이번 실행] 요청 0회, 추정 크레딧 0 (약 $0.000) | 오늘(UTC) 누적 1532/8000 (일일 총 10,000 = $1)


In [9]:
#

In [ ]:
# 먼저 전체 그림부터 말씀드릴게요. 이 연구는 교통 분야 논문을 11만 편쯤 모아서 "어느 나라 연구자들이 쓴 논문일까?"를 세어 보는 연구입니다. 컴퓨터가 논문마다 나라를 자동으로 붙였는데, 컴퓨터가 틀렸을 수도 있어서 사람이 표본을 뽑아 채점해 보는 겁니다. 세 파일은 그 채점표예요.
# ## 1. manual_verify.csv: "컴퓨터가 붙인 나라가 맞나?"
# 각 줄에는 저자 한 명의 소속(예: "Beijing Institute of Technology, Beijing, China")과 컴퓨터가 붙인 나라(`CN`)가 적혀 있어요. 맞으면 `ok`에 1, 틀리면 0을 씁니다.
# 숙제 검사에 비유하면, 컴퓨터가 "이 학교는 중국에 있어요"라고 답을 냈고, 우리는 채점자로서 맞았는지 보는 거예요. 200개를 채점해서 95% 이상 맞아야 "이 자동 방식을 믿어도 된다"고 통과시킵니다. 이 연구에서 제일 중요한 채점표예요.
# ## 2. unknown_check.csv: "나라를 못 붙인 것 중에 사실 알 수 있는 게 있나?"
# 컴퓨터가 "이 사람이 어느 나라 소속인지 모르겠어요" 하고 포기한 경우들이에요. 예를 들면 "Maricopa Association of Governments, Phoenix, AZ"처럼 회사나 관청 이름은 컴퓨터가 모르는데, 글자를 읽어 보면 Phoenix, AZ는 미국이잖아요? 이렇게 사람이 보면 알 수 있으면 `missed`에 1, 아무리 봐도 모르겠으면 0을 씁니다.
# 100개 중 97개는 알 수 있었어요. 컴퓨터가 쉬운 걸 꽤 놓치고 있다는 뜻이고, 이런 걸 규칙으로 되살리면 결과가 더 정확해질 수 있습니다.
# ## 3. target_lens_review.csv: "논문 제목에서 뽑은 나라가 진짜 연구 대상인가?"
# 이건 저자 소속이 아니라 **논문이 어느 나라를 연구했는지**를 보는 파일이에요. 컴퓨터가 제목에서 나라 이름을 찾아냅니다.
# - "Lagos, **Nigeria**"에서 NG를 찾았다면 맞는 매칭이니 1입니다.
# - "I-40 Bridges in Guadalupe County, **New Mexico**"에서 MX(멕시코)를 찾았다면, New Mexico는 미국 주인데 "Mexico"라는 글자만 보고 잘못 뽑은 거라 0입니다.
# 200편 중 5편만 이렇게 틀려서 195편이 맞았어요. 그래서 이 방법은 꽤 정확합니다.
# ## 한 줄 요약
# | 파일 | 한마디 질문 | 결과 |
# |---|---|---|
# | manual_verify | 붙인 나라가 맞아? | 200개 모두 맞음 |
# | unknown_check | 못 붙인 것 중 알 수 있는 게 있어? | 97개는 알 수 있음 |
# | target_lens_review | 제목에서 뽑은 나라가 진짜 연구 대상이야? | 195개 맞고 5개 틀림 |
# 다만 저는 논문 원문을 열어 본 게 아니라 파일 안의 글자만 보고 채점했어요. 그래서 일부는 직접 원문을 열어 다시 확인해야 합니다.

In [ ]:
# # S1 검수 판정 기준 및 확인 기록
# - 작성일: 2026-10-06
# - 대상 파일: `manual_verify.csv`(200건), `unknown_check.csv`(100건), `target_lens_review.csv`(200편)
# - 위치 권장: `paper1/01_S1/S1_검수판정_기준_및_확인기록.md`

# ## 1. 기록 시점에 관한 사실 (먼저 읽을 것)
# 이 문서의 판정 기준은 **AI 사전 판정이 끝난 뒤, `evaluate` 실행 전에** 작성했다. 판정 전 기록이 아니다. 기준 자체는 판정 작업에 쓰인 규칙을 그대로 옮긴 것이며, `evaluate` 결과를 본 뒤에 바꾸지 않는다. 이 점을 CHANGELOG와 논문 한계 절에 같은 표현으로 적는다.

# ## 2. 판정 기준
# ### 2.1 manual_verify (저자 단위 국가 정확도)
# - `ok`=1: `raw_aff`(소속 문자열)에 비추어 `countries`가 맞다.
# - `ok`=0: 틀리거나 빠진 국가가 있다. 이때 `true_country`에 정답을 적는다.
# - 국가는 **소속 기관의 소재지**로 판정한다. 해외 분교는 분교 소재지(예: Nottingham Ningbo China → CN).
# - 영토(HK, TW, PR 등)는 OpenAlex 코드 그대로 두고 일관되게 적용한다.
# - `raw_aff`는 저자당 2개까지만 저장되어 있다. 3번째 이후 소속에서 온 국가는 문자열만으로 대조할 수 없다.
# ### 2.2 unknown_check (국가 미상 중 복구 가능 비율)
# - `missed`=1: `raw_aff` 문자열만으로 국가를 판단할 수 있다(국가명, 주 약어, 도시와 국가 조합 등). `true_country`에 국가 코드를 적는다.
# - `missed`=0: 문자열에 지명이 없어 판단할 수 없다. 외부 지식(기관의 실제 위치 추정)은 쓰지 않는다.
# - 이 값은 통과 판정이 아니라 **참고 지표**다.
# ### 2.3 target_lens_review (제목 지명 대상 렌즈 정밀도)
# - `precision_ok`=1: 매칭된 국가가 그 논문의 **연구 대상 지역**이다. 비교 대상으로 명시된 국가도 포함한다.
# - `precision_ok`=0: 지명이 아닌 글자 일치(예: New Mexico → MX), 출판사 주소, 연구실 소개글, 연구 대상이 아닌 인접 국가.
# - 매칭에서 **빠진** 국가는 정밀도에 넣지 않는다(재현율 문제).

# ## 3. 판정 절차와 AI 사용 범위
# 1. Claude가 파일 안의 `raw_aff`, 제목 텍스트만 읽고 전 행을 판정했다. 논문 원문(PDF, 출판사 페이지)은 열지 않았다.
# 2. 예외 1건(manual_verify 25038)만 웹 검색으로 확인했다. 저자 목록(rank 1 = Gautam Srivastava, Brandon University)은 확인했으나, 논문 소속란에서 대만 소속을 직접 확인하지는 못했다. 현재 ok=1로 두었고 원문 확인 대상이다.
# 3. 연구자가 AI 사전 판정 전체를 검토하고 **모두 적절하다고 확인했다**(확인일: 2026-10-06).

# ## 4. 결과
# | 시트 | 결과 | Wilson 95% 구간 | 용도 |
# |---|---|---|---|
# | manual_verify (200건) | 정확 200, 오류 0 (1.000) | [0.981, 1.000] | S1 통과 판정(점추정 ≥ 0.95, 하한 ≥ 0.90) |
# | unknown_check (100건) | 복구 가능 97, 불가 3 (0.970) | [0.915, 0.990] | 참고 지표 |
# | target_lens_review (200편) | 정확 195, 오류 5 (0.975) | [0.943, 0.989] | 대상 렌즈 정밀도(`evaluate` 밖에서 계산) |

# ## 5. 예외와 경계 사례
# - **ok 또는 precision 0으로 판정한 건**
#   - unknown_check 불가 3건: 65744, 84931, 68765 (소속 문자열에 지명 없음. 84931은 외부 지식으로 추정 가능하나 기준에 따라 0).
#   - target_lens 오류 5건: 90265(New Mexico → MX), 100997(New Jersey → JE), 101469(서평 속 출판사 주소 → DE), 74350(연구실 소개), 11912(우크라이나 접경국 연구인데 UA 매칭).
# - **1로 판정한 경계 사례**: 77297(뉴스 칼럼, KZ|UA), 44032(비교 대상 JP), 63026(PR), 25038(CA\|LB\|TW, 아래 원문 확인 대상).
# - **재현율 한계(정밀도에는 미반영)**: 94445(Britain 미포착), 17180(Europe 미포착), 77297(Sydney 미포착).

# ## 6. 원문 대조 기록 (연구자 기입)
# 계획서 기준은 원문 대조다. 아래를 직접 채운다. 채우기 전까지 이 항목은 **미완료**로 남긴다.
# - 원문 대조 건수: n = ____ (25038 포함 여부: ____)
# - 대조 방법(논문 페이지, PDF 등): ____
# - 판정이 바뀐 건수와 pid: ____
# - 확인자 / 확인일: ____

# ## 7. 한계
# - 소속 국가는 국적이 아니다.
# - 국가는 단일 출처(OpenAlex)에 의존한다(Lens 교차 확인은 생략).
# - 판정은 AI 사전 판정과 연구자 검토에 기반하며 원문 대조는 표본에 한정된다(6절).
# - `raw_aff` 2개 제한 때문에 다중 소속 저자의 3번째 이후 국가는 대조하지 못했다.
# - `unknown_check`에서 복구 가능한 미상은 US 실무기관과 CN 기업·연구소에 치우쳐 있을 가능성이 있다. 규칙 기반 복구는 민감도 분석으로 별도 사전 기록한다.

# ## 8. 후속
# 1. 6절 기입 후 완성본 3종을 `paper1/01_S1`에 덮어쓴다(원본 백업).
# 2. `STAGES = ["evaluate"]` 실행 후 `validation_report.md`를 이 문서와 함께 보관한다.
# 3. CHANGELOG_S1에 기록하고 S0와 S1 산출물을 git commit과 태그로 동결한다.

In [11]:
#

In [17]:
"""
S1 후속 작업 통합 스크립트  (OpenAlex 호출 없음, 크레딧 소모 없음)

권장 실행 순서 (STEPS에 넣어 한 번에 하나 이상 실행)
  1) install         완성본 CSV 3종 검증 -> 백업 -> paper1/01_S1에 설치
  2) source_sample   원문 대조용 표본(기본 40건 + 25038) 시트 생성
     --- 여기서 사람이 논문 원문을 열어 manual_source_check.csv를 채움 ---
  3) source_ingest   대조 결과 집계, 불일치 반영, 기록 문서 6절 자동 기입
  4) (원 스크립트에서 STAGES = ["evaluate"] 실행)
  5) lens_precision  대상 렌즈 정밀도 계산
  6) changelog       CHANGELOG_S1.md에 기록
  7) territory_doc   영토·구 국가 표기 규칙 문서화
  8) recovery        국가 미상 규칙 복구(민감도용)를 동결 기록 후 적용
  9) git_freeze      산출물 해시 기록, 커밋, 태그 (기본 DRY_RUN)
 언제든: nauthors_check, income_history, venue_check
"""
import os, re, json, math, html, hashlib, shutil, subprocess, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)

# ---------------- 설정 (여기만 수정) ----------------
STEPS = ["source_sample"]
SRC_DIR = Path.home() / "Downloads"          # 완성본 CSV 3종이 있는 폴더
ALLOW_OVERWRITE_JUDGED = False               # 01_S1의 시트에 이미 판정이 있으면 덮어쓰지 않음
SOURCE_N = 40                                # 원문 대조 표본 크기(25038은 별도 추가)
FORCE_PIDS = ["25038"]                       # 반드시 원문 확인할 pid
CHECKER = ""                                 # 원문 대조 담당자 이름(ingest 전에 기입)
SOURCE_METHOD = ""                           # 예: "출판사 DOI 페이지의 저자 소속란"
APPLY_CORRECTIONS = True                     # 원문 대조에서 틀린 건을 manual_verify.csv에 반영
SEED = 2026
INCOME_XLSX = Path.home() / "Downloads" / "OGHIST.xlsx"   # World Bank 소득군 이력 파일(직접 내려받기)
EXPECTED_PREREG_PREFIX = "f16e76b4"          # 보정된 PREREG 해시 앞부분
IGNORE_PREREG_HASH_MISMATCH = False
DRY_RUN = True                               # False로 바꿔야 실제 커밋·태그
TAG_NAME = None                              # None이면 S0S1-freeze-YYYY-MM-DD
CFG = dict(acc_min=0.95, wilson_low=0.90, window=2007)
# ----------------------------------------------------

hdr = lambda t: print("\n" + "=" * 90 + f"\n{t}\n" + "=" * 90)
sha_b = lambda b: hashlib.sha256(b).hexdigest()
sha_f = lambda p: sha_b(Path(p).read_bytes())
now = lambda: pd.Timestamp.now().isoformat(timespec="seconds")
rd = lambda p: pd.read_csv(p, dtype=str, encoding="utf-8-sig").fillna("")
DATA = ROOT = S0 = S1 = None
DOC = "S1_검수판정_기준_및_확인기록.md"


def paths():
    c = ([Path(os.environ["S1_DATA_DIR"])] if os.environ.get("S1_DATA_DIR") else []) + [
        Path("../analysis_data/final"), Path("analysis_data/final"), Path("./final"),
        Path.home() / "Research/Minnesota_Transport/analysis_data/final"]
    d = next((p for p in c if (p / "F1_papers.csv").exists()), None)
    if d is None: raise SystemExit("F1_papers.csv를 찾지 못했습니다. 환경변수 S1_DATA_DIR로 analysis_data/final 경로를 지정하세요")
    r = d.resolve().parent.parent
    return d, r, r / "paper1" / "00_S0", r / "paper1" / "01_S1"


def changelog(line):
    cl = S1 / "CHANGELOG_S1.md"
    if not cl.exists(): cl.write_text("# CHANGELOG S1 (날짜 | 변경 | 사유 | 영향)\n\n", encoding="utf-8")
    with cl.open("a", encoding="utf-8") as f: f.write(line + "\n")


def changelog_once(key, line):
    cl = S1 / "CHANGELOG_S1.md"
    if cl.exists() and key in cl.read_text(encoding="utf-8"): return False
    changelog(line); return True


def read_tbl(name):
    p = S1 / f"{name}.parquet"
    return pd.read_parquet(p) if p.exists() else pd.read_csv(S1 / f"{name}.csv.gz")


def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n); h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return (c - h) / d, (c + h) / d


# ============================================================
# 1) install
# ============================================================
SHEETS = {"manual_verify.csv": (["pid", "rank"], "ok"), "unknown_check.csv": (["pid", "rank"], "missed"),
          "target_lens_review.csv": (["pid"], "precision_ok")}


def st_install():
    hdr("[install] 완성본 검증 후 설치")
    plan = []
    for f, (key, col) in SHEETS.items():
        src = SRC_DIR / f
        if not src.exists(): raise SystemExit(f"{src} 없음. SRC_DIR을 확인하세요")
        df = rd(src)
        if not df[col].isin(["0", "1"]).all(): raise SystemExit(f"{f}: {col}에 0/1이 아닌 값이 있습니다")
        if f == "manual_verify.csv" and ((df.ok == "0") & (df.true_country == "")).any():
            raise SystemExit("manual_verify.csv: ok=0인데 true_country가 비어 있는 행이 있습니다")
        cur = S1 / f
        if cur.exists():
            c = rd(cur)
            if list(c.columns) != list(df.columns): raise SystemExit(f"{f}: 열 구성이 기존 파일과 다릅니다")
            ks = lambda x: {tuple(r) for r in x[key].itertuples(index=False)}
            if ks(c) != ks(df): raise SystemExit(f"{f}: 행 키({key})가 기존 파일과 다릅니다. 표본이 바뀐 파일은 설치하지 않습니다")
            judged = int(c[col].isin(["0", "1"]).sum())
            if judged and not ALLOW_OVERWRITE_JUDGED:
                raise SystemExit(f"{f}: 01_S1의 기존 파일에 이미 판정 {judged}건이 있습니다. 확인 후 ALLOW_OVERWRITE_JUDGED=True로 바꾸세요")
        plan.append((f, src, cur)); print(f"  OK  {f}: {len(df)}행, {col} 분포 {df[col].value_counts().to_dict()}")
    bk = S1 / f"_backup_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}"; bk.mkdir(parents=True)
    log = []
    for f, src, cur in plan:
        if cur.exists(): shutil.copy2(cur, bk / f)
        shutil.copy2(src, cur); log.append(dict(file=f, sha256=sha_f(cur), installed_at=now()))
    lp = S1 / "S1_SHEETS_LOG.csv"
    pd.DataFrame(log).to_csv(lp, mode="a", header=not lp.exists(), index=False)
    print(f"설치 완료. 백업: {bk.name} | 해시 기록: {lp.name}")
    return True


# ============================================================
# 2) source_sample / 3) source_ingest
# ============================================================
SRC_FILE = "manual_source_check.csv"


def st_source_sample():
    hdr("[source_sample] 원문 대조 표본")
    out = S1 / SRC_FILE
    if out.exists(): print(f"{out.name}가 이미 있습니다(판정 보호를 위해 덮어쓰지 않음)"); return True
    M = rd(S1 / "manual_verify.csv")
    forced = M[M.pid.isin(FORCE_PIDS)]
    rest = M[~M.pid.isin(FORCE_PIDS)].sample(min(SOURCE_N, len(M) - len(forced)), random_state=SEED)
    X = pd.concat([forced, rest])
    X = X.assign(doi_url="https://doi.org/" + X.doi_use, source_aff_seen="", source_countries="", source_ok="", note="")
    X = X[["pid", "rank", "year", "venue_short", "doi_url", "title", "raw_aff", "countries", "source_aff_seen", "source_countries", "source_ok", "note"]]
    X.to_csv(out, index=False, encoding="utf-8-sig")
    print(f"{out.name} {len(X)}행 생성(seed={SEED}, 강제 포함 {FORCE_PIDS}).")
    print("채우는 법: 논문 페이지에서 rank번째(0부터) 저자의 소속을 찾아 source_aff_seen에 적고,")
    print("  source_countries에 국가 코드(복수면 |), source_ok에는 countries가 원문과 일치하면 1, 아니면 0.")
    return True


def st_source_ingest():
    hdr("[source_ingest] 원문 대조 집계")
    if not CHECKER.strip(): raise SystemExit("CHECKER(대조 담당자)를 설정에 기입하세요")
    X = rd(S1 / SRC_FILE); done = X[X.source_ok.isin(["0", "1"])]
    if len(done) < len(X): print(f"⚠ 미판정 {len(X) - len(done)}건이 있습니다. 판정된 {len(done)}건만 집계합니다")
    if len(done) == 0: raise SystemExit("판정된 행이 없습니다")
    k, n = int((done.source_ok == "1").sum()), len(done); lo, hi = wilson(k, n)
    bad = done[done.source_ok == "0"]
    print(f"원문 대조 {n}건 | 일치 {k} ({k / n:.3f}, Wilson [{lo:.3f}, {hi:.3f}]) | 불일치 {len(bad)}")
    if len(bad): print(bad[["pid", "rank", "countries", "source_countries", "note"]].to_string(index=False))
    inc = bool(done.pid.isin(FORCE_PIDS).any())
    summ = dict(n_checked=n, n_agree=k, n_disagree=len(bad), disagree_pids=[f"{r.pid}:{r.rank}" for r in bad.itertuples()],
                wilson=[round(lo, 3), round(hi, 3)], includes_forced=inc, checker=CHECKER, method=SOURCE_METHOD, at=now())
    (S1 / "SOURCE_CHECK_SUMMARY.json").write_text(json.dumps(summ, ensure_ascii=False, indent=2), encoding="utf-8")
    if len(bad) and APPLY_CORRECTIONS:
        M = rd(S1 / "manual_verify.csv"); shutil.copy2(S1 / "manual_verify.csv", S1 / f"manual_verify.before_source_{pd.Timestamp.now().strftime('%H%M%S')}.csv")
        for r in bad.itertuples():
            m = (M.pid == r.pid) & (M["rank"] == r.rank); M.loc[m, "ok"] = "0"; M.loc[m, "true_country"] = r.source_countries
            if not r.source_countries: print(f"⚠ {r.pid}: source_countries가 비어 있습니다. true_country를 직접 채우세요")
        M.to_csv(S1 / "manual_verify.csv", index=False, encoding="utf-8-sig")
        print("manual_verify.csv에 불일치 반영함. evaluate를 다시 실행하세요")
    dp = S1 / DOC
    if dp.exists():
        rep = {"- 원문 대조 건수:": f"- 원문 대조 건수: n = {n} (25038 포함 여부: {'예' if inc else '아니오'})",
               "- 대조 방법": f"- 대조 방법(논문 페이지, PDF 등): {SOURCE_METHOD or '(미기입)'}",
               "- 판정이 바뀐 건수와 pid:": f"- 판정이 바뀐 건수와 pid: {len(bad)}건 {summ['disagree_pids'] or ''}",
               "- 확인자 / 확인일:": f"- 확인자 / 확인일: {CHECKER} / {pd.Timestamp.now().strftime('%Y-%m-%d')}"}
        L = dp.read_text(encoding="utf-8").splitlines()
        L = [next((v for p, v in rep.items() if l.startswith(p)), l) for l in L]
        dp.write_text("\n".join(L) + "\n", encoding="utf-8"); print(f"{DOC} 6절 갱신")
    else: print(f"※ {DOC}가 01_S1에 없어 6절 자동 기입은 건너뜀(문서를 01_S1에 복사한 뒤 다시 실행)")
    print("다음: 원 스크립트에서 STAGES = ['evaluate'] 실행")
    return True


# ============================================================
# 5) lens_precision
# ============================================================
def st_lens_precision():
    hdr("[lens_precision] 대상 렌즈 정밀도")
    T = rd(S1 / "target_lens_review.csv"); T = T[T.precision_ok.isin(["0", "1"])]
    T["year"] = T.year.astype(int); k, n = int((T.precision_ok == "1").sum()), len(T); lo, hi = wilson(k, n)
    out = dict(n=n, correct=k, precision=round(k / n, 3), wilson=[round(lo, 3), round(hi, 3)], wrong_pids=T[T.precision_ok == "0"].pid.tolist(), by_period={})
    T["per"] = pd.cut(T.year, [0, 2006, 2013, 2019, 2100], labels=["~2006", "2007-13", "2014-19", "2020-"])
    for p, g in T.groupby("per", observed=True):
        kk = int((g.precision_ok == "1").sum()); out["by_period"][str(p)] = f"{kk}/{len(g)}"
    print(json.dumps(out, ensure_ascii=False, indent=2))
    (S1 / "target_lens_precision.json").write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding="utf-8")
    print("※ 재현율(빠진 지역)은 이 값에 포함되지 않습니다")
    return True


# ============================================================
# 6) changelog
# ============================================================
def st_changelog():
    hdr("[changelog] CHANGELOG_S1 기록")
    sp, lp = S1 / "SOURCE_CHECK_SUMMARY.json", S1 / "target_lens_precision.json"
    if not sp.exists(): raise SystemExit("먼저 source_ingest를 실행하세요(원문 대조 건수가 필요합니다)")
    s = json.loads(sp.read_text(encoding="utf-8")); lens = json.loads(lp.read_text(encoding="utf-8")) if lp.exists() else None
    a = changelog_once("검수 3종 사전 판정", f"- {now()} | 검수 3종 사전 판정(manual_verify 200, unknown_check 100, target_lens_review 200): Claude가 raw_aff·제목을 읽고 판정, 연구자가 전체 검토 후 확인. 원문 대조 n={s['n_checked']}건(일치 {s['n_agree']}, 불일치 {s['n_disagree']}, 담당 {s['checker']}). 판정 기준 문서({DOC})는 사전 판정 후 evaluate 실행 전에 작성(판정 전 기록 아님) | 판정 방식 | S1 통과 판정, S10.4 생성형 AI 사용 선언")
    b = changelog_once("본 수집 후 audit 재계산", f"- {now()} | 본 수집 후 audit 재계산: 합산 규칙 {CFG['window']} 유지, 고정 저널 266칸 중 0.80 미만 1칸(TranspPol 2008, 0.800), 엄격 규칙 2009. 확정 창은 변경하지 않고 엄격 규칙은 민감도로 보고 | 분석 창 | S5.1(c), S7, S8")
    c = changelog_once("fuzzy_probe 생략", f"- {now()} | fuzzy_probe 생략: 창 안 DOI 없는 논문 160편은 국가 미상으로 계산(보수적) | 요청·크레딧 절감 | 국가 몫의 미상 비중에 포함")
    d = changelog_once("대상 렌즈 보조", f"- {now()} | 대상 렌즈(제목 지명) 보조 변수로 사용: 정밀도 {lens['correct']}/{lens['n']} (Wilson {lens['wilson']}), 재현율은 미측정(예: Britain, Europe 미포착) | 보조 변수 | RQ 대상 지역 분석은 정밀도 한계 명시" if lens else "")
    print("추가된 항목:", [x for x, y in zip("abcd", [a, b, c, d if lens else False]) if y] or "없음(이미 기록됨)")
    return True


# ============================================================
# 7) territory_doc
# ============================================================
TERRITORY = {"HK", "MO", "TW", "PR", "GU", "VI", "AS", "MP", "GI", "FO", "GL", "AW", "CW", "SX", "NC", "PF", "RE", "GP", "MQ", "GF", "YT", "KY", "BM", "XK", "PS", "EH"}
FORMER = {"SU", "YU", "CS", "DD", "AN"}


def st_territory_doc():
    hdr("[territory_doc] 영토·구 국가 표기 규칙")
    txt = ["# 영토·구 국가 표기 규칙 (S1)", f"- 작성: {now()}  (S7 결과 열람 전)", "",
           "## 규칙", "1. 국가는 OpenAlex가 부여한 `country_code`를 그대로 쓴다. 영토(HK, TW, PR 등)는 모국과 합치지 않고 별도 국가로 둔다.",
           "2. 영토 코드에는 `is_territory=True` 열을 붙여 민감도 분석에서 분리할 수 있게 한다.",
           "3. 구 국가 코드(SU, YU, CS, DD, AN)는 `ZZ_former`로 묶고 승계국에 배분하지 않는다.",
           "4. 다중 소속 저자는 소속 국가에 균등 분할한다. 소속이 없는 몫은 `UNK`로 유지한다.",
           "5. 소속 국가는 국적이 아니다.", "", "## 영토 코드 목록", ", ".join(sorted(TERRITORY)), "", "## 구 국가 코드 목록", ", ".join(sorted(FORMER)), ""]
    try:
        P = read_tbl("country_authorship"); P["ws"] = P.w * P.samp_w; tot = P.ws.sum()
        terr = P[P.is_territory].groupby("country").ws.sum().sort_values(ascending=False) / tot
        fm = P[P.country == "ZZ_former"].ws.sum() / tot
        txt += ["## 현재 자료의 규모 (기술 통계, 가설 검정 아님)", f"- 영토 합계 몫: {terr.sum():.4f}", f"- 구 국가(ZZ_former) 몫: {fm:.5f}", "- 영토별 몫: " + ", ".join(f"{c} {v:.4f}" for c, v in terr.head(8).items()), ""]
        print(f"영토 합계 {terr.sum():.4f} | ZZ_former {fm:.5f}")
    except Exception as e: print("※ country_authorship 읽기 실패, 규모 절 생략:", e)
    (S1 / "TERRITORY_RULES.md").write_text("\n".join(txt), encoding="utf-8"); print("저장: TERRITORY_RULES.md")
    return True


# ============================================================
# 8) recovery  (규칙을 먼저 동결 기록, 그 규칙으로만 적용)
# ============================================================
SPEC_FILE = "RECOVERY_RULES_PREREG.json"
AMB = ["Georgia", "Jordan", "Chad", "Niger", "Guinea", "Malta", "Cuba", "Peru", "Oman", "Togo", "Samoa"]
ALIASES = {"USA": "US", "U.S.A.": "US", "U.S.A": "US", "United States": "US", "United States of America": "US", "New Mexico": "US",
           "UK": "GB", "U.K.": "GB", "U.K": "GB", "United Kingdom": "GB", "England": "GB", "Scotland": "GB", "Wales": "GB",
           "Northern Ireland": "GB", "Great Britain": "GB", "Britain": "GB", "New South Wales": "AU", "Papua New Guinea": "PG",
           "PR China": "CN", "P.R. China": "CN", "P. R. China": "CN", "People's Republic of China": "CN", "People’s Republic of China": "CN", "China": "CN",
           "Hong Kong": "HK", "Taiwan": "TW", "Macau": "MO", "Macao": "MO", "The Netherlands": "NL", "Netherlands": "NL", "Holland": "NL",
           "Russia": "RU", "Russian Federation": "RU", "South Korea": "KR", "Republic of Korea": "KR", "Korea": "KR", "North Korea": "KP",
           "Democratic People's Republic of Korea": "KP", "Democratic People’s Republic of Korea": "KP", "Iran": "IR", "Türkiye": "TR", "Turkey": "TR",
           "Czech Republic": "CZ", "Czechia": "CZ", "Vietnam": "VN", "Viet Nam": "VN", "UAE": "AE", "United Arab Emirates": "AE", "Deutschland": "DE",
           "España": "ES", "Italia": "IT", "México": "MX", "Brasil": "BR"}
US_ABBR = "AL AK AZ AR CA CO CT DE FL GA HI ID IL IN IA KS KY LA ME MD MA MI MN MS MO MT NE NV NH NJ NM NY NC ND OH OK OR PA RI SC SD TN TX UT VT VA WA WV WI WY DC".split()
US_NAMES = ["Alabama", "Alaska", "Arizona", "Arkansas", "California", "Colorado", "Connecticut", "Delaware", "Florida", "Hawaii", "Idaho", "Illinois", "Indiana", "Iowa",
            "Kansas", "Kentucky", "Louisiana", "Maine", "Maryland", "Massachusetts", "Michigan", "Minnesota", "Mississippi", "Missouri", "Montana", "Nebraska", "Nevada",
            "New Hampshire", "New Jersey", "New York", "North Carolina", "North Dakota", "Ohio", "Oklahoma", "Oregon", "Pennsylvania", "Rhode Island", "South Carolina",
            "South Dakota", "Tennessee", "Texas", "Utah", "Vermont", "Virginia", "Washington", "West Virginia", "Wisconsin", "Wyoming"]
CA_ABBR = "ON QC BC AB NS MB NB NL PE SK YT NT NU".split()
CA_NAMES = ["Ontario", "Quebec", "Québec", "British Columbia", "Alberta", "Nova Scotia", "Manitoba", "Saskatchewan", "New Brunswick"]


def make_spec():
    names = {}
    try:
        import pycountry
        for c in pycountry.countries:
            for nm in {c.name, getattr(c, "common_name", c.name)}:
                if len(nm) >= 4 and nm not in AMB and "," not in nm: names[nm] = c.alpha_2
    except Exception: print("※ pycountry 없음: 국가명 목록이 별칭 사전으로만 구성됩니다(pip install pycountry 권장)")
    spec = dict(version=1, created=now(), country_names=names, aliases=ALIASES, ambiguous=AMB, us_abbr=US_ABBR, us_names=US_NAMES, ca_abbr=CA_ABBR, ca_names=CA_NAMES,
                tiers={"1": "국가명·별칭(단어 경계)", "2": "미국 주 약어·주 이름(쉼표 뒤, 문자열 끝 또는 || 앞, 우편번호 허용)", "3": "캐나다 주 이름 또는 주 약어+우편번호"})
    spec["sha256"] = sha_b(json.dumps({k: v for k, v in spec.items() if k not in ("sha256", "created")}, sort_keys=True, ensure_ascii=False).encode())
    return spec


def load_spec():
    p = S1 / SPEC_FILE
    if not p.exists():
        spec = make_spec(); p.write_text(json.dumps(spec, ensure_ascii=False, indent=1), encoding="utf-8")
        print(f"규칙 동결 기록: {SPEC_FILE} (sha256 {spec['sha256'][:12]})"); return spec
    spec = json.loads(p.read_text(encoding="utf-8"))
    chk = sha_b(json.dumps({k: v for k, v in spec.items() if k not in ("sha256", "created")}, sort_keys=True, ensure_ascii=False).encode())
    if chk != spec["sha256"]: raise SystemExit(f"{SPEC_FILE}이 동결 후 변경되었습니다. 변경 사유를 CHANGELOG에 기록하고 새 버전으로 분리하세요")
    return spec


class Recoverer:
    def __init__(self, spec):
        al = {**spec["country_names"], **spec["aliases"]}; self.al = {k.lower(): v for k, v in al.items()}
        alt = "|".join(re.escape(a) for a in sorted(al, key=len, reverse=True))
        self.t1 = re.compile(r"(?<![A-Za-z])(" + alt + r")(?![A-Za-z])", re.I)
        end = r"(?:[\s,]+\d{5}(?:-\d{4})?)?\s*(?=\|\||;|\.|$)"
        self.us_abbr = re.compile(r",\s*(" + "|".join(spec["us_abbr"]) + r")\b" + end)
        self.us_name = re.compile(r",\s*(" + "|".join(sorted(spec["us_names"], key=len, reverse=True)) + r")\b" + end)
        self.ca_name = re.compile(r"(?<![A-Za-z])(" + "|".join(spec["ca_names"]) + r")(?![A-Za-z])")
        self.ca_post = re.compile(r",\s*(" + "|".join(spec["ca_abbr"]) + r")\b[\s,]+[A-Z]\d[A-Z]\s?\d[A-Z]\d")

    def __call__(self, aff):
        s = html.unescape(aff or "").replace("#TAB#", " ").strip()
        if not s: return [], 0
        cs = sorted({self.al[m.lower()] for m in self.t1.findall(s)})
        if cs: return cs, 1
        if self.us_abbr.search(s) or self.us_name.search(s): return ["US"], 2
        if self.ca_name.search(s) or self.ca_post.search(s): return ["CA"], 3
        return [], 0


def st_recovery():
    hdr("[recovery] 국가 미상 규칙 복구 (민감도용, 원 테이블은 변경하지 않음)")
    R = Recoverer(load_spec())
    uc = S1 / "unknown_check.csv"
    if uc.exists():
        U = rd(uc); U = U[(U.missed == "1") & (U.true_country != "")]
        res = [R(a) for a in U.raw_aff]; rec = [bool(c) for c, _ in res]
        ok = [set(c) == set(t.split("|")) for (c, _), t in zip(res, U.true_country)]
        rc = [o for o, r in zip(ok, rec) if r]
        print(f"[표본 점검] 판단 가능 {len(U)}건 중 규칙 복구 {sum(rec)}건 ({sum(rec) / len(U):.1%}), 복구분 정답 {sum(rc)}/{len(rc)}  ※ 규칙을 만든 표본이라 낙관적일 수 있음(in-sample)")
        for (c, t), tc, a in zip(res, U.true_country, U.raw_aff):
            if c and set(c) != set(tc.split("|")): print("  오답:", c, "정답", tc, "|", a[:90])
    A = read_tbl("authorship_level"); A["countries"] = A.countries.fillna(""); A["raw_aff"] = A.raw_aff.fillna("")
    T = A[(A.countries == "") & (A.raw_aff != "")][["pid", "rank", "raw_aff"]].copy()
    out = [R(a) for a in T.raw_aff]; T["countries_rec"] = ["|".join(c) for c, _ in out]; T["rec_tier"] = [t for _, t in out]
    p = S1 / "authorship_level_recovered.csv.gz"; T.to_csv(p, index=False, compression="gzip")
    n, k = len(T), int((T.rec_tier > 0).sum())
    print(f"소속 문자열은 있으나 국가 없음 {n:,}건 중 복구 {k:,}건 ({k / max(n, 1):.1%}) | 티어별 {T.rec_tier.value_counts().sort_index().to_dict()}")
    print("복구 국가 상위:", T[T.countries_rec != ""].countries_rec.value_counts().head(8).to_dict())
    print(f"저장: {p.name}  (민감도 분석에서만 사용. 주 분석 테이블은 그대로)")
    return True


# ============================================================
# 점검용: nauthors_check / income_history / venue_check
# ============================================================
def st_nauthors_check():
    hdr("[nauthors_check] F1 n_authors 출처 점검")
    F = pd.read_csv(DATA / "F1_papers.csv", low_memory=False); F["pid"] = F.pid.astype(str)
    cols = [c for c in F.columns if "author" in c.lower()]; print("author 관련 열:", cols)
    PS = pd.read_csv(S1 / "paper_status.csv", dtype={"pid": str}); X = PS.merge(F[["pid", "n_authors"]], on="pid"); X = X[X.n_oa > 0]
    eq = (X.n_oa == X.n_authors); print(f"수집 논문 {len(X):,}편 중 n_oa == n_authors: {eq.mean():.2%}")
    print("차이 분포(n_oa - n_authors):", (X.n_oa - X.n_authors).describe().round(2).to_dict())
    txt = [c for c in cols if F[c].dtype == object and c != "n_authors"]
    if txt:
        c = txt[0]; est = F[c].fillna("").map(lambda s: 0 if not s.strip() else len(re.split(r";|\|", s))); m = F.n_authors.notna()
        print(f"독립 후보 열 '{c}'의 구분자 개수와 n_authors 일치율: {(est[m] == F.n_authors[m]).mean():.2%}")
    print("해석: 일치율이 100%에 가깝고 F1에 독립적인 저자 목록 열이 없으면, n_authors가 OpenAlex에서 온 값일 가능성이 큽니다(같은 출처끼리의 비교).")
    return True


def st_income_history():
    hdr("[income_history] World Bank 소득군 이력 -> wb_income_history.csv")
    if not INCOME_XLSX.exists(): raise SystemExit(f"{INCOME_XLSX} 없음. World Bank의 'Historical classification by income' 엑셀(OGHIST)을 내려받아 경로를 지정하세요")
    import pycountry
    a3 = {c.alpha_3: c.alpha_2 for c in pycountry.countries}
    xl = pd.ExcelFile(INCOME_XLSX); sh = next((s for s in xl.sheet_names if "analytical" in s.lower() and "history" in s.lower()), xl.sheet_names[1])
    raw = xl.parse(sh, header=None)
    hr = next(i for i in range(len(raw)) if sum(isinstance(v, (int, float)) and 1980 <= v <= 2035 for v in raw.iloc[i]) >= 10)
    years = {j: int(v) for j, v in enumerate(raw.iloc[hr]) if isinstance(v, (int, float)) and not pd.isna(v) and 1980 <= v <= 2035}
    mp = {"L": "LIC", "LM": "LMC", "UM": "UMC", "H": "HIC"}; rows = []
    for i in range(hr + 1, len(raw)):
        r = raw.iloc[i]; code = next((str(v).strip() for v in r.iloc[:3] if isinstance(v, str) and str(v).strip() in a3), None)
        if not code: continue
        for j, y in years.items():
            v = r.iloc[j]
            if isinstance(v, str) and v.strip() in mp: rows.append(dict(iso2=a3[code], year=y, income=mp[v.strip()]))
    H = pd.DataFrame(rows); H.to_csv(S1 / "wb_income_history.csv", index=False)
    print(f"{len(H):,}행, 국가 {H.iso2.nunique()}개, 연도 {H.year.min()}~{H.year.max()} | 시트 '{sh}'")
    print("검산용 예시:", H[(H.iso2.isin(["CN", "KR", "IN"])) & (H.year.isin([2007, 2015, 2024]))].to_dict("records"))
    print("다음: 원 스크립트에서 STAGES = ['build'] 실행(API 호출 없음). 영토·구 국가는 파일에 없을 수 있어 income_tv가 비는 것은 정상")
    return True


def st_venue_check():
    hdr("[venue_check] 저널 창간 연도와 코퍼스 최초 연도")
    F = pd.read_csv(DATA / "F1_papers.csv", low_memory=False); F["pid"] = F.pid.astype(str)
    EXC = set(rd(S0 / "EXCLUDE_PIDS_FROZEN.csv").pid); F = F[~F.pid.isin(EXC)]
    V = pd.read_csv(S1 / "venue_meta.csv"); V = V.merge(F.groupby("venue_short").year.min().rename("min_year"), on="venue_short", how="left")
    V["founding"] = pd.to_numeric(V.E1_founding, errors="coerce"); V["before_founding"] = V.min_year < V.founding
    print(V[["venue_short", "n_papers", "fixed", "founding", "E1_corpus_entry", "min_year", "before_founding", "entry_ruling"]].to_string(index=False))
    print("※ 창간 연도 값은 파일(E1_founding)을 그대로 보여 줄 뿐입니다. 실제 창간 연도는 저널 홈페이지로 확인하고, JIF는 jif_template.csv에 직접 채우세요")
    return True


# ============================================================
# 9) git_freeze
# ============================================================
def git(*a, check=False):
    r = subprocess.run(["git", "-C", str(ROOT), *a], capture_output=True, text=True)
    if check and r.returncode: raise SystemExit(f"git {' '.join(a)} 실패: {r.stderr.strip()}")
    return r


def st_git_freeze():
    hdr("[git_freeze] 산출물 해시 기록 + 커밋 + 태그")
    pj = S0 / "PREREG.json"; h = sha_f(pj)
    print(f"PREREG.json sha256 {h[:8]} (기대 앞부분 {EXPECTED_PREREG_PREFIX})")
    if not h.startswith(EXPECTED_PREREG_PREFIX) and not IGNORE_PREREG_HASH_MISMATCH:
        raise SystemExit("PREREG 해시가 기대값과 다릅니다. 보정 이후 또 수정됐는지 확인하거나, 해시 산출 방식이 다르면 IGNORE_PREREG_HASH_MISMATCH=True로 두세요")
    keys = ["manual_verify.csv", "unknown_check.csv", "target_lens_review.csv", SRC_FILE, "SOURCE_CHECK_SUMMARY.json", "target_lens_precision.json", "validation_report.md",
            "coverage_by_year.csv", "WINDOW_DRAFT.json", "paper_status.csv", DOC, SPEC_FILE, "TERRITORY_RULES.md", "CHANGELOG_S1.md"]
    M = [dict(file=f"01_S1/{f}", sha256=sha_f(S1 / f)) for f in keys if (S1 / f).exists()] + [dict(file="00_S0/PREREG.json", sha256=h)]
    missing = [f for f in ["validation_report.md", "SOURCE_CHECK_SUMMARY.json"] if not (S1 / f).exists()]
    if missing: print("⚠ 아직 없는 핵심 산출물:", missing, "(evaluate, source_ingest를 먼저 실행하는 편이 좋습니다)")
    pd.DataFrame(M).assign(at=now()).to_csv(S1 / "S1_FREEZE_MANIFEST.csv", index=False); print(f"S1_FREEZE_MANIFEST.csv {len(M)}개 해시 기록")
    top = git("rev-parse", "--show-toplevel")
    if top.returncode: raise SystemExit("git 저장소가 아닙니다: " + top.stderr.strip())
    spec = ["--", "paper1/00_S0", "paper1/01_S1", ":(exclude)paper1/01_S1/oa_raw", ":(exclude)paper1/01_S1/_backup_*"]
    print(git("status", "--short", *spec).stdout or "(변경 없음)")
    tag = TAG_NAME or f"S0S1-freeze-{pd.Timestamp.now().strftime('%Y-%m-%d')}"
    if DRY_RUN:
        print(f"[DRY_RUN] 실제로는 git add {' '.join(spec)} / git commit / git tag -a {tag} 를 실행합니다. DRY_RUN=False로 바꾸세요")
        print("※ oa_raw(원자료)는 용량 때문에 제외했습니다. 재현에 필요하면 별도 보관(해시는 manifest 참고)")
        return True
    if git("tag", "-l", tag).stdout.strip(): raise SystemExit(f"태그 {tag}가 이미 있습니다")
    git("add", *spec, check=True)
    c = git("commit", "-m", "S0/S1 freeze: 검수 3종(AI 사전 판정+연구자 확인+원문 대조), 규칙 동결, 보정 PREREG 포함")
    print(c.stdout.strip() or c.stderr.strip())
    if c.returncode and "nothing to commit" not in (c.stdout + c.stderr): raise SystemExit("커밋 실패")
    git("tag", "-a", tag, "-m", "S0/S1 freeze", check=True); print(f"태그 생성: {tag} (원격에 올리려면 git push --follow-tags)")
    return True


FUNCS = {"install": st_install, "source_sample": st_source_sample, "source_ingest": st_source_ingest, "lens_precision": st_lens_precision,
         "changelog": st_changelog, "territory_doc": st_territory_doc, "recovery": st_recovery, "nauthors_check": st_nauthors_check,
         "income_history": st_income_history, "venue_check": st_venue_check, "git_freeze": st_git_freeze}

if __name__ == "__main__":
    DATA, ROOT, S0, S1 = paths()
    print(f"ROOT={ROOT} | STEPS={STEPS}")
    for s in STEPS:
        if FUNCS[s]() is False: break

ROOT=/home/yjlee/Research/Minnesota_Transport | STEPS=['source_sample']

[source_sample] 원문 대조 표본
manual_source_check.csv 41행 생성(seed=2026, 강제 포함 ['25038']).
채우는 법: 논문 페이지에서 rank번째(0부터) 저자의 소속을 찾아 source_aff_seen에 적고,
  source_countries에 국가 코드(복수면 |), source_ok에는 countries가 원문과 일치하면 1, 아니면 0.


In [18]:
#

In [23]:
# build_income.py  (OpenAlex 호출 없음)
from pathlib import Path
import pandas as pd, pycountry

# ---------------- 설정 ----------------
S1 = Path.home() / "Research/Minnesota_Transport/paper1/01_S1"
SRC = S1 / "OGHIST_2026_07_15.xlsx"
YEAR_BASIS = "data"      # "data" = GNI 산정 달력연도 / "fy" = 세계은행 회계연도 (S7 결과 보기 전에 확정)
# --------------------------------------

assert SRC.exists(), f"{SRC} 없음"
a3 = {c.alpha_3: c.alpha_2 for c in pycountry.countries}
a3.update({"XKX": "XK", "ROM": "RO", "ZAR": "CD", "TMP": "TL", "WBG": "PS",
           "SUN": "SU", "YUG": "YU", "YUGf": "YU", "CSK": "CS", "ANT": "AN"})
mp = {"L": "LIC", "LM": "LMC", "UM": "UMC", "H": "HIC"}

d = pd.read_excel(SRC, sheet_name="Country Analytical History", header=None)
fy = {j: int(str(v)[2:]) for j, v in d.iloc[4].items() if str(v).startswith("FY")}
fy = {j: (1900 + x if x > 50 else 2000 + x) for j, x in fy.items()}
cy = {j: int(d.iloc[5, j]) for j in fy}

rows, unm = [], []
for i in range(6, len(d)):
    code = d.iloc[i, 0]
    if not isinstance(code, str): continue
    code = code.strip()
    if code not in a3: unm.append((code, d.iloc[i, 1])); continue
    for j in fy:
        v = d.iloc[i, j]
        if isinstance(v, str):
            v = v.strip().rstrip("*")
            if v in mp: rows.append(dict(iso2=a3[code], year_data=cy[j], fy=fy[j], income=mp[v]))
H = pd.DataFrame(rows)

col = "year_data" if YEAR_BASIS == "data" else "fy"
out = S1 / "wb_income_history.csv"
if out.exists() and len(out.read_text(encoding="utf-8").splitlines()) > 1:
    out.replace(S1 / f"wb_income_history.bak_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}.csv")
H[["iso2", col, "income"]].rename(columns={col: "year"}).to_csv(out, index=False)
H.to_csv(S1 / "wb_income_history_long.csv", index=False)       # 두 연도 열이 모두 있는 참조용

print(f"저장: {out} | 기준 {YEAR_BASIS} | {len(H):,}행, 국가 {H.iso2.nunique()}개")
print("미매칭 코드:", unm, "(CHI = Channel Islands 하나만 나오면 정상)")

# 검산: FY2027 페이지 값과 대조
exp = {"CN": "UMC", "KR": "HIC", "US": "HIC", "TW": "HIC", "IN": "LMC", "NG": "LMC", "AF": "LIC", "VE": "LMC", "AR": "UMC", "TR": "UMC"}
got = H[H.fy == 2027].set_index("iso2").income.to_dict()
bad = [k for k, v in exp.items() if got.get(k) != v]
print("FY27 검산:", "전부 일치" if not bad else f"불일치 {bad}")
print("FY27 분포(기대 HIC 86, UMC 59, LMC 47, LIC 25):", H[H.fy == 2027].income.value_counts().to_dict())

저장: /home/yjlee/Research/Minnesota_Transport/paper1/01_S1/wb_income_history.csv | 기준 data | 8,056행, 국가 222개
미매칭 코드: [('CHI', 'Channel Islands')] (CHI = Channel Islands 하나만 나오면 정상)
FY27 검산: 전부 일치
FY27 분포(기대 HIC 86, UMC 59, LMC 47, LIC 25): {'HIC': 86, 'UMC': 59, 'LMC': 47, 'LIC': 25}


In [24]:
#

In [31]:
# -*- coding: utf-8 -*-
# s1_build_standalone.py : v3.2의 build 단계만 독립 실행 (OpenAlex 호출 없음, API 키 불필요)
import json, re, hashlib, warnings
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, requests
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)

# ---------------- 설정 ----------------
ROOT = Path.home() / "Research/Minnesota_Transport"
WINDOW = 2007                      # 확정 분석 창 시작 연도
# --------------------------------------
DATA = ROOT / "analysis_data/final"
S0, S1 = ROOT / "paper1/00_S0", ROOT / "paper1/01_S1"
RAW = S1 / "oa_raw"
assert (DATA / "F1_papers.csv").exists(), f"{DATA}/F1_papers.csv 없음. ROOT를 확인하세요"
assert (RAW / "works_by_doi.jsonl").exists(), f"{RAW}/works_by_doi.jsonl 없음"

now = lambda: pd.Timestamp.now().isoformat(timespec="seconds")
sha = lambda b: hashlib.sha256(b).hexdigest()

def changelog_once(key, line):
    cl = S1 / "CHANGELOG_S1.md"
    if not cl.exists(): cl.write_text("# CHANGELOG S1 (날짜 | 변경 | 사유 | 영향)\n\n", encoding="utf-8")
    if key in cl.read_text(encoding="utf-8"): return
    with cl.open("a", encoding="utf-8") as f: f.write(line + "\n")

def nd(s):
    if not isinstance(s, str) or not s.strip(): return None
    s = s.strip().lower(); s = re.sub(r"^(https?://(dx\.)?doi\.org/|doi:\s*)", "", s); return s or None

# ---------------- F1X ----------------
F1 = pd.read_csv(DATA / "F1_papers.csv", low_memory=False); F1["pid"] = F1.pid.astype(str)
exf = S0 / "EXCLUDE_PIDS_FROZEN.csv"
log = pd.read_csv(S0 / "FREEZE_LOG.csv"); last = log[log.file == exf.name].iloc[-1]
assert sha(exf.read_bytes()) == last.sha256, "EXCLUDE_PIDS_FROZEN.csv 해시가 동결 기록과 다릅니다"
EXC = set(pd.read_csv(exf, dtype=str, encoding="utf-8-sig").pid)
F1X = F1[~F1.pid.isin(EXC)].copy(); F1X["doi_use"] = F1X.doi_norm.map(nd)
pf = S0 / "doi_fixes.csv"
if pf.exists():
    fx = pd.read_csv(pf, dtype=str, encoding="utf-8-sig"); m = dict(zip(fx.pid.astype(str), fx.doi_fixed.map(nd)))
    F1X["doi_use"] = [m.get(p, d) for p, d in zip(F1X.pid, F1X.doi_use)]
print(f"F1 {len(F1):,} -> F1X {len(F1X):,} | DOI 보유 {F1X.doi_use.notna().mean():.1%}")

# ---------------- 원자료 읽기 ----------------
def slim(w):
    return {"doi": w.get("doi"),
            "authorships": [{"author": {"id": (a.get("author") or {}).get("id")}, "author_position": a.get("author_position"),
                             "institutions": [{"country_code": i.get("country_code")} for i in (a.get("institutions") or [])],
                             "countries": a.get("countries"), "raw_affiliation_strings": (a.get("raw_affiliation_strings") or [])[:2]}
                            for a in (w.get("authorships") or [])]}

def load_raw():
    W = {}
    for l in (RAW / "works_by_doi.jsonl").open(encoding="utf-8"):
        w = json.loads(l); W[nd(w.get("doi"))] = slim(w)
    mp = RAW / "missing_dois.txt"
    return W, (set(mp.read_text().split()) if mp.exists() else set())

TERRITORY = {"HK", "MO", "TW", "PR", "GU", "VI", "AS", "MP", "GI", "FO", "GL", "AW", "CW", "SX", "NC", "PF", "RE", "GP", "MQ", "GF", "YT", "KY", "BM", "XK", "PS", "EH"}
FORMER = {"SU", "YU", "CS", "DD", "AN"}
def cn(c): return "ZZ_former" if c in FORMER else c
def countries_of(a):
    cs = {i.get("country_code") for i in (a.get("institutions") or []) if i.get("country_code")}
    if not cs: cs = set(a.get("countries") or [])
    return sorted(cn(c) for c in cs if c)

def save_tbl(df, name):
    try: df.to_parquet(S1 / f"{name}.parquet", index=False); return f"{name}.parquet"
    except Exception: df.to_csv(S1 / f"{name}.csv.gz", index=False, compression="gzip"); return f"{name}.csv.gz"

def wb_income():
    p = S1 / "wb_income_current.csv"
    if not p.exists():
        j = requests.get("https://api.worldbank.org/v2/country", params={"format": "json", "per_page": 400}, timeout=60).json()[1]
        pd.DataFrame([dict(iso2=c["iso2Code"], name=c["name"], income=c["incomeLevel"]["id"], region=c["region"]["id"]) for c in j]).to_csv(p, index=False)
    Wb = pd.read_csv(p, keep_default_na=False)
    return Wb[(Wb.income != "INX") & (Wb.region != "NA")]

# ---------------- build ----------------
y0 = WINDOW
W, miss = load_raw(); att = set(W) | miss
D = F1X[F1X.year >= y0].copy(); D["has_doi"] = D.doi_use.notna(); D["att"] = D.doi_use.isin(att)
n_doi = D[D.has_doi].groupby(["year", "venue_short"]).size(); n_att = D[D.att].groupby(["year", "venue_short"]).size()
cw = (n_doi / n_att).rename("samp_w"); D = D.join(cw, on=["year", "venue_short"])
D["samp_w"] = np.where(D.has_doi, D.samp_w, 1.0)
D = D[D.att | ~D.has_doi]
P, A, ST = [], [], []
for r in D.itertuples():
    w = W.get(r.doi_use) if r.has_doi else None
    au = (w or {}).get("authorships") or []; n = len(au)
    if n == 0:
        P.append((r.pid, "UNK", 1.0)); ST.append((r.pid, w is not None, "doi" if w else "", 0, False, 0.0, r.samp_w)); continue
    cnt, known = Counter(), 0
    for k, a in enumerate(au):
        cs = countries_of(a); aff = " || ".join((a.get("raw_affiliation_strings") or [])[:2])[:300]
        A.append((r.pid, k, (a.get("author") or {}).get("id"), a.get("author_position"), "|".join(cs), aff, n == r.n_authors))
        if cs:
            known += 1
            for c in cs: cnt[c] += 1.0 / (n * len(cs))
        else: cnt["UNK"] += 1.0 / n
    P += [(r.pid, c, v) for c, v in cnt.items()]; ST.append((r.pid, True, "doi", n, n == r.n_authors, known / n, r.samp_w))
P = pd.DataFrame(P, columns=["pid", "country", "w"])
A = pd.DataFrame(A, columns=["pid", "rank", "oa_author_id", "author_position", "countries", "raw_aff", "aligned"])
ST = pd.DataFrame(ST, columns=["pid", "matched", "how", "n_oa", "aligned", "known_share", "samp_w"])
inc = wb_income().set_index("iso2").income
P["is_territory"] = P.country.isin(TERRITORY); P["income_fixed"] = P.country.map(inc).fillna("UNK")
P = P.merge(F1X[["pid", "year", "venue_short"]], on="pid").merge(ST[["pid", "samp_w", "aligned"]], on="pid")
hist = S1 / "wb_income_history.csv"
assert hist.exists(), f"{hist} 없음: 먼저 소득군 이력 파일을 만드세요"
H = pd.read_csv(hist, keep_default_na=False)
P = P.merge(H.rename(columns={"income": "income_tv"}), left_on=["country", "year"], right_on=["iso2", "year"], how="left").drop(columns="iso2")
print("저장:", save_tbl(P, "country_authorship"), save_tbl(A, "authorship_level")); ST.to_csv(S1 / "paper_status.csv", index=False)
wm = np.average(ST.known_share, weights=ST.samp_w); mt = ST[ST.n_oa > 0]
print(f"창 {y0}~ | 수집·포함 논문 {len(ST):,} (DOI 없는 {int((~D.has_doi).sum()):,}) | 가중 평균 국가 보유율 {wm:.1%} | samp_w 범위 {ST.samp_w.min():.2f}~{ST.samp_w.max():.2f}")
print(f"저자 수 정렬 일치: {mt.aligned.mean():.1%} (수집 논문 {len(mt):,}편)")
ps = (P.w * P.samp_w).groupby(P.country).sum(); print("국가 몫 상위:"); print((ps / ps.sum()).sort_values(ascending=False).head(10).round(4))
changelog_once("build: 창", f"- {now()} | build: 창 {y0}~, 표본 가중치 samp_w, 다중 소속 균등 분할, unknown 칸 유지, aligned 열 추가 | S7")
changelog_once("income_tv 부착", f"- {now()} | build 재실행: 시변 소득군(OGHIST_2026_07_15, 산정 달력연도 기준) 부착, 기존 테이블은 _backup_before_income_tv에 보관 | 시변 소득군 | S7 소득군 분석")

# ---------------- 확인 ----------------
P["ws"] = P.w * P.samp_w
miss_ = P[P.income_tv.isna()]
print("\n결측 몫(income_tv):", round(miss_.ws.sum() / P.ws.sum(), 4))
print((miss_.groupby("country").ws.sum() / P.ws.sum()).sort_values(ascending=False).head(10).round(4))
print("연도별 결측 몫:", (miss_.groupby("year").ws.sum() / P.groupby("year").ws.sum()).round(3).to_dict())

F1 111,685 -> F1X 111,416 | DOI 보유 89.6%
저장: country_authorship.parquet authorship_level.parquet
창 2007~ | 수집·포함 논문 81,645 (DOI 없는 160) | 가중 평균 국가 보유율 94.7% | samp_w 범위 1.00~1.00
저자 수 정렬 일치: 100.0% (수집 논문 81,441편)
국가 몫 상위:
country
US     0.2546
CN     0.2170
UNK    0.0531
GB     0.0434
CA     0.0410
AU     0.0386
DE     0.0304
NL     0.0276
IT     0.0203
HK     0.0203
dtype: float64

결측 몫(income_tv): 0.0531
country
UNK    0.0531
ET     0.0000
RE     0.0000
Name: ws, dtype: float64
연도별 결측 몫: {2007: 0.127, 2008: 0.107, 2009: 0.1, 2010: 0.094, 2011: 0.085, 2012: 0.077, 2013: 0.069, 2014: 0.057, 2015: 0.061, 2016: 0.06, 2017: 0.056, 2018: 0.044, 2019: 0.04, 2020: 0.037, 2021: 0.037, 2022: 0.038, 2023: 0.035, 2024: 0.041, 2025: 0.049}


In [32]:
# 결과는 정상으로 보이지만, ET 한 건이 소득군이 비어 있어서 OGHIST 원본을 확인하겠습니다.`build`와 `income_tv` 부착이 모두 정상적으로 끝났습니다.
# **주 분석 값이 그대로인지**
# - 국가 몫이 이전과 같습니다. US 25.5%, CN 21.7%, GB 4.3%이고 국가 보유율도 94.7%입니다. 소득군 열만 추가되었다는 뜻입니다.
# - 수집·포함 논문은 81,645편(DOI 없는 160편 포함)이고, 저자 수 정렬은 100%입니다.
# **`income_tv` 결측**
# - 결측 몫은 5.31%로, `UNK`의 몫(5.31%)과 같습니다. 국가가 없는 몫이 소득군도 없는 것이라 당연한 결과입니다.
# - 실제 국가에서 빠진 것은 ET(에티오피아)와 RE(레위니옹)뿐이고 둘 다 몫이 0.0000입니다.
# - **RE:** OGHIST에 없는 영토입니다. 정상입니다.
# - **ET:** OGHIST에는 있는데 **2024년 한 칸만 비어 있습니다**(그 해만 결측값). 이것이 원본 자료의 빈칸이라서, 값을 임의로 채우지 않고 결측으로 둔 상태가 맞습니다. 몫이 0에 가깝기 때문에 분석에는 영향이 없고, 소득군 열의 결측 처리 규칙에만 한 줄 기록해 두세요.
# - HK와 TW는 OGHIST에 있어서 결측이 아닙니다. 앞서 "영토는 비는 것이 정상"이라고 한 말은 이 파일에서는 해당되지 않았습니다.
# - 연도별 결측 몫에 2026년이 없습니다. 이 코퍼스는 2025년까지라서 앞서 걱정한 2026년 문제는 없습니다.
# **연도별 결측 몫이 눈에 띕니다**
# - 2007년 12.7%에서 2025년 4.9%까지 낮아지고, 최저는 2023년의 3.5%입니다.
# - 이 값은 사실상 국가 미상(`UNK`) 몫의 시간 추이입니다. 같은 해에 따라 국가 미상 비중이 3~4배 차이 난다는 뜻이어서, 시계열 분석에서 "소득군별 몫의 변화"가 일부는 단순한 측정 누락 변화일 수 있습니다.
# - 지금 단계에서 할 일은 없습니다. 다만 S2와 S7에서 국가 미상 몫을 연도별로 같이 보고하고, 앞서 준비한 `recovery` 단계(민감도용 복구)의 결과로 추이가 달라지는지 확인하는 것이 좋겠습니다.
# **CHANGELOG와 백업**
# - `build`가 CHANGELOG에 "income_tv 부착" 항목을 기록했습니다.
# - 이전 테이블은 `_backup_before_income_tv/`에 있습니다. 위 값이 이전과 일치하므로 필요 없다면 나중에 정리하셔도 됩니다. `git_freeze` 전에는 지우지 않고 그대로 두어도 `_backup_*` 규칙에 걸려 커밋에서 제외됩니다.
# **남은 일**
# - 소득군 CSV는 완료입니다. 아직 원문 대조 `manual_source_check.csv` 41건이 1순위로 남아 있습니다.
# - 그다음은 `source_ingest` → `evaluate` → `lens_precision` → `changelog` → `git_freeze` 순입니다.
# 다음으로 `venue_founding`이나 JIF 쪽을 진행하시겠다면, 원 스크립트의 `samples`가 만든 `01_S1/venue_meta.csv`에서 `E1_founding` 열 값을 붙여 주세요. 저널별 창간 연도 확인 목록을 만들어 드리겠습니다.

In [ ]:
# ## 지금까지 한 일
# ### S0 사전등록·동결 (완료, 보정 1건 남음)
# - 오염 레코드 269편 제외(F1 111,685편 → F1X 111,416편), 창간 전 논문 검토, E1 충돌 판정(RailEng 2020, TRR 1967).
# - 고정 저널 14개 확정, 사건 목록·우연 적중률 진단. 주 시나리오는 NONE이라 연속 노출 시계열이 주 분석이고 사건 대조는 민감도입니다.
# - 동결 검사 32건 전부 PASS, 해시 9개 기록.
# - 동결 후 위험군 변형 7종 이름을 보정했고(PREREG 해시 `eb2f284f` → `f16e76b4`), CHANGELOG에 사유를 남겼습니다. **다시 커밋하는 일은 확인되지 않았습니다.**

# ### S1 국가 층 (자동 단계, 시변 소득군, 검수 시트 판정 완료, 원문 대조 진행 중)
# - 표본 점검(probe) 후 분석 창을 **2007년**으로 확정했습니다. 사유는 CHANGELOG_S1에 기록했습니다.
# - 창 안 DOI 보유 논문 81,485편을 전수 수집했습니다. 931회 요청, 약 $0.093이 들었습니다.
# - 국가 층을 구축했습니다(`country_authorship`, `authorship_level`).
#   - 포함 논문은 81,645편(DOI 없는 160편은 국가 미상으로 포함)입니다.
#   - 가중 평균 국가 보유율 94.7%, 국가 미상 5.3%입니다.
#   - 상위 몫은 US 25.5%, CN 21.7%, GB 4.3%입니다.
#   - 정렬 일치율은 100%인데 같은 출처끼리의 비교일 가능성이 있습니다.
# - **시변 소득군을 부착했습니다(2026-10-06).**
#   - 원자료는 `OGHIST_2026_07_15.xlsx`(Country Analytical History 시트)입니다. `build_income.py`로 `wb_income_history.csv`를 만들었습니다(8,056행, 222개 국가, 산정 달력연도 1987~2025).
#   - 검산은 FY2027 페이지의 분포(HIC 86, UMC 59, LMC 47, LIC 25)와 10개국 값이 모두 일치했습니다. 미매칭 코드는 CHI(Channel Islands)뿐입니다.
#   - 독립 실행본 `s1_build_standalone.py`로 `build`를 다시 실행했습니다. 국가 몫과 보유율은 이전 값과 같았고 `income_tv` 열이 추가되었습니다. 이전 테이블은 `_backup_before_income_tv/`에 보관했습니다.
#   - `income_tv` 결측 몫은 5.31%로 국가 미상(`UNK`)의 몫과 같습니다. 실제 국가 중 결측은 ET(OGHIST 2024년 한 칸이 원본에서 빔)와 RE(OGHIST에 없음)뿐이고 몫은 0.0000입니다. HK와 TW는 OGHIST에 있어 결측이 아닙니다.
#   - 값을 임의로 채우지 않았습니다.
# - 본 수집 후 audit를 다시 돌렸습니다. 합산 규칙은 2007로 유지되고, 고정 저널 266칸 중 0.80 미만은 1칸(TranspPol 2008, 0.800)입니다. 엄격 규칙은 2009입니다.
# - 대상 렌즈(제목 지명) 보유율은 창 안 5.7~7.1%입니다.
# - **검수 시트 3종 판정을 채웠고 `01_S1`에 이미 들어 있습니다.** Claude가 `raw_aff`와 제목을 읽고 사전 판정했고, 연구자가 전체를 검토해 적절하다고 확인했습니다(2026-10-06). 연구자가 `01_S1`의 시트 분포를 직접 확인했습니다(`manual_verify` ok=1 200건, `unknown_check` missed 1:97·0:3과 `true_country` 97건, `target_lens_review` 1:195·0:5).
# - **판정 기준을 문서로 기록했습니다**(`S1_검수판정_기준_및_확인기록.md`). 사전 판정 후, `evaluate` 실행 전에 작성했으므로 "판정 전 기록"이 아닙니다. 이 문서가 `01_S1`에 있는지는 확인이 필요합니다(`source_ingest`가 6절을 자동으로 채우려면 그 폴더에 있어야 함).
# - **후속 작업 코드 `s1_next_steps.py`**를 만들었습니다. 가짜 폴더에서 `install`, `source_sample`, `source_ingest`, `lens_precision`, `changelog`, `territory_doc`, `recovery`를 시험했습니다. `nauthors_check`, `venue_check`, `git_freeze`는 문법 검사만 했고 실행은 해 보지 못했습니다. 실제 자료에서는 `source_sample`만 실행했습니다.
#   - **`income_history` 단계는 쓰지 마세요.** OGHIST 구조(회계연도 행과 달력연도 행이 따로 있음, 구 WB 코드 포함)에 맞지 않아 `build_income.py`로 대체했습니다.
# - **원문 대조 표본을 만들었습니다**(`manual_source_check.csv`, 41행, seed 2026, 25038 포함). 지금 사람이 채우는 단계입니다.

# | 시트 | 결과 | Wilson 95% 구간 | 비고 |
# |---|---|---|---|
# | `manual_verify` 200건 | 정확 200, 오류 0 (점추정 1.000) | [0.981, 1.000] | 기준(0.95, 하한 0.90) 충족. 25038(`CA\|LB\|TW`)은 웹으로 대만 소속 가능성만 확인했고 논문 소속란은 직접 보지 못함 |
# | `unknown_check` 100건 | 국가 판단 가능(누락) 97, 불가 3 | [0.915, 0.990] | 불가 3건은 65744, 84931, 68765. 가능한 97건은 `true_country`에 복구 국가를 기입 |
# | `target_lens_review` 200편 | 정확 195, 오류 5 (0.975) | [0.943, 0.989] | 오류는 90265, 100997, 101469, 74350, 11912. 일부 논문은 대상 지역이 누락됨(94445 Britain, 17180 Europe, 77297 Sydney). 이는 정밀도가 아니라 재현율 문제 |

# ## 앞으로 할 일

# ### A. S1 마무리 (우선순위 순)
# | 순위 | 할 일 | 실행 단계 | 상태 | 비고 |
# |---|---|---|---|---|
# | 1순위-1 | 판정 기준 문서화 | (수작업) | **완료** | 기관 소재지 기준, 해외 분교는 분교 소재지, 영토(HK, TW, PR)는 OpenAlex 코드 그대로, 대상 렌즈는 "매칭 국가가 연구 대상 지역이면 1" |
# | 1순위-2 | 완성본 3종을 `01_S1`에 설치 | `install` | **완료(이미 들어 있음)** | `install`은 실행하지 않아도 됨. 연구자가 만든 것으로 보이는 `manual_verify_backup.csv`가 있으나 내용은 확인하지 않음 |
# | 1순위-3 | **원문 대조 표본 만들기** | `source_sample` | **완료** | `manual_source_check.csv` 41행 |
# | 1순위-4 | **원문 대조 41건 채우기** (`source_aff_seen`, `source_countries`, `source_ok`, `note`) | (수작업) | **진행 중, 가장 큰 일** | 25038을 먼저. 유료 논문은 `source_ok`를 비우고 `note`에 이유. 30건 이상 채우는 것이 좋음 |
# | 1순위-5 | 대조 결과 집계, 불일치 반영, 문서 6절 기입 | `source_ingest` | 남음 | 먼저 `CHECKER`, `SOURCE_METHOD` 설정. 기록 문서가 `01_S1`에 있어야 6절 자동 기입 |
# | 1순위-6 | `STAGES = ["evaluate"]` 실행 | (원 스크립트) | 남음 | `validation_report.md` 생성. 불일치를 반영했다면 반드시 이 뒤에 실행 |
# | 1순위-7 | CHANGELOG_S1 기록 | `changelog` | 남음 | `source_ingest` 이후 실행(원문 대조 건수 필요). `lens_precision`을 먼저 실행하면 렌즈 정밀도 항목도 함께 기록됨 |
# | 2순위-8 | 대상 렌즈 정밀도 계산 | `lens_precision` | 남음 | `evaluate`에 포함되지 않음. 현재 값 0.975 [0.943, 0.989] |
# | 2순위-9 | 국가 미상 규칙 기반 복구를 동결 기록하고 민감도용으로 적용 | `recovery` | 남음 | 실행 전에 규칙(별칭, 주 약어)을 검토. 동결 뒤에는 변경 불가. S7 결과를 보기 전에 실행. 가짜 폴더에서 표본 점검은 97/97이었으나 규칙을 만든 표본이라 낙관적. `pycountry` 설치 필요. 도시 사전을 규칙에 넣으려면 동결 전에 |
# | 2순위-10 | 영토·구 국가 표기 규칙 문서화 | `territory_doc` | 남음 | HK, TW 등. S7 결과를 보기 전에 |
# | 2순위-11 | **소득군 연도 기준 문서화** | (수작업, CHANGELOG) | **새 항목** | `income_tv`는 산정 달력연도 기준으로 부착함(OGHIST의 FY = 산정연도 + 2). 회계연도 기준은 `wb_income_history_long.csv`로 민감도 가능. 사유와 함께 S7 결과를 보기 전에 기록. 결측 규칙(ET 2024는 원본 빈칸, 채우지 않음)도 같이 |
# | 2순위-12 | S0와 S1 산출물 git commit과 태그 | `git_freeze` | 남음 | 기본 `DRY_RUN=True`. 보정된 PREREG(`f16e76b4`) 해시를 확인함. 1순위-7 직후를 권장. **해시 목록(`keys`)에 `wb_income_history.csv`, `wb_income_history_long.csv`가 없으니 추가할 것.** `build_income.py`, `s1_build_standalone.py`를 같은 폴더에 보관한다면 함께 |
# | 3순위-13 | F1의 `n_authors` 출처 확인 | `nauthors_check` | 남음 | 정렬 100%의 해석에 필요. 실행 결과를 직접 해석해야 함 |
# | 3순위-14 | 시변 소득군 파일 작성 후 build 재실행 | `build_income.py`, `s1_build_standalone.py` | **완료** | 위 "S1" 절 참조 |
# | 3순위-15 | 저널 창간 연도 확인 | `venue_check` + (수작업) | 남음 | `venue_founding.csv` 34개 저널이 비어 있음. 맥락 변수라 본 분석을 막지 않음. 홈페이지로 직접 확인하고 `source_url`, `checked_on` 기입 |
# | 3순위-16 | JIF 수집 | (수작업, JCR) | 남음 | `01_S1/jif_template.csv`에 저널×2000~2025년 빈 골격이 이미 있음. 구독이 없으면 빈칸으로 두고 한계로 기록 |
# | 3순위-17 | 도시 사전 `gazetteer_cities.csv` | (선택) | 남음, 안 채워도 됨 | **채운 뒤 원 스크립트의 `samples`를 다시 실행하면 `manual_verify.csv`, `unknown_check.csv`, `target_lens_review.csv`가 새로 만들어져 지금까지의 판정이 사라집니다.** 반드시 백업하고, 가능하면 `samples` 전체가 아니라 대상 렌즈 부분만 따로 실행할 것 |

# 시간이 부족하면 1순위만 하고 2순위는 S2를 시작하기 전에 하세요.
# **CHANGELOG에 들어갈 내용** (`changelog` 단계가 자동으로 기록합니다. 아래 항목은 수동 기록)
# - audit 재계산 결과.
# - fuzzy_probe 생략(창 안 DOI 없는 논문 160편).
# - 대상 렌즈 보조 결정과 재현율 한계.
# - 검수 3종 사전 판정을 Claude가 수행했고 연구자가 확인했다는 기록(원문 대조 n=__건).
# - 판정 기준 문서가 사전 판정 후에 작성되었다는 사실.
# - 시변 소득군 부착(`build` 재실행 시 자동 기록됨)과 연도 기준(산정 달력연도) 결정 사유.

# 통과하면 RQ4·RQ5를 본 논문에 유지합니다. 실패하면 창을 줄이거나 부록으로 내립니다.

# ### B. S1 이후 (계획서 기준)
# 1. **S2 측정 교정:** 가중치 4종, 위험군 재현율, 토픽 결측 21% 성향 가중, 비연구 항목 제거, K 민감도. **국가 미상 몫이 연도별로 크게 다릅니다**(2007년 12.7%, 2014년 5.7%, 2023년 3.5%, 2025년 4.9%). 소득군별 몫의 시간 변화에 측정 누락 변화가 섞일 수 있어서, 국가 미상 몫을 연도별로 함께 보고하고 `recovery` 민감도로 추이가 유지되는지 확인해야 합니다.
# 2. **S3~S4:** 토픽 증거 카드와 신뢰 등급, 양×질 4분면, 저널 구성 분해.
# 3. **S5 체제 식별(핵심):** 연속 노출 시계열이 주 분석이고 사건 시나리오는 민감도입니다. 지리 변화점은 국가 층이 2007년부터라서 19개 연도만 쓰게 됩니다.
# 4. **S6~S8:** 갱신 분해, 지역 집중, 공급-부담 정렬. S8은 정책 도메인 사전 이중 코딩이라 코더 2명이 필요합니다.
# 5. **S9~S10:** 후보 풀과 추출 규칙 동결, 투고 패키지. S10.4 생성형 AI 사용 선언에는 검수 3종 사전 판정 범위를 그대로 적어야 합니다.

# ## 지금 가장 시간이 걸리는 일

# **`manual_source_check.csv` 41건의 논문 원문 대조**입니다. 한 건에 2~5분으로 잡으면 1.5~3시간쯤 걸립니다. 지금까지의 판정은 파일 안 텍스트끼리의 일치만 본 것이라, 계획서가 요구하는 원문 대조가 S1 통과 판정에서 가장 약한 고리입니다.

# - 진행 순서는 원문 대조 채우기, `source_ingest`, `evaluate`, `lens_precision`, `changelog`, `git_freeze` 순입니다.
# - 끝나면 `evaluate` 출력을 붙여 주세요. S1 종료 여부를 같이 판단하겠습니다.

In [36]:
#

In [37]:
# -*- coding: utf-8 -*-
"""
manual_source_check.csv 검증 (읽기 전용: 어떤 파일도 수정하지 않음, 크레딧 소모 없음)
실행: python check_source_sheet.py   (source_ingest 실행 전에 돌리세요)
종료 코드: ERROR가 있으면 1, 없으면 0
"""
import math, re
from pathlib import Path
import pandas as pd

# ---------------- 설정 ----------------
S1 = Path.home() / "Research/Minnesota_Transport/paper1/01_S1"
SRC = S1 / "manual_source_check.csv"
BASE = S1 / "manual_verify.csv"
FORCE_PIDS = ["25038"]
MIN_JUDGED = 30            # 권장 최소 판정 건수
# --------------------------------------

errs, warns = [], []
E = lambda m: errs.append(m)
W = lambda m: warns.append(m)
rd = lambda p: pd.read_csv(p, dtype=str, encoding="utf-8-sig").fillna("")

def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n)
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)); return (c - h) / d, (c + h) / d

try:
    import pycountry
    VALID = {c.alpha_2 for c in pycountry.countries} | {"XK", "ZZ_former"}
except Exception:
    VALID = None; W("pycountry 없음: 국가 코드 유효성 검사를 건너뜀 (pip install pycountry)")

assert SRC.exists(), f"{SRC} 없음"
X = rd(SRC)
need = ["pid", "rank", "year", "venue_short", "doi_url", "title", "raw_aff", "countries",
        "source_aff_seen", "source_countries", "source_ok", "note"]
miss = [c for c in need if c not in X.columns]
if miss: raise SystemExit(f"[ERROR] 열 누락: {miss}")

# 공백 정리본으로 검사 (원본의 앞뒤 공백도 경고)
for c in ["source_aff_seen", "source_countries", "source_ok", "note"]:
    if (X[c] != X[c].str.strip()).any(): W(f"{c}: 앞뒤 공백이 있는 셀이 있음(ingest는 source_ok의 공백을 처리하지 못함)")
S = X.copy()
for c in ["source_aff_seen", "source_countries", "source_ok", "note"]: S[c] = S[c].str.strip()

# 1. 구조
print(f"행 수 {len(S)}")
dup = S[S.duplicated(["pid", "rank"], keep=False)]
if len(dup): E(f"(pid, rank) 중복 {len(dup)}행: {dup[['pid','rank']].values.tolist()}")

# 2. 원본(manual_verify.csv)과 키·원자료 열 일치
if BASE.exists():
    B = rd(BASE)[["pid", "rank", "raw_aff", "countries"]]
    M = S.merge(B, on=["pid", "rank"], how="left", suffixes=("", "_base"), indicator=True)
    nb = M[M._merge == "left_only"]
    if len(nb): E(f"manual_verify.csv에 없는 (pid, rank): {nb[['pid','rank']].values.tolist()}")
    both = M[M._merge == "both"]
    ch = both[(both.raw_aff != both.raw_aff_base) | (both.countries != both.countries_base)]
    if len(ch): E(f"raw_aff/countries가 원본과 달라진 행(실수로 수정?): {ch.pid.tolist()}")
else:
    W("manual_verify.csv가 없어 원본 대조를 건너뜀")

# 3. 강제 포함 pid
for p in FORCE_PIDS:
    r = S[S.pid == p]
    if r.empty: E(f"강제 포함 pid {p}가 없음")
    elif r.source_ok.iloc[0] not in ("0", "1"): E(f"강제 포함 pid {p}가 아직 판정되지 않음 (source_ok 비어 있음)")

# 4. source_ok 값 형식
bad_fmt = S[~S.source_ok.isin(["", "0", "1"])]
if len(bad_fmt):
    E(f"source_ok에 0/1/빈칸이 아닌 값: {bad_fmt[['pid','source_ok']].values.tolist()} (예: '1.0', '예', 'O'는 ingest가 무시함)")
J = S[S.source_ok.isin(["0", "1"])]; U = S[S.source_ok == ""]
print(f"판정 {len(J)}건 | 미판정 {len(U)}건")

# 5. 미판정 행에는 사유(note) 필요
for r in U.itertuples():
    if not r.note: W(f"pid {r.pid} rank {r.rank}: 미판정인데 note에 사유가 없음(유료 논문 등)")
    if r.source_aff_seen or r.source_countries: W(f"pid {r.pid}: 미판정인데 source_* 칸이 채워져 있음(source_ok를 빠뜨렸나?)")

# 6. 판정 행의 필수 칸, 국가 코드 형식, 일치 여부와 판정의 정합성
cset = lambda s: {x for x in s.split("|") if x}
for r in J.itertuples():
    tag = f"pid {r.pid} rank {r.rank}"
    if not r.source_aff_seen: E(f"{tag}: source_ok는 있는데 source_aff_seen이 비어 있음(원문 소속 미기재)")
    if not r.source_countries: E(f"{tag}: source_ok는 있는데 source_countries가 비어 있음(ingest가 true_country를 빈 값으로 덮어씀)"); continue
    codes = cset(r.source_countries)
    if not re.fullmatch(r"[A-Za-z_]+(\|[A-Za-z_]+)*", r.source_countries): E(f"{tag}: source_countries 형식 오류 '{r.source_countries}' (| 로만 구분)")
    if any(c != c.upper() and c != "ZZ_former" for c in codes): E(f"{tag}: 국가 코드는 대문자 2글자여야 함 '{r.source_countries}'")
    if VALID is not None:
        inv = [c for c in codes if c not in VALID]
        if inv: E(f"{tag}: 유효하지 않은 국가 코드 {inv}")
    same = codes == cset(r.countries)
    if r.source_ok == "1" and not same: E(f"{tag}: source_ok=1인데 source_countries({r.source_countries}) != countries({r.countries})")
    if r.source_ok == "0" and same: E(f"{tag}: source_ok=0인데 source_countries가 countries와 같음")
    if r.source_ok == "0" and not r.note: W(f"{tag}: 불일치인데 note에 사유가 없음")
    # 휴리스틱: 원문 소속 문구에 국가 이름 단서가 전혀 없으면 확인용 경고
    if VALID is not None and r.source_aff_seen and len(r.source_aff_seen) < 8: W(f"{tag}: source_aff_seen이 너무 짧음 '{r.source_aff_seen}'")

# 7. 복붙 의심: source_aff_seen이 raw_aff와 글자 그대로 같은 비율
if len(J):
    same_aff = (J.source_aff_seen.str.lower() == J.raw_aff.str.lower()).mean()
    print(f"source_aff_seen이 raw_aff와 완전히 동일한 비율: {same_aff:.0%}")
    if same_aff > 0.8: W("대부분 raw_aff를 그대로 복사한 것으로 보임: 원문에서 직접 읽은 문자열인지 확인하세요")
    if (J.source_ok == "1").all() and len(J) >= 20: W("전부 일치(1)입니다. 가능하지만, 확인 안 한 칸이 섞여 있지 않은지 무작위 3건만 다시 열어 보세요")

# 8. 판정 건수와 예비 집계
n = len(J)
if n < MIN_JUDGED: W(f"판정 {n}건 < 권장 {MIN_JUDGED}건")
if n:
    k = int((J.source_ok == "1").sum()); lo, hi = wilson(k, n)
    print(f"[예비 집계] 일치 {k}/{n} = {k/n:.3f}, Wilson 95% [{lo:.3f}, {hi:.3f}] | 불일치 {n-k}건")
    if n - k: print("불일치 행:\n", J[J.source_ok == "0"][["pid", "rank", "countries", "source_countries", "note"]].to_string(index=False))
    if n < 200: print("※ 41건 표본의 구간은 넓습니다(전부 일치해도 하한이 낮음). 이 값은 200건 판정을 대체하는 통과 기준이 아니라 보강 증거입니다.")

# 9. 요약
print("\n" + "=" * 60)
for m in errs: print("[ERROR]", m)
for m in warns: print("[WARN ]", m)
print(f"ERROR {len(errs)}건, WARN {len(warns)}건")
print("→ ERROR가 0이면 source_ingest 실행 가능" if not errs else "→ ERROR를 모두 고친 뒤 source_ingest 실행")
raise SystemExit(1 if errs else 0)

행 수 41
판정 41건 | 미판정 0건
source_aff_seen이 raw_aff와 완전히 동일한 비율: 56%
[예비 집계] 일치 41/41 = 1.000, Wilson 95% [0.914, 1.000] | 불일치 0건
※ 41건 표본의 구간은 넓습니다(전부 일치해도 하한이 낮음). 이 값은 200건 판정을 대체하는 통과 기준이 아니라 보강 증거입니다.

[WARN ] 전부 일치(1)입니다. 가능하지만, 확인 안 한 칸이 섞여 있지 않은지 무작위 3건만 다시 열어 보세요
ERROR 0건, WARN 1건
→ ERROR가 0이면 source_ingest 실행 가능


SystemExit: 0

In [38]:
#

In [42]:
# -*- coding: utf-8 -*-
# =====================================================================================
# S1 마무리 단일 셀 (OpenAlex 호출 없음, 크레딧 소모 없음, API 키 불필요)
# 순서: source_ingest -> evaluate -> lens_precision -> changelog -> git_freeze
# 사용법
#   1) CHECKER, SOURCE_METHOD를 실제 값으로 채운다(비어 있으면 멈춤).
#   2) DRY_RUN=True로 한 번 실행해 전체를 점검한다(git은 계획만 출력, 커밋·태그 안 함).
#   3) 결과에 문제가 없으면 DRY_RUN=False로 바꾸고 STEPS=["git_freeze"]만 다시 실행한다.
# 어느 단계에서든 FAIL/SKIP/오류가 나면 거기서 멈추고 이유를 출력한다.
# =====================================================================================
import json, math, hashlib, shutil, subprocess, warnings
from pathlib import Path
import pandas as pd
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)

# ---------------- 설정 (여기만 수정) ----------------
CHECKER = "LEEYJ"            # 필수: 원문 대조 담당자 이름 (예: "이○○")
SOURCE_METHOD = "출판사 DOI 페이지의 저자 소속란과 대조"      # 필수: 실제로 쓴 방법 (예: "출판사 DOI 페이지의 저자 소속란, 유료 논문 n건은 미판정")
STEPS = ["source_ingest", "evaluate", "lens_precision", "changelog", "git_freeze"]
ROOT = Path.home() / "Research/Minnesota_Transport"
FORCE_PIDS = ["25038"]
APPLY_CORRECTIONS = True          # 원문 대조 불일치를 manual_verify.csv에 반영
WINDOW = 2007
ACC_MIN, WILSON_LOW = 0.95, 0.90
EXPECTED_PREREG_PREFIX = "f16e76b4"
IGNORE_PREREG_HASH_MISMATCH = False
DRY_RUN = True                    # False여야 실제 git commit + tag
TAG_NAME = None                   # None이면 S0S1-freeze-YYYY-MM-DD
# ----------------------------------------------------

S0, S1 = ROOT / "paper1/00_S0", ROOT / "paper1/01_S1"
DOC = "S1_검수판정_기준_및_확인기록.md"
SRC_FILE = "manual_source_check.csv"
hdr = lambda t: print("\n" + "=" * 90 + f"\n{t}\n" + "=" * 90)
now = lambda: pd.Timestamp.now().isoformat(timespec="seconds")
sha_b = lambda b: hashlib.sha256(b).hexdigest()
sha_f = lambda p: sha_b(Path(p).read_bytes())
rd = lambda p: pd.read_csv(p, dtype=str, encoding="utf-8-sig").fillna("")

class Halt(Exception): pass

def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n)
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)); return (c - h) / d, (c + h) / d

def changelog(line):
    cl = S1 / "CHANGELOG_S1.md"
    if not cl.exists(): cl.write_text("# CHANGELOG S1 (날짜 | 변경 | 사유 | 영향)\n\n", encoding="utf-8")
    with cl.open("a", encoding="utf-8") as f: f.write(line + "\n")

def changelog_once(key, line):
    cl = S1 / "CHANGELOG_S1.md"
    if cl.exists() and key in cl.read_text(encoding="utf-8"): return False
    changelog(line); return True

# ============================================================
# 1) source_ingest: 원문 대조 집계, 불일치 반영, 기록 문서 6절 기입
# ============================================================
def st_source_ingest():
    hdr("[source_ingest] 원문 대조 집계")
    if not CHECKER.strip(): raise Halt("CHECKER(대조 담당자)를 설정에 기입하세요")
    if not SOURCE_METHOD.strip(): raise Halt("SOURCE_METHOD(실제 대조 방법)를 설정에 기입하세요")
    X = rd(S1 / SRC_FILE); done = X[X.source_ok.isin(["0", "1"])]
    if len(done) == 0: raise Halt("판정된 행이 없습니다")
    if len(done) < len(X): print(f"⚠ 미판정 {len(X) - len(done)}건: 판정된 {len(done)}건만 집계합니다")
    k, n = int((done.source_ok == "1").sum()), len(done); lo, hi = wilson(k, n)
    bad = done[done.source_ok == "0"]
    print(f"원문 대조 {n}건 | 일치 {k} ({k / n:.3f}, Wilson [{lo:.3f}, {hi:.3f}]) | 불일치 {len(bad)}")
    if len(bad): print(bad[["pid", "rank", "countries", "source_countries", "note"]].to_string(index=False))
    inc = bool(done.pid.isin(FORCE_PIDS).any())
    summ = dict(n_checked=n, n_agree=k, n_disagree=len(bad), n_unjudged=len(X) - n,
                disagree_pids=[f"{r.pid}:{r.rank}" for r in bad.itertuples()], wilson=[round(lo, 3), round(hi, 3)],
                includes_forced=inc, checker=CHECKER, method=SOURCE_METHOD, at=now())
    (S1 / "SOURCE_CHECK_SUMMARY.json").write_text(json.dumps(summ, ensure_ascii=False, indent=2), encoding="utf-8")
    if len(bad) and APPLY_CORRECTIONS:
        M = rd(S1 / "manual_verify.csv")
        shutil.copy2(S1 / "manual_verify.csv", S1 / f"manual_verify.before_source_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}.csv")
        for r in bad.itertuples():
            if not r.source_countries: raise Halt(f"{r.pid}: 불일치인데 source_countries가 비어 있습니다")
            m = (M.pid == r.pid) & (M["rank"] == r.rank); M.loc[m, "ok"] = "0"; M.loc[m, "true_country"] = r.source_countries
        M.to_csv(S1 / "manual_verify.csv", index=False, encoding="utf-8-sig"); print("manual_verify.csv에 불일치 반영(원본은 before_source_* 로 백업)")
    dp = S1 / DOC
    if dp.exists():
        rep = {"- 원문 대조 건수:": f"- 원문 대조 건수: n = {n} (25038 포함 여부: {'예' if inc else '아니오'})",
               "- 대조 방법": f"- 대조 방법(논문 페이지, PDF 등): {SOURCE_METHOD}",
               "- 판정이 바뀐 건수와 pid:": f"- 판정이 바뀐 건수와 pid: {len(bad)}건 {summ['disagree_pids'] or ''}",
               "- 확인자 / 확인일:": f"- 확인자 / 확인일: {CHECKER} / {pd.Timestamp.now().strftime('%Y-%m-%d')}"}
        L = dp.read_text(encoding="utf-8").splitlines()
        L = [next((v for p, v in rep.items() if l.startswith(p)), l) for l in L]
        dp.write_text("\n".join(L) + "\n", encoding="utf-8"); print(f"{DOC} 6절 갱신")
    else: print(f"※ {DOC}가 01_S1에 없어 6절 자동 기입을 건너뜀(복사 후 이 단계만 다시 실행)")
    return True

# ============================================================
# 2) evaluate: 통과 판정 (원 스크립트의 st_evaluate와 같은 규칙)
# ============================================================
def st_evaluate():
    hdr("[evaluate] 통과 판정")
    R = []; ws = json.loads((S1 / "WINDOW_DRAFT.json").read_text()); used = WINDOW
    R.append((f"1.5 분석 창 시작 연도(확정 {used}, 재계산 {ws['window_start']}; 2010 이하면 통과)", used, used <= 2010))
    mp = S1 / "manual_verify.csv"
    Mv = rd(mp); done = Mv[Mv.ok.isin(["0", "1"])]
    if len(done) >= 150:
        k, n = int((done.ok == "1").sum()), len(done); lo, hi = wilson(k, n)
        R.append(("1.7 수작업 정확도(점추정≥0.95, Wilson 하한≥0.90)", f"{k / n:.3f} [{lo:.3f}, {hi:.3f}] n={n}", k / n >= ACC_MIN and lo >= WILSON_LOW))
    else: R.append(("1.7 수작업 정확도", f"판정 {len(done)}건(<150): 더 채우세요", None))
    U = rd(S1 / "unknown_check.csv"); U = U[U.missed.isin(["0", "1"])]
    if len(U) >= 80:
        k, n = int((U.missed == "1").sum()), len(U); lo, hi = wilson(k, n)
        R.append(("1.6 대체: 소속 문자열 있는데 국가 누락된 비율(참고)", f"{k / n:.3f} [{lo:.3f}, {hi:.3f}] n={n}", "참고"))
    else: R.append(("1.6 대체: 국가 누락 점검", f"판정 {len(U)}건(<80)", None))
    T = pd.DataFrame(R, columns=["검사", "값", "통과"]); print(T.to_string(index=False))
    fail = any(x is False for x in T.통과); skip = any(x is None for x in T.통과)
    msg = ("# S1 검증 보고서\n\n" + T.to_string(index=False) + f"\n\n작성 {now()}\n\n"
           + ("**일부 검사 미실시(SKIP)**: 완료로 간주하지 말 것.\n" if skip else "")
           + ("**FAIL 존재**: FALLBACK 적용(창을 2010년 이후로 축소 / 고정 저널만 / 비DOI는 부록). 창이 2010년 이후면 RQ4·RQ5를 부록으로, RQ1~RQ3 중심 논문으로 전환.\n" if fail else "")
           + "\n한계: 소속 국가는 국적이 아님. 구 자료는 소속 기재가 성김. 국가는 단일 출처(OpenAlex)에 의존(Lens 교차 확인 안 함). 창 안 DOI 보유 논문은 전수 수집, DOI 없는 논문은 국가 미상으로 계산.\n")
    (S1 / "validation_report.md").write_text(msg, encoding="utf-8"); print("저장: validation_report.md")
    if fail: raise Halt("FAIL 존재: 계획서 FALLBACK 규칙을 따르세요. 이후 단계(changelog, git_freeze)를 실행하지 않습니다")
    if skip: raise Halt("SKIP 존재: 미실시 검사를 채운 뒤 다시 실행하세요")
    print("→ 전부 PASS(참고 항목 제외)")
    return True

# ============================================================
# 3) lens_precision
# ============================================================
def st_lens_precision():
    hdr("[lens_precision] 대상 렌즈 정밀도")
    T = rd(S1 / "target_lens_review.csv"); T = T[T.precision_ok.isin(["0", "1"])].copy()
    if len(T) == 0: raise Halt("target_lens_review.csv에 판정이 없습니다")
    T["year"] = T.year.astype(int); k, n = int((T.precision_ok == "1").sum()), len(T); lo, hi = wilson(k, n)
    out = dict(n=n, correct=k, precision=round(k / n, 3), wilson=[round(lo, 3), round(hi, 3)], wrong_pids=T[T.precision_ok == "0"].pid.tolist(), by_period={})
    T["per"] = pd.cut(T.year, [0, 2006, 2013, 2019, 2100], labels=["~2006", "2007-13", "2014-19", "2020-"])
    for p, g in T.groupby("per", observed=True): out["by_period"][str(p)] = f"{int((g.precision_ok == '1').sum())}/{len(g)}"
    print(json.dumps(out, ensure_ascii=False, indent=2))
    (S1 / "target_lens_precision.json").write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding="utf-8")
    print("※ 재현율(빠진 지역)은 이 값에 포함되지 않습니다")
    return True

# ============================================================
# 4) changelog
# ============================================================
def st_changelog():
    hdr("[changelog] CHANGELOG_S1 기록")
    sp, lp = S1 / "SOURCE_CHECK_SUMMARY.json", S1 / "target_lens_precision.json"
    if not sp.exists() or not lp.exists(): raise Halt("source_ingest와 lens_precision을 먼저 실행하세요")
    s = json.loads(sp.read_text(encoding="utf-8")); lens = json.loads(lp.read_text(encoding="utf-8")); added = []
    items = [
     ("검수 3종 사전 판정", f"- {now()} | 검수 3종 사전 판정(manual_verify 200, unknown_check 100, target_lens_review 200): Claude가 raw_aff·제목을 읽고 판정, 연구자가 전체 검토 후 확인. 원문 대조 n={s['n_checked']}건(일치 {s['n_agree']}, 불일치 {s['n_disagree']}, 미판정 {s['n_unjudged']}, 담당 {s['checker']}, 방법: {s['method']}). 판정 기준 문서({DOC})는 사전 판정 후 evaluate 실행 전에 작성(판정 전 기록 아님) | 판정 방식 | S1 통과 판정, S10.4 생성형 AI 사용 선언"),
     ("본 수집 후 audit 재계산", f"- {now()} | 본 수집 후 audit 재계산: 합산 규칙 {WINDOW} 유지, 고정 저널 266칸 중 0.80 미만 1칸(TranspPol 2008, 0.800), 엄격 규칙 2009. 확정 창은 변경하지 않고 엄격 규칙은 민감도로 보고 | 분석 창 | S5.1(c), S7, S8"),
     ("fuzzy_probe 생략", f"- {now()} | fuzzy_probe 생략: 창 안 DOI 없는 논문 160편은 국가 미상으로 계산(보수적) | 요청·크레딧 절감 | 국가 몫의 미상 비중에 포함"),
     ("대상 렌즈(제목 지명) 보조", f"- {now()} | 대상 렌즈(제목 지명) 보조 변수로 사용: 정밀도 {lens['correct']}/{lens['n']} (Wilson {lens['wilson']}), 재현율은 미측정(예: Britain, Europe 미포착) | 보조 변수 | RQ 대상 지역 분석은 정밀도 한계 명시"),
     ("소득군 연도 기준", f"- {now()} | 소득군 연도 기준: income_tv는 OGHIST 산정 달력연도 기준으로 부착(회계연도 기준은 wb_income_history_long.csv로 민감도). 결측 규칙: ET 2024는 원본 빈칸이라 채우지 않음, RE는 OGHIST에 없음, UNK는 소득군도 UNK | 시변 소득군 | S7 소득군 분석. S7 결과 열람 전에 기록"),
     ("정렬 일치율 해석", f"- {now()} | 저자 수 정렬 일치율 100%는 같은 출처(OpenAlex) 간 비교일 가능성이 있어 독립 검증으로 해석하지 않음(F1 n_authors 출처는 nauthors_check로 확인 예정) | 해석 제한 | 논문 한계 절"),
    ]
    for key, line in items:
        if changelog_once(key, line): added.append(key)
    print("추가된 항목:", added or "없음(이미 기록됨)")
    return True

# ============================================================
# 5) git_freeze: 해시 기록 + 커밋 + 태그 (DRY_RUN=True면 계획만)
# ============================================================
def git(*a, check=False):
    r = subprocess.run(["git", "-C", str(ROOT), *a], capture_output=True, text=True)
    if check and r.returncode: raise Halt(f"git {' '.join(a)} 실패: {r.stderr.strip()}")
    return r

def st_git_freeze():
    hdr("[git_freeze] 산출물 해시 기록 + 커밋 + 태그")
    pj = S0 / "PREREG.json"; h = sha_f(pj)
    print(f"PREREG.json sha256 {h[:8]} (기대 앞부분 {EXPECTED_PREREG_PREFIX})")
    if not h.startswith(EXPECTED_PREREG_PREFIX) and not IGNORE_PREREG_HASH_MISMATCH:
        raise Halt("PREREG 해시가 기대값과 다릅니다. 보정 이후 또 수정됐는지 확인하세요(해시 산출 방식이 다르면 IGNORE_PREREG_HASH_MISMATCH=True)")
    keys = ["manual_verify.csv", "unknown_check.csv", "target_lens_review.csv", SRC_FILE, "SOURCE_CHECK_SUMMARY.json", "target_lens_precision.json",
            "validation_report.md", "coverage_by_year.csv", "coverage_low_cells.csv", "WINDOW_DRAFT.json", "paper_status.csv", "wb_income_history.csv",
            "wb_income_history_long.csv", "S1_SHEETS_LOG.csv", DOC, "TERRITORY_RULES.md", "RECOVERY_RULES_PREREG.json", "CHANGELOG_S1.md"]
    M = [dict(file=f"01_S1/{f}", sha256=sha_f(S1 / f)) for f in keys if (S1 / f).exists()] + [dict(file="00_S0/PREREG.json", sha256=h)]
    missing = [f for f in ["validation_report.md", "SOURCE_CHECK_SUMMARY.json", "target_lens_precision.json"] if not (S1 / f).exists()]
    if missing: raise Halt(f"핵심 산출물이 없습니다: {missing}")
    print("※ 아직 없는 선택 산출물(나중에 만들면 다음 freeze에 포함):", [f for f in keys if not (S1 / f).exists()])
    pd.DataFrame(M).assign(at=now()).to_csv(S1 / "S1_FREEZE_MANIFEST.csv", index=False); print(f"S1_FREEZE_MANIFEST.csv {len(M)}개 해시 기록")
    big = [(p.name, p.stat().st_size / 1e6) for p in S1.glob("*") if p.is_file() and p.stat().st_size > 50e6]
    if big: print("⚠ 50MB 초과 파일(커밋 용량 주의):", [(n, f"{m:.0f}MB") for n, m in big])
    top = git("rev-parse", "--show-toplevel")
    if top.returncode: raise Halt("git 저장소가 아닙니다: " + top.stderr.strip())
    spec = ["--", "paper1/00_S0", "paper1/01_S1", ":(exclude)paper1/01_S1/oa_raw", ":(exclude)paper1/01_S1/_backup_*"]
    print(git("status", "--short", *spec).stdout or "(변경 없음)")
    tag = TAG_NAME or f"S0S1-freeze-{pd.Timestamp.now().strftime('%Y-%m-%d')}"
    if DRY_RUN:
        print(f"[DRY_RUN] 실제 실행 시: git add {' '.join(spec)} / git commit / git tag -a {tag}")
        print("※ oa_raw(원자료)는 용량 때문에 제외. 재현에 필요하면 별도 보관(해시는 manifest 참고)")
        print("→ 문제가 없으면 DRY_RUN=False, STEPS=['git_freeze']로 다시 실행")
        return True
    if git("tag", "-l", tag).stdout.strip(): raise Halt(f"태그 {tag}가 이미 있습니다(TAG_NAME을 바꾸세요)")
    git("add", *spec, check=True)
    c = git("commit", "-m", "S0/S1 freeze: 검수 3종(AI 사전 판정+연구자 확인+원문 대조), 규칙 동결, 보정 PREREG 포함")
    print(c.stdout.strip() or c.stderr.strip())
    if c.returncode and "nothing to commit" not in (c.stdout + c.stderr): raise Halt("커밋 실패")
    git("tag", "-a", tag, "-m", "S0/S1 freeze", check=True); print(f"태그 생성: {tag} (원격 반영: git push --follow-tags)")
    return True

FUNCS = {"source_ingest": st_source_ingest, "evaluate": st_evaluate, "lens_precision": st_lens_precision,
         "changelog": st_changelog, "git_freeze": st_git_freeze}
assert S1.exists() and S0.exists(), f"{S0} 또는 {S1}이 없습니다. ROOT를 확인하세요"
print(f"ROOT={ROOT} | STEPS={STEPS} | DRY_RUN={DRY_RUN}")
try:
    for s in STEPS: FUNCS[s]()
    print("\n✔ 요청한 단계 완료")
except Halt as e:
    print(f"\n⛔ 중단: {e}")

ROOT=/home/yjlee/Research/Minnesota_Transport | STEPS=['source_ingest', 'evaluate', 'lens_precision', 'changelog', 'git_freeze'] | DRY_RUN=True

[source_ingest] 원문 대조 집계
원문 대조 41건 | 일치 41 (1.000, Wilson [0.914, 1.000]) | 불일치 0
※ S1_검수판정_기준_및_확인기록.md가 01_S1에 없어 6절 자동 기입을 건너뜀(복사 후 이 단계만 다시 실행)

[evaluate] 통과 판정
                                            검사                          값   통과
1.5 분석 창 시작 연도(확정 2007, 재계산 2007; 2010 이하면 통과)                       2007 True
         1.7 수작업 정확도(점추정≥0.95, Wilson 하한≥0.90) 1.000 [0.981, 1.000] n=200 True
              1.6 대체: 소속 문자열 있는데 국가 누락된 비율(참고) 0.970 [0.915, 0.990] n=100   참고
저장: validation_report.md
→ 전부 PASS(참고 항목 제외)

[lens_precision] 대상 렌즈 정밀도
{
  "n": 200,
  "correct": 195,
  "precision": 0.975,
  "wilson": [
    0.943,
    0.989
  ],
  "wrong_pids": [
    "90265",
    "11912",
    "101469",
    "100997",
    "74350"
  ],
  "by_period": {
    "~2006": "34/36",
    "2007-13": "36/37",
    "2014-19": "53/54",
    "2020-": "72/73"
  }
}
※ 재

In [40]:
#

In [3]:
# -*- coding: utf-8 -*-
# =====================================================================================
# S1 freeze 최종 단일 셀 (OpenAlex 호출 없음, 크레딧 소모 없음)
# 순서: A 기록 문서 설치·6절 기입 -> B (선택) territory_doc·recovery -> C CHANGELOG
#       -> D git 사전 점검 -> E manifest 해시 기록 + 실제 commit + tag
# 이 셀은 이전 셀과 독립적으로 돈다(이전 셀을 다시 실행할 필요 없음).
# 중요: DRY_RUN=False이므로 마지막에 실제로 커밋·태그가 만들어진다. 아래 설정을 먼저 확인할 것.
# =====================================================================================
import json, hashlib, subprocess, shutil, importlib.util, warnings
from pathlib import Path
import pandas as pd
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)

# ---------------- 설정 (여기만 수정) ----------------
ROOT = Path.home() / "Research/Minnesota_Transport"
DOC = "S1_검수판정_기준_및_확인기록.md"
DOC_SRC = Path.home() / "Downloads" / DOC          # 기록 문서가 있는 위치(이미 01_S1에 있으면 무시)
REASON_41 = "단계적 확대 방식으로 진행했다. 먼저 41건(무작위 40건과 25038)을 원문과 대조하고, 불일치가 나오면 200건으로 확대할 계획이었다. 41건에서 불일치가 0건(Wilson 95% 하한 0.914)이어서 확대하지 않고 중단했다. 확대 기준(불일치 허용 건수)은 수치로 사전 기록하지 않았고, 중단은 41건 결과를 본 뒤의 판단이다."
RUN_STEP4 = False     # True면 territory_doc + recovery 실행(recovery 규칙은 실행 즉시 동결됨: 먼저 규칙을 검토할 것)
NEXT_STEPS_PATH = None  # s1_next_steps.py 경로(RUN_STEP4=True일 때만 사용). None이면 흔한 위치에서 탐색
EXPECTED_PREREG_PREFIX = "f16e76b4"
IGNORE_PREREG_HASH_MISMATCH = False
DRY_RUN = False       # False여야 실제 git commit + tag
TAG_NAME = None       # None이면 S0S1-freeze-YYYY-MM-DD
# ----------------------------------------------------

S0, S1 = ROOT / "paper1/00_S0", ROOT / "paper1/01_S1"
hdr = lambda t: print("\n" + "=" * 90 + f"\n{t}\n" + "=" * 90)
now = lambda: pd.Timestamp.now().isoformat(timespec="seconds")
sha_f = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()

class Halt(Exception): pass

def changelog_once(key, line):
    cl = S1 / "CHANGELOG_S1.md"
    if not cl.exists(): cl.write_text("# CHANGELOG S1 (날짜 | 변경 | 사유 | 영향)\n\n", encoding="utf-8")
    if key in cl.read_text(encoding="utf-8"): return False
    with cl.open("a", encoding="utf-8") as f: f.write(line + "\n")
    return True

def git(*a, check=False):
    r = subprocess.run(["git", "-C", str(ROOT), *a], capture_output=True, text=True)
    if check and r.returncode: raise Halt(f"git {' '.join(a)} 실패: {r.stderr.strip()}")
    return r

# ---------------------------------------------------------------- A
def step_doc():
    hdr("[A] 기록 문서 설치 + 6절 기입")
    dp = S1 / DOC
    if not dp.exists():
        if not DOC_SRC.exists(): raise Halt(f"{DOC}가 01_S1에도 {DOC_SRC}에도 없습니다. DOC_SRC를 확인하세요")
        shutil.copy2(DOC_SRC, dp); print(f"복사: {DOC_SRC} -> {dp}")
    sp = S1 / "SOURCE_CHECK_SUMMARY.json"
    if not sp.exists(): raise Halt("SOURCE_CHECK_SUMMARY.json이 없습니다(source_ingest를 먼저 실행)")
    s = json.loads(sp.read_text(encoding="utf-8"))
    inc = "예" if s.get("includes_forced") else "아니오"
    rep = {"- 원문 대조 건수:": f"- 원문 대조 건수: n = {s['n_checked']} (25038 포함 여부: {inc})",
           "- 대조 방법": f"- 대조 방법(논문 페이지, PDF 등): {s['method']}",
           "- 판정이 바뀐 건수와 pid:": f"- 판정이 바뀐 건수와 pid: {s['n_disagree']}건 {s['disagree_pids'] or ''}".rstrip(),
           "- 확인자 / 확인일:": f"- 확인자 / 확인일: {s['checker']} / {s['at'][:10]}"}
    L = dp.read_text(encoding="utf-8").splitlines()
    L = [next((v for p, v in rep.items() if l.startswith(p)), l) for l in L]
    dp.write_text("\n".join(L) + "\n", encoding="utf-8")
    sec6 = [l for l in L if l.startswith("- 원문 대조 건수") or l.startswith("- 대조 방법") or l.startswith("- 판정이 바뀐") or l.startswith("- 확인자")]
    print("\n".join(sec6))
    if any("____" in l for l in sec6): raise Halt("6절에 아직 빈칸(____)이 남아 있습니다. 문서의 줄 형식을 확인하세요")
    if "미완료" in dp.read_text(encoding="utf-8").split("## 6.")[-1].split("## 7.")[0]: print("⚠ 6절에 '미완료' 문구가 남아 있습니다. 문서에서 직접 지우세요")

# ---------------------------------------------------------------- B
def step_territory_recovery():
    hdr("[B] territory_doc + recovery (선택)")
    cands = [NEXT_STEPS_PATH] if NEXT_STEPS_PATH else [ROOT / "s1_next_steps.py", ROOT / "paper1/s1_next_steps.py", S1 / "s1_next_steps.py",
                                                      Path.cwd() / "s1_next_steps.py", Path.home() / "Downloads/s1_next_steps.py"]
    p = next((Path(c) for c in cands if c and Path(c).exists()), None)
    if p is None: raise Halt("s1_next_steps.py를 찾지 못했습니다. NEXT_STEPS_PATH를 지정하세요")
    spec = importlib.util.spec_from_file_location("s1_next_steps", p); m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m)
    m.DATA, m.ROOT, m.S0, m.S1 = m.paths()
    m.st_territory_doc(); m.st_recovery()

# ---------------------------------------------------------------- C
def step_changelog():
    hdr("[C] CHANGELOG 기록")
    if not REASON_41.strip(): raise Halt("REASON_41(원문 대조를 41건으로 줄인 사유)을 설정에 직접 기입하세요")
    added = []
    if changelog_once("계획서 1.7(200 authorship 원문 대조) 대비 변경",
        f"- {now()} | 계획서 1.7(200 authorship 원문 대조) 대비 변경: 원문 대조는 200건 중 41건(무작위 40 + 25038)으로 축소, 나머지는 raw_aff 텍스트 기준 판정을 연구자가 검토. 사유: {REASON_41.strip()} | 계획 대비 축소 | 정확도 근거 강도: 원문 41/41, Wilson [0.914, 1.000]. 논문 한계 절에 명시"): added.append("1.7 축소")
    if RUN_STEP4:
        if changelog_once("territory_doc·recovery 동결", f"- {now()} | territory_doc·recovery 동결: 영토·구 국가 표기 규칙(TERRITORY_RULES.md)과 국가 미상 규칙 복구 규칙(RECOVERY_RULES_PREREG.json)을 S7 결과 열람 전에 기록. 복구는 민감도 분석용이며 주 분석 테이블은 변경하지 않음 | 규칙 사전 기록 | S2, S7 민감도"): added.append("step4")
    print("추가된 항목:", added or "없음(이미 기록됨)")

# ---------------------------------------------------------------- D
def step_git_check():
    hdr("[D] git 사전 점검")
    if git("rev-parse", "--show-toplevel").returncode: raise Halt("git 저장소가 아닙니다")
    if not git("config", "user.name").stdout.strip() or not git("config", "user.email").stdout.strip():
        raise Halt("git user.name/user.email이 설정되지 않았습니다. 터미널에서 git config --global user.name/email을 먼저 설정하세요")
    pj = "paper1/00_S0/PREREG.json"
    if git("check-ignore", "-q", pj).returncode == 0: raise Halt(f"{pj}가 .gitignore에 걸려 있습니다. 이 상태로는 보정 PREREG가 태그에 포함되지 않습니다")
    h = sha_f(ROOT / pj); print(f"PREREG.json sha256 {h[:8]} (기대 {EXPECTED_PREREG_PREFIX})")
    if not h.startswith(EXPECTED_PREREG_PREFIX) and not IGNORE_PREREG_HASH_MISMATCH: raise Halt("PREREG 해시가 기대값과 다릅니다")
    last = git("log", "-1", "--format=%h %ad %s", "--date=short", "--", pj).stdout.strip()
    committed = git("rev-parse", f"HEAD:{pj}")
    if committed.returncode or not last:
        print("PREREG.json: 아직 커밋된 적 없음 -> 이번 커밋에 포함됩니다")
    else:
        same = committed.stdout.strip() == git("hash-object", str(ROOT / pj)).stdout.strip()
        print(f"PREREG.json 마지막 커밋: {last} | 현재 파일과 {'동일(보정본이 이미 커밋됨)' if same else '다름(보정본이 이번 커밋에 포함됨)'}")
    st = git("status", "--short", "--", "paper1/00_S0", "paper1/01_S1", ":(exclude)paper1/01_S1/oa_raw", ":(exclude)paper1/01_S1/_backup_*").stdout
    print("커밋 대상 변경 목록(상위 25줄):\n" + "\n".join(st.splitlines()[:25]) if st else "(변경 없음)")

# ---------------------------------------------------------------- E
def step_freeze():
    hdr("[E] manifest 해시 기록 + commit + tag")
    keys = ["manual_verify.csv", "unknown_check.csv", "target_lens_review.csv", "manual_source_check.csv", "SOURCE_CHECK_SUMMARY.json", "target_lens_precision.json",
            "validation_report.md", "coverage_by_year.csv", "coverage_low_cells.csv", "WINDOW_DRAFT.json", "paper_status.csv", "wb_income_history.csv",
            "wb_income_history_long.csv", DOC, "TERRITORY_RULES.md", "RECOVERY_RULES_PREREG.json", "authorship_level_recovered.csv.gz", "CHANGELOG_S1.md"]
    need = ["validation_report.md", "SOURCE_CHECK_SUMMARY.json", "target_lens_precision.json", DOC, "CHANGELOG_S1.md"]
    miss = [f for f in need if not (S1 / f).exists()]
    if miss: raise Halt(f"핵심 산출물이 없습니다: {miss}")
    M = [dict(file=f"01_S1/{f}", sha256=sha_f(S1 / f)) for f in keys if (S1 / f).exists()] + [dict(file="00_S0/PREREG.json", sha256=sha_f(S0 / "PREREG.json"))]
    pd.DataFrame(M).assign(at=now()).to_csv(S1 / "S1_FREEZE_MANIFEST.csv", index=False); print(f"S1_FREEZE_MANIFEST.csv {len(M)}개 해시 기록")
    print("※ manifest에 없는 선택 산출물:", [f for f in keys if not (S1 / f).exists()])
    big = [(p.name, p.stat().st_size / 1e6) for p in S1.glob("*") if p.is_file() and p.stat().st_size > 50e6]
    if big: print("⚠ 50MB 초과 파일:", [(n, f"{x:.0f}MB") for n, x in big])
    spec = ["--", "paper1/00_S0", "paper1/01_S1", ":(exclude)paper1/01_S1/oa_raw", ":(exclude)paper1/01_S1/_backup_*"]
    tag = TAG_NAME or f"S0S1-freeze-{pd.Timestamp.now().strftime('%Y-%m-%d')}"
    if DRY_RUN: print(f"[DRY_RUN] git add / commit / tag -a {tag} 를 실행하지 않았습니다"); return
    if git("tag", "-l", tag).stdout.strip(): raise Halt(f"태그 {tag}가 이미 있습니다(TAG_NAME을 바꾸세요)")
    git("add", *spec, check=True)
    c = git("commit", "-m", "S0/S1 freeze: 검수 3종(AI 사전 판정+연구자 확인+원문 대조 41건), 규칙 동결, 보정 PREREG 포함")
    print((c.stdout.strip() or c.stderr.strip())[:600])
    if c.returncode and "nothing to commit" not in (c.stdout + c.stderr): raise Halt("커밋 실패")
    git("tag", "-a", tag, "-m", "S0/S1 freeze", check=True)
    print(f"\n태그 생성: {tag} | commit {git('rev-parse', '--short', 'HEAD').stdout.strip()}")
    print("검증:", git("tag", "-n", "-l", tag).stdout.strip())
    print("원격 반영(원하면 터미널에서): git push --follow-tags")
    print("※ oa_raw(원자료)는 커밋에서 제외됨: 별도 보관본을 만들어 두세요(해시는 필요 시 따로 기록)")

assert S0.exists() and S1.exists(), "ROOT를 확인하세요"
print(f"ROOT={ROOT} | RUN_STEP4={RUN_STEP4} | DRY_RUN={DRY_RUN}")
try:
    step_doc()
    if RUN_STEP4: step_territory_recovery()
    step_changelog(); step_git_check(); step_freeze()
    print("\n✔ 완료")
except Halt as e:
    print(f"\n⛔ 중단: {e}\n(이 지점까지의 파일 변경은 유지됩니다. 문제를 고친 뒤 같은 셀을 다시 실행하세요. 모든 단계가 여러 번 실행해도 안전합니다)")

ROOT=/home/yjlee/Research/Minnesota_Transport | RUN_STEP4=False | DRY_RUN=False

[A] 기록 문서 설치 + 6절 기입
- 원문 대조 건수: n = 41 (25038 포함 여부: 예)
- 대조 방법(논문 페이지, PDF 등): 출판사 DOI 페이지의 저자 소속란과 대조
- 판정이 바뀐 건수와 pid: 0건
- 확인자 / 확인일: LEEYJ / 2026-10-13

[C] CHANGELOG 기록
추가된 항목: 없음(이미 기록됨)

[D] git 사전 점검
PREREG.json sha256 f16e76b4 (기대 f16e76b4)
PREREG.json 마지막 커밋: 90416c0 2026-10-06 S0 PREREG fix: variant_names (7 named, defined), gate hardened | 현재 파일과 동일(보정본이 이미 커밋됨)
커밋 대상 변경 목록(상위 25줄):
?? paper1/01_S1/

[E] manifest 해시 기록 + commit + tag
S1_FREEZE_MANIFEST.csv 16개 해시 기록
※ manifest에 없는 선택 산출물: ['TERRITORY_RULES.md', 'RECOVERY_RULES_PREREG.json', 'authorship_level_recovered.csv.gz']
[main 0823d46] S0/S1 freeze: 검수 3종(AI 사전 판정+연구자 확인+원문 대조 41건), 규칙 동결, 보정 PREREG 포함
 30 files changed, 228261 insertions(+)
 create mode 100644 paper1/01_S1/CHANGELOG_S1.md
 create mode 100644 paper1/01_S1/OGHIST_2026_07_15.xlsx
 create mode 100644 paper1/01_S1/S1_FREEZE_MANIFEST.csv
 create mode 100644 "paper1/01_S1/S1_

In [2]:
#

In [ ]:
# # 논문 1 진행 현황 (2026-10-13 기준)

# ## 한눈에 보기
# - **S0 사전등록·동결:** 완료. 보정된 PREREG는 커밋됨(`90416c0`). `FREEZE_LOG`에 새 해시가 남았는지는 확인되지 않음.
# - **S1 국가 층·커버리지:** **완료 및 동결.** 커밋 `0823d46`, 태그 `S0S1-freeze-2026-10-13`. 창 2007, 수작업 정확도 PASS. RQ4·RQ5를 본 논문에 유지.
# - **다음:** S2 측정 교정. 그 전에 S1의 소규모 잔여 항목(아래 B절)을 정리.

# ---

# ## A. 지금까지 한 일

# ### S0 사전등록·동결
# - 오염 레코드 269편 제외(F1 111,685편 → F1X 111,416편), 창간 전 논문 검토, E1 충돌 판정(RailEng 2020, TRR 1967).
# - 고정 저널 14개 확정. 사건 목록·우연 적중률 진단. 주 시나리오는 NONE이므로 **연속 노출 시계열이 주 분석**, 사건 대조는 민감도.
# - 동결 검사 32건 전부 PASS, 해시 9개 기록.
# - 동결 후 위험군 변형 7종 이름을 보정(PREREG 해시 `eb2f284f` → `f16e76b4`), CHANGELOG에 사유 기록. **보정본은 `90416c0`(2026-10-06)으로 커밋됨**을 이번 freeze에서 확인(현재 파일과 동일).
# - pending으로 남은 사건: AMAR(창간 2014 / 코퍼스 2013), TR-E(창간 1972 / 코퍼스 1997).

# ### S1 국가 층 구축
# - **분석 창 2007 확정.** probe 표본 audit 기준이며 사후 판단임을 CHANGELOG에 기록. 본 수집 후 재계산도 2007(엄격 규칙은 2009, 민감도로만 보고).
# - **전수 수집:** 창 안 DOI 보유 논문 81,485편, 931회 요청, 약 $0.093.
# - **국가 층 구축:** 포함 논문 81,645편(DOI 없는 160편은 국가 미상으로 포함). 가중 평균 국가 보유율 94.7%, 미상 5.3%. 상위 몫 US 25.5%, CN 21.7%, GB 4.3%.
# - **커버리지:** 고정 저널 266칸 중 0.80 미만은 1칸(TranspPol 2008, 정확히 0.800).
# - **시변 소득군 부착:** OGHIST_2026_07_15, 산정 달력연도 기준(8,056행, 222개국). FY2027 분포와 10개국 검산 일치. `income_tv` 결측은 UNK와 같고, 실제 국가 중 결측은 ET 2024(원본 빈칸)와 RE뿐(몫 0.0000).
# - **검수 3종:** Claude가 파일 텍스트로 사전 판정, 연구자가 전체 검토·확인. 판정 기준 문서(`S1_검수판정_기준_및_확인기록.md`)는 사전 판정 후 `evaluate` 전에 작성(판정 전 기록 아님).
# - **원문 대조 41건:** 무작위 40건 + 25038, 전부 일치. 200건 대비 축소했고, 사유(단계적 확대, 불일치 0건이라 중단, 확대 기준은 사전 기록 없음)를 CHANGELOG와 기록 문서 6절에 남김.
# - **`evaluate` 결과:** 전 항목 PASS.
# - **S1 동결:** manifest 16개 해시 기록, 30개 파일 커밋, 태그 생성.

# ### S1 검증 결과 요약
# | 항목 | 결과 | Wilson 95% | 용도 |
# |---|---|---|---|
# | 분석 창 | 2007(재계산 2007) | - | 기준 2010 이하: PASS |
# | `manual_verify` 200건(파일 텍스트 기준) | 200/200 | [0.981, 1.000] | S1 통과 판정: **PASS** |
# | 원문 대조 41건 | 41/41 | [0.914, 1.000] | 보강 증거(단독 통과 기준 아님) |
# | `unknown_check` 100건 | 복구 가능 97 | [0.915, 0.990] | 참고 지표 |
# | `target_lens_review` 200편 | 195/200 | [0.943, 0.989] | 대상 렌즈 정밀도(재현율 미측정) |

# ### 기록으로 남긴 해석 제한
# - 정렬 일치율 100%는 같은 출처(OpenAlex) 간 비교일 가능성이 있어 독립 검증으로 해석하지 않음.
# - 소속 국가는 국적이 아님. Lens 교차 확인은 생략하고 단일 출처에 의존.
# - 원문 대조는 41건뿐이며 나머지 159건은 텍스트 기준 판정을 연구자가 검토한 것.
# - 대상 렌즈(제목 지명)는 창 안 보유율 5.7~7.1%의 보조 변수.

# ---

# ## B. 남은 일

# ### B-1. S1 잔여 (S2 시작 전 권장)
# | 항목 | 상태 | 비고 |
# |---|---|---|
# | `00_S0/FREEZE_LOG.csv`에 PREREG `f16e76b4` 기록 확인 | **미확인** | 없으면 S0 최종 freeze 실행이 남은 일 |
# | `territory_doc` (영토·구 국가 표기 규칙 문서화) | 남음 | S7 결과 열람 전에 |
# | `recovery` (국가 미상 규칙 복구 동결, 민감도용) | 남음 | 실행 즉시 규칙 동결. 별칭·주 약어를 먼저 검토. S2의 연도별 미상 몫 점검에서 사용 |
# | 위 두 항목 동결 후 새 태그 | 남음 | `RUN_STEP4=True`로 최종 셀 재실행 |
# | `oa_raw` 별도 보관본 | 남음 | 커밋에서 제외됨 |
# | 원격 push | 남음 | 터미널에서 `git push --follow-tags` |

# ### B-2. 맥락 변수 (본 분석을 막지 않음, S2와 병행 가능)
# | 항목 | 비고 |
# |---|---|
# | `nauthors_check` | F1 `n_authors` 출처 확인. 정렬 100%의 해석에 필요. 아직 실행하지 않음 |
# | 저널 창간연도 확인 | `venue_meta.csv`의 `E1_founding`을 출판사 기록으로 확인. **pending 사건(AMAR, TR-E) 해소와 연결.** 사건 목록 수정 시 CHANGELOG 필수 |
# | JIF 수집 | `jif_template.csv`에 빈 골격 있음. 구독이 없으면 빈칸으로 두고 한계로 기록 |
# | `gazetteer_cities.csv` (선택) | 채운 뒤 원 스크립트의 `samples`를 다시 돌리면 판정 시트 3종이 새로 생성되어 지금까지의 판정이 사라짐. 반드시 백업하고 대상 렌즈 부분만 따로 실행 |

# ### B-3. S0 요약에서 넘어온 미확인 항목
# - F1 연도 대 Crossref 연도 불일치율(400편 표본): 이번 대화에서 결과가 확인되지 않음.
# - T-ITS 등 DOI 없는 논문의 추가 오귀속 점검, 번역 레코드 제외분의 영문 원본 중복 확인(S2.8).
# - `doi_fixes.csv` 13편의 치환은 `F1X`의 `doi_use`에 반영되어 있음(코드 기준).

# ---

# ## C. S1 이후 (계획서 기준)
# 1. **S2 측정 교정:** 가중치 4종, 위험군 재현율, 토픽 결측 21% 성향 가중, 비연구 항목 제거, K 민감도.
#    - **국가 미상 몫이 연도별로 크게 다름**(2007년 12.7%, 2014년 5.7%, 2023년 3.5%, 2025년 4.9%). 소득군별 몫의 시간 변화에 측정 누락 변화가 섞일 수 있으므로 연도별 미상 몫을 함께 보고하고, `recovery` 민감도로 추이가 유지되는지 확인.
# 2. **S3~S4:** 토픽 증거 카드와 신뢰 등급, 양×질 4분면, 저널 구성 분해.
# 3. **S5 체제 식별(핵심):** 연속 노출 시계열이 주 분석. 지리 변화점은 국가 층이 2007년부터라 19개 연도만 사용.
# 4. **S6~S8:** 갱신 분해, 지역 집중, 공급-부담 정렬. S8은 정책 도메인 사전 이중 코딩이라 코더 2명 필요.
# 5. **S9~S10:** 후보 풀과 추출 규칙 동결, 투고 패키지. S10.4 생성형 AI 사용 선언에는 검수 3종 사전 판정 범위를 그대로 기재.

# ---

# ## D. 참고: 동결 기록 위치
# | 항목 | 위치 |
# |---|---|
# | S1 태그 | `S0S1-freeze-2026-10-13` (커밋 `0823d46`) |
# | 보정 PREREG 커밋 | `90416c0` (2026-10-06) |
# | S1 manifest | `paper1/01_S1/S1_FREEZE_MANIFEST.csv` (16개 해시) |
# | 변경 기록 | `paper1/01_S1/CHANGELOG_S1.md` |
# | 검수 기록 | `paper1/01_S1/S1_검수판정_기준_및_확인기록.md` |
# | 통과 판정 | `paper1/01_S1/validation_report.md` |

In [4]:
# -*- coding: utf-8 -*-
# =====================================================================================
# S1 후속: territory_doc + recovery 규칙 점검(preview) 및 동결(freeze)  (OpenAlex 호출 없음)
# 사용법
#   1) FREEZE=False(기본)로 실행 -> 디스크에 아무것도 쓰지 않고 규칙을 점검만 한다.
#      점검 결과(미상 복구 표본 점검 + 국가가 이미 있는 저자 표본으로 한 정밀도 시험)를 읽는다.
#   2) 문제가 없으면 FREEZE=True로 바꿔 다시 실행 -> 규칙 동결, 문서 작성, 복구 적용, CHANGELOG, 커밋, 태그.
# 주의: 규칙을 고치려면 FREEZE=True 전에 고칠 것. 동결 파일이 생기면 수정 불가(수정 시 새 버전으로 분리).
# 필요: pip install pycountry
# =====================================================================================
import re, json, html, hashlib, subprocess, warnings
from pathlib import Path
from collections import Counter
import pandas as pd
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)

# ---------------- 설정 (여기만 수정) ----------------
FREEZE = False                 # False=점검만, True=동결
ROOT = Path.home() / "Research/Minnesota_Transport"
VALIDATE_N = 50000             # 국가가 이미 있는 저자 중 정밀도 시험에 쓸 표본 수
SEED = 2026
CRIT_OVERALL = 0.95            # 정밀도 시험 통과 기준(전체 일치율). 임의 출발값이며 결과를 보기 전에 정해 둠
CRIT_TIER = 0.90               # 티어별 기준(표본 30건 이상인 티어만)
OVERRIDE_CRITERIA = False      # True면 기준 미달이어도 동결(사유를 CHANGELOG에 남김)
COMMIT = True                  # FREEZE=True일 때 git commit + tag 수행
TAG_NAME = None                # None이면 S1-territory-recovery-freeze-YYYY-MM-DD
# ----------------------------------------------------

S0, S1 = ROOT / "paper1/00_S0", ROOT / "paper1/01_S1"
SPEC_FILE = "RECOVERY_RULES_PREREG.json"
hdr = lambda t: print("\n" + "=" * 90 + f"\n{t}\n" + "=" * 90)
now = lambda: pd.Timestamp.now().isoformat(timespec="seconds")
sha_b = lambda b: hashlib.sha256(b).hexdigest()
sha_f = lambda p: sha_b(Path(p).read_bytes())
rd = lambda p: pd.read_csv(p, dtype=str, encoding="utf-8-sig").fillna("")
class Halt(Exception): pass

def read_tbl(name):
    p = S1 / f"{name}.parquet"
    return pd.read_parquet(p) if p.exists() else pd.read_csv(S1 / f"{name}.csv.gz")

def changelog_once(key, line):
    cl = S1 / "CHANGELOG_S1.md"
    if not cl.exists(): cl.write_text("# CHANGELOG S1 (날짜 | 변경 | 사유 | 영향)\n\n", encoding="utf-8")
    if key in cl.read_text(encoding="utf-8"): return False
    with cl.open("a", encoding="utf-8") as f: f.write(line + "\n")
    return True

def git(*a, check=False):
    r = subprocess.run(["git", "-C", str(ROOT), *a], capture_output=True, text=True)
    if check and r.returncode: raise Halt(f"git {' '.join(a)} 실패: {r.stderr.strip()}")
    return r

# ============================================================
# 규칙 정의 (동결 대상)
# ============================================================
TERRITORY = {"HK", "MO", "TW", "PR", "GU", "VI", "AS", "MP", "GI", "FO", "GL", "AW", "CW", "SX", "NC", "PF", "RE", "GP", "MQ", "GF", "YT", "KY", "BM", "XK", "PS", "EH"}
FORMER = {"SU", "YU", "CS", "DD", "AN"}
AMB = ["Georgia", "Jordan", "Chad", "Niger", "Guinea", "Malta", "Cuba", "Peru", "Oman", "Togo", "Samoa"]
ALIASES = {"USA": "US", "U.S.A.": "US", "U.S.A": "US", "United States": "US", "United States of America": "US", "New Mexico": "US",
           "New Jersey": "US", "Jersey City": "US",   # 'Jersey'(JE) 오매칭 방지(대상 렌즈 오류 사례 100997과 같은 유형)
           "UK": "GB", "U.K.": "GB", "U.K": "GB", "United Kingdom": "GB", "England": "GB", "Scotland": "GB", "Wales": "GB",
           "Northern Ireland": "GB", "Great Britain": "GB", "Britain": "GB", "New South Wales": "AU", "Papua New Guinea": "PG",
           "PR China": "CN", "P.R. China": "CN", "P. R. China": "CN", "People's Republic of China": "CN", "People’s Republic of China": "CN", "China": "CN",
           "Hong Kong": "HK", "Taiwan": "TW", "Macau": "MO", "Macao": "MO", "The Netherlands": "NL", "Netherlands": "NL", "Holland": "NL",
           "Russia": "RU", "Russian Federation": "RU", "South Korea": "KR", "Republic of Korea": "KR", "Korea": "KR", "North Korea": "KP",
           "Democratic People's Republic of Korea": "KP", "Democratic People’s Republic of Korea": "KP", "Iran": "IR", "Türkiye": "TR", "Turkey": "TR",
           "Czech Republic": "CZ", "Czechia": "CZ", "Vietnam": "VN", "Viet Nam": "VN", "UAE": "AE", "United Arab Emirates": "AE", "Deutschland": "DE",
           "España": "ES", "Italia": "IT", "México": "MX", "Brasil": "BR"}
# 영토 코드가 잡히면 모국 코드는 제외(영토는 모국과 합치지 않는다는 TERRITORY_RULES 1번과 일관). 예: "Hong Kong, China" -> HK만
PARENT = {"HK": "CN", "MO": "CN", "TW": "CN", "PR": "US", "GU": "US", "VI": "US", "AS": "US", "MP": "US", "GI": "GB", "KY": "GB", "BM": "GB",
          "FO": "DK", "GL": "DK", "AW": "NL", "CW": "NL", "SX": "NL", "NC": "FR", "PF": "FR", "RE": "FR", "GP": "FR", "MQ": "FR", "GF": "FR", "YT": "FR"}
US_ABBR = "AL AK AZ AR CA CO CT DE FL GA HI ID IL IN IA KS KY LA ME MD MA MI MN MS MO MT NE NV NH NJ NM NY NC ND OH OK OR PA RI SC SD TN TX UT VT VA WA WV WI WY DC".split()
US_NAMES = ["Alabama", "Alaska", "Arizona", "Arkansas", "California", "Colorado", "Connecticut", "Delaware", "Florida", "Hawaii", "Idaho", "Illinois", "Indiana", "Iowa",
            "Kansas", "Kentucky", "Louisiana", "Maine", "Maryland", "Massachusetts", "Michigan", "Minnesota", "Mississippi", "Missouri", "Montana", "Nebraska", "Nevada",
            "New Hampshire", "New Jersey", "New York", "North Carolina", "North Dakota", "Ohio", "Oklahoma", "Oregon", "Pennsylvania", "Rhode Island", "South Carolina",
            "South Dakota", "Tennessee", "Texas", "Utah", "Vermont", "Virginia", "Washington", "West Virginia", "Wisconsin", "Wyoming"]
CA_ABBR = "ON QC BC AB NS MB NB NL PE SK YT NT NU".split()
CA_NAMES = ["Ontario", "Quebec", "Québec", "British Columbia", "Alberta", "Nova Scotia", "Manitoba", "Saskatchewan", "New Brunswick"]

def make_spec():
    try: import pycountry
    except Exception: raise Halt("pycountry가 없습니다: pip install pycountry (없으면 국가명 목록이 빠진 약한 규칙이 동결됨)")
    names = {}
    for c in pycountry.countries:
        for nm in {c.name, getattr(c, "common_name", c.name)}:
            if len(nm) >= 4 and nm not in AMB and "," not in nm: names[nm] = c.alpha_2
    spec = dict(version=1, created=now(), pycountry=getattr(pycountry, "__version__", "unknown"), country_names=names, aliases=ALIASES, parent_drop=PARENT,
                ambiguous=AMB, us_abbr=US_ABBR, us_names=US_NAMES, ca_abbr=CA_ABBR, ca_names=CA_NAMES,
                tiers={"1": "국가명·별칭(단어 경계), 영토가 잡히면 모국 제외", "2": "미국 주 약어·주 이름(쉼표 뒤, 문자열 끝 또는 || 앞, 우편번호 허용)", "3": "캐나다 주 이름 또는 주 약어+우편번호"})
    spec["sha256"] = spec_hash(spec); return spec

def spec_hash(spec):
    return sha_b(json.dumps({k: v for k, v in spec.items() if k not in ("sha256", "created")}, sort_keys=True, ensure_ascii=False).encode())

class Recoverer:
    def __init__(self, spec):
        al = {**spec["country_names"], **spec["aliases"]}; self.al = {k.lower(): v for k, v in al.items()}; self.parent = spec["parent_drop"]
        alt = "|".join(re.escape(a) for a in sorted(al, key=len, reverse=True))
        self.t1 = re.compile(r"(?<![A-Za-z])(" + alt + r")(?![A-Za-z])", re.I)
        end = r"(?:[\s,]+\d{5}(?:-\d{4})?)?\s*(?=\|\||;|\.|$)"
        self.us_abbr = re.compile(r",\s*(" + "|".join(spec["us_abbr"]) + r")\b" + end)
        self.us_name = re.compile(r",\s*(" + "|".join(sorted(spec["us_names"], key=len, reverse=True)) + r")\b" + end)
        self.ca_name = re.compile(r"(?<![A-Za-z])(" + "|".join(spec["ca_names"]) + r")(?![A-Za-z])")
        self.ca_post = re.compile(r",\s*(" + "|".join(spec["ca_abbr"]) + r")\b[\s,]+[A-Z]\d[A-Z]\s?\d[A-Z]\d")
    def __call__(self, aff):
        s = html.unescape(aff or "").replace("#TAB#", " ").strip()
        if not s: return [], 0
        cs = {self.al[m.lower()] for m in self.t1.findall(s)}
        if cs:
            for c in list(cs):
                if self.parent.get(c) in cs: cs.discard(self.parent[c])
            return sorted(cs), 1
        if self.us_abbr.search(s) or self.us_name.search(s): return ["US"], 2
        if self.ca_name.search(s) or self.ca_post.search(s): return ["CA"], 3
        return [], 0

# ============================================================
# preview: 디스크에 쓰지 않고 규칙 점검
# ============================================================
def preview(R, spec):
    hdr("[preview] 규칙 점검 (디스크에 쓰지 않음)")
    print(f"국가명 {len(spec['country_names'])}개, 별칭 {len(spec['aliases'])}개, 영토→모국 제외 {len(spec['parent_drop'])}개, 모호어 제외 {spec['ambiguous']}")
    print(f"규칙 해시(예정) {spec['sha256'][:12]} | pycountry {spec['pycountry']}")
    ok = True
    uc = S1 / "unknown_check.csv"
    if uc.exists():
        U = rd(uc); U = U[(U.missed == "1") & (U.true_country != "")]
        res = [R(a) for a in U.raw_aff]; rec = [bool(c) for c, _ in res]
        hit = [set(c) == set(t.split("|")) for (c, _), t in zip(res, U.true_country)]
        rc = [h for h, r in zip(hit, rec) if r]
        print(f"\n[A. unknown_check 표본] 판단 가능 {len(U)}건 중 규칙 복구 {sum(rec)}건 ({sum(rec) / max(len(U), 1):.1%}), 복구분 정답 {sum(rc)}/{len(rc)}")
        print("※ 이 표본은 규칙 설계에 참고했으므로 in-sample(낙관적)입니다. 판단은 아래 B를 보세요.")
        for (c, t), tc, a in zip(res, U.true_country, U.raw_aff):
            if c and set(c) != set(tc.split("|")): print("  오답:", c, "정답", tc, "|", a[:90])
    A = read_tbl("authorship_level"); A["countries"] = A.countries.fillna(""); A["raw_aff"] = A.raw_aff.fillna("")
    K = A[(A.countries != "") & (A.raw_aff != "")]
    K = K.sample(min(VALIDATE_N, len(K)), random_state=SEED).copy()
    out = [R(a) for a in K.raw_aff]; K["rec"] = ["|".join(c) for c, _ in out]; K["tier"] = [t for _, t in out]
    got = K[K.tier > 0].copy(); got["exact"] = [set(r.split("|")) == set(k.split("|")) for r, k in zip(got.rec, got.countries)]
    print(f"\n[B. 국가가 이미 있는 저자 {len(K):,}명으로 정밀도 시험] 규칙이 값을 낸 경우 {len(got):,}명({len(got) / len(K):.1%})")
    print("※ OpenAlex가 소속 2개 이상에서 국가를 만든 경우 불일치로 잡힐 수 있어 정밀도의 하한에 가깝습니다.")
    tot = got.exact.mean() if len(got) else float("nan"); print(f"전체 일치율 {tot:.3f} (기준 {CRIT_OVERALL})")
    if not (tot >= CRIT_OVERALL): ok = False
    for t, g in got.groupby("tier"):
        a = g.exact.mean(); judged = len(g) >= 30
        print(f"  티어 {t}: n={len(g):,}, 일치율 {a:.3f}" + (f" (기준 {CRIT_TIER})" if judged else " (표본<30, 기준 미적용)"))
        if judged and a < CRIT_TIER: ok = False
    bad = got[~got.exact]
    if len(bad):
        print("불일치 쌍 상위(규칙 -> OpenAlex):", Counter(zip(bad.rec, bad.countries)).most_common(10))
        print(bad[["raw_aff", "rec", "countries"]].head(8).to_string(index=False, max_colwidth=70))
    print("\n판정:", "기준 통과" if ok else "기준 미달 -> 규칙을 고친 뒤 다시 점검하세요(고쳤다면 CHANGELOG에 '점검 후 수정'으로 남길 것)")
    return ok

# ============================================================
# territory_doc
# ============================================================
def write_territory_doc():
    txt = ["# 영토·구 국가 표기 규칙 (S1)", f"- 작성: {now()}  (S7 결과 열람 전)", "",
           "## 규칙", "1. 국가는 OpenAlex가 부여한 `country_code`를 그대로 쓴다. 영토(HK, TW, PR 등)는 모국과 합치지 않고 별도 국가로 둔다.",
           "2. 영토 코드에는 `is_territory=True` 열을 붙여 민감도 분석에서 분리할 수 있게 한다.",
           "3. 구 국가 코드(SU, YU, CS, DD, AN)는 `ZZ_former`로 묶고 승계국에 배분하지 않는다.",
           "4. 다중 소속 저자는 소속 국가에 균등 분할한다. 소속이 없는 몫은 `UNK`로 유지한다.",
           "5. 소속 국가는 국적이 아니다.",
           "6. 국가 미상 복구(recovery) 규칙에서도 영토 코드가 잡히면 모국 코드는 제외한다(예: 'Hong Kong, China' -> HK). 복구는 민감도 분석에서만 쓰고 주 분석 테이블은 바꾸지 않는다.",
           "", "## 영토 코드 목록", ", ".join(sorted(TERRITORY)), "", "## 구 국가 코드 목록", ", ".join(sorted(FORMER)), ""]
    try:
        P = read_tbl("country_authorship"); P["ws"] = P.w * P.samp_w; tot = P.ws.sum()
        terr = P[P.is_territory].groupby("country").ws.sum().sort_values(ascending=False) / tot
        fm = P[P.country == "ZZ_former"].ws.sum() / tot
        txt += ["## 현재 자료의 규모 (기술 통계, 가설 검정 아님)", f"- 영토 합계 몫: {terr.sum():.4f}", f"- 구 국가(ZZ_former) 몫: {fm:.5f}",
                "- 영토별 몫: " + ", ".join(f"{c} {v:.4f}" for c, v in terr.head(8).items()), ""]
        print(f"영토 합계 {terr.sum():.4f} | ZZ_former {fm:.5f}")
    except Exception as e: print("※ country_authorship 읽기 실패, 규모 절 생략:", e)
    (S1 / "TERRITORY_RULES.md").write_text("\n".join(txt), encoding="utf-8"); print("저장: TERRITORY_RULES.md")

# ============================================================
# freeze
# ============================================================
def freeze(R, spec, ok):
    hdr("[freeze] 규칙 동결 + 문서 + 복구 적용 + 기록")
    if not ok and not OVERRIDE_CRITERIA: raise Halt("정밀도 시험 기준 미달: 규칙을 고치거나 OVERRIDE_CRITERIA=True(사유를 기록)로 동결하세요")
    p = S1 / SPEC_FILE
    if p.exists():
        old = json.loads(p.read_text(encoding="utf-8"))
        if spec_hash({k: v for k, v in old.items()}) != old.get("sha256"): raise Halt(f"{SPEC_FILE}이 동결 후 변경되었습니다")
        if old["sha256"] != spec["sha256"]: raise Halt(f"이미 동결된 규칙({old['sha256'][:12]})과 현재 규칙({spec['sha256'][:12]})이 다릅니다. 새 버전으로 분리하고 CHANGELOG에 사유를 남기세요")
        print(f"기존 동결 규칙과 동일({old['sha256'][:12]}): 재사용")
    else:
        p.write_text(json.dumps(spec, ensure_ascii=False, indent=1), encoding="utf-8"); print(f"규칙 동결 기록: {SPEC_FILE} (sha256 {spec['sha256'][:12]})")
    write_territory_doc()
    A = read_tbl("authorship_level"); A["countries"] = A.countries.fillna(""); A["raw_aff"] = A.raw_aff.fillna("")
    T = A[(A.countries == "") & (A.raw_aff != "")][["pid", "rank", "raw_aff"]].copy()
    out = [R(a) for a in T.raw_aff]; T["countries_rec"] = ["|".join(c) for c, _ in out]; T["rec_tier"] = [t for _, t in out]
    T.to_csv(S1 / "authorship_level_recovered.csv.gz", index=False, compression="gzip")
    n, k = len(T), int((T.rec_tier > 0).sum())
    print(f"소속 문자열은 있으나 국가 없음 {n:,}건 중 복구 {k:,}건 ({k / max(n, 1):.1%}) | 티어별 {T.rec_tier.value_counts().sort_index().to_dict()}")
    print("복구 국가 상위:", T[T.countries_rec != ""].countries_rec.value_counts().head(8).to_dict())
    changelog_once("territory_doc·recovery 동결",
        f"- {now()} | territory_doc·recovery 동결: 영토·구 국가 표기 규칙(TERRITORY_RULES.md)과 국가 미상 규칙 복구 규칙({SPEC_FILE}, sha256 {spec['sha256'][:12]})을 S7 결과 열람 전에 기록. 복구는 민감도용이며 주 분석 테이블은 변경하지 않음. 규칙은 unknown_check 100건(in-sample)을 참고해 설계했고, 동결 전에 국가가 이미 있는 저자 표본으로 정밀도를 점검함. 점검 중 발견한 오매칭 유형(New Jersey->JE, 'Hong Kong, China'의 CN 중복)을 별칭과 영토-모국 제외 규칙으로 반영함{' (기준 미달을 OVERRIDE로 동결)' if (not ok) else ''} | 규칙 사전 기록 | S2 연도별 미상 몫 점검, S7 민감도")
    keys = [SPEC_FILE, "TERRITORY_RULES.md", "authorship_level_recovered.csv.gz", "CHANGELOG_S1.md"]
    pd.DataFrame([dict(file=f"01_S1/{f}", sha256=sha_f(S1 / f)) for f in keys]).assign(at=now()).to_csv(S1 / "S1_FREEZE_MANIFEST_territory_recovery.csv", index=False)
    print("S1_FREEZE_MANIFEST_territory_recovery.csv 기록")
    if not COMMIT: print("COMMIT=False: 커밋·태그는 하지 않았습니다"); return
    if git("rev-parse", "--show-toplevel").returncode: raise Halt("git 저장소가 아닙니다")
    if not git("config", "user.name").stdout.strip() or not git("config", "user.email").stdout.strip(): raise Halt("git user.name/user.email을 먼저 설정하세요")
    tag = TAG_NAME or f"S1-territory-recovery-freeze-{pd.Timestamp.now().strftime('%Y-%m-%d')}"
    if git("tag", "-l", tag).stdout.strip(): raise Halt(f"태그 {tag}가 이미 있습니다(TAG_NAME을 바꾸세요)")
    spec_p = ["--", "paper1/01_S1", ":(exclude)paper1/01_S1/oa_raw", ":(exclude)paper1/01_S1/_backup_*"]
    print(git("status", "--short", *spec_p).stdout or "(변경 없음)")
    git("add", *spec_p, check=True)
    c = git("commit", "-m", "S1 후속 freeze: 영토·구 국가 표기 규칙, 국가 미상 복구 규칙 동결(민감도용)")
    print((c.stdout.strip() or c.stderr.strip())[:500])
    if c.returncode and "nothing to commit" not in (c.stdout + c.stderr): raise Halt("커밋 실패")
    git("tag", "-a", tag, "-m", "S1 territory/recovery freeze", check=True)
    print(f"태그 생성: {tag} | commit {git('rev-parse', '--short', 'HEAD').stdout.strip()} | 원격 반영: git push --follow-tags")

assert S0.exists() and S1.exists(), "ROOT를 확인하세요"
print(f"ROOT={ROOT} | FREEZE={FREEZE}")
try:
    spec = make_spec(); R = Recoverer(spec)
    ok = preview(R, spec)
    if FREEZE: freeze(R, spec, ok); print("\n✔ 동결 완료")
    else: print("\n(점검만 수행: 디스크에 아무것도 쓰지 않았습니다. 문제가 없으면 FREEZE=True로 바꿔 다시 실행하세요)")
except Halt as e:
    print(f"\n⛔ 중단: {e}")

ROOT=/home/yjlee/Research/Minnesota_Transport | FREEZE=False

[preview] 규칙 점검 (디스크에 쓰지 않음)
국가명 234개, 별칭 55개, 영토→모국 제외 23개, 모호어 제외 ['Georgia', 'Jordan', 'Chad', 'Niger', 'Guinea', 'Malta', 'Cuba', 'Peru', 'Oman', 'Togo', 'Samoa']
규칙 해시(예정) 6835d25194ae | pycountry 26.2.16

[A. unknown_check 표본] 판단 가능 97건 중 규칙 복구 97건 (100.0%), 복구분 정답 97/97
※ 이 표본은 규칙 설계에 참고했으므로 in-sample(낙관적)입니다. 판단은 아래 B를 보세요.

[B. 국가가 이미 있는 저자 50,000명으로 정밀도 시험] 규칙이 값을 낸 경우 49,465명(98.9%)
※ OpenAlex가 소속 2개 이상에서 국가를 만든 경우 불일치로 잡힐 수 있어 정밀도의 하한에 가깝습니다.
전체 일치율 0.983 (기준 0.95)
  티어 1: n=44,009, 일치율 0.982 (기준 0.9)
  티어 2: n=5,447, 일치율 0.996 (기준 0.9)
  티어 3: n=9, 일치율 0.889 (표본<30, 기준 미적용)
불일치 쌍 상위(규칙 -> OpenAlex): [(('HK', 'CN|HK'), 131), (('CN', 'CN|US'), 71), (('HK', 'CN'), 54), (('IT', 'BE|IT'), 33), (('CN|MN', 'CN'), 21), (('MO', 'CN|MO'), 19), (('US', 'CN|US'), 14), (('GB|US', 'US'), 14), (('HK|MO', 'CN'), 10), (('CN', 'US'), 8)]
                                                               raw_aff   rec countries
Curren

In [5]:
# -*- coding: utf-8 -*-
# =====================================================================================
# S1 후속: territory_doc + recovery 규칙 점검(preview) 및 동결(freeze)  (OpenAlex 호출 없음)
# 사용법
#   1) FREEZE=False(기본)로 실행 -> 디스크에 아무것도 쓰지 않고 규칙을 점검만 한다.
#      점검 결과(미상 복구 표본 점검 + 국가가 이미 있는 저자 표본으로 한 정밀도 시험)를 읽는다.
#   2) 문제가 없으면 FREEZE=True로 바꿔 다시 실행 -> 규칙 동결, 문서 작성, 복구 적용, CHANGELOG, 커밋, 태그.
# 주의: 규칙을 고치려면 FREEZE=True 전에 고칠 것. 동결 파일이 생기면 수정 불가(수정 시 새 버전으로 분리).
# 필요: pip install pycountry
# =====================================================================================
import re, json, html, hashlib, subprocess, warnings
from pathlib import Path
from collections import Counter
import pandas as pd
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)

# ---------------- 설정 (여기만 수정) ----------------
FREEZE = False                 # False=점검만, True=동결
ROOT = Path.home() / "Research/Minnesota_Transport"
VALIDATE_N = 50000             # 국가가 이미 있는 저자 중 정밀도 시험에 쓸 표본 수
SEED = 2026
CRIT_OVERALL = 0.95            # 정밀도 시험 통과 기준(전체 일치율). 임의 출발값이며 결과를 보기 전에 정해 둠
CRIT_TIER = 0.90               # 티어별 기준(표본 30건 이상인 티어만)
OVERRIDE_CRITERIA = False      # True면 기준 미달이어도 동결(사유를 CHANGELOG에 남김)
COMMIT = True                  # FREEZE=True일 때 git commit + tag 수행
TAG_NAME = None                # None이면 S1-territory-recovery-freeze-YYYY-MM-DD
# ----------------------------------------------------

S0, S1 = ROOT / "paper1/00_S0", ROOT / "paper1/01_S1"
SPEC_FILE = "RECOVERY_RULES_PREREG.json"
hdr = lambda t: print("\n" + "=" * 90 + f"\n{t}\n" + "=" * 90)
now = lambda: pd.Timestamp.now().isoformat(timespec="seconds")
sha_b = lambda b: hashlib.sha256(b).hexdigest()
sha_f = lambda p: sha_b(Path(p).read_bytes())
rd = lambda p: pd.read_csv(p, dtype=str, encoding="utf-8-sig").fillna("")
class Halt(Exception): pass

def read_tbl(name):
    p = S1 / f"{name}.parquet"
    return pd.read_parquet(p) if p.exists() else pd.read_csv(S1 / f"{name}.csv.gz")

def changelog_once(key, line):
    cl = S1 / "CHANGELOG_S1.md"
    if not cl.exists(): cl.write_text("# CHANGELOG S1 (날짜 | 변경 | 사유 | 영향)\n\n", encoding="utf-8")
    if key in cl.read_text(encoding="utf-8"): return False
    with cl.open("a", encoding="utf-8") as f: f.write(line + "\n")
    return True

def git(*a, check=False):
    r = subprocess.run(["git", "-C", str(ROOT), *a], capture_output=True, text=True)
    if check and r.returncode: raise Halt(f"git {' '.join(a)} 실패: {r.stderr.strip()}")
    return r

# ============================================================
# 규칙 정의 (동결 대상)
# ============================================================
TERRITORY = {"HK", "MO", "TW", "PR", "GU", "VI", "AS", "MP", "GI", "FO", "GL", "AW", "CW", "SX", "NC", "PF", "RE", "GP", "MQ", "GF", "YT", "KY", "BM", "XK", "PS", "EH"}
FORMER = {"SU", "YU", "CS", "DD", "AN"}
AMB = ["Georgia", "Jordan", "Chad", "Niger", "Guinea", "Malta", "Cuba", "Peru", "Oman", "Togo", "Samoa"]
ALIASES = {"USA": "US", "U.S.A.": "US", "U.S.A": "US", "United States": "US", "United States of America": "US", "New Mexico": "US",
           "New Jersey": "US", "Jersey City": "US",   # 'Jersey'(JE) 오매칭 방지(대상 렌즈 오류 사례 100997과 같은 유형)
           "Inner Mongolia": "CN",                      # 'Mongolia'(MN) 오매칭 방지(정밀도 시험에서 CN|MN 21건 발견)
           "UK": "GB", "U.K.": "GB", "U.K": "GB", "United Kingdom": "GB", "England": "GB", "Scotland": "GB", "Wales": "GB",
           "Northern Ireland": "GB", "Great Britain": "GB", "Britain": "GB", "New South Wales": "AU", "Papua New Guinea": "PG",
           "PR China": "CN", "P.R. China": "CN", "P. R. China": "CN", "People's Republic of China": "CN", "People’s Republic of China": "CN", "China": "CN",
           "Hong Kong": "HK", "Taiwan": "TW", "Macau": "MO", "Macao": "MO", "The Netherlands": "NL", "Netherlands": "NL", "Holland": "NL",
           "Russia": "RU", "Russian Federation": "RU", "South Korea": "KR", "Republic of Korea": "KR", "Korea": "KR", "North Korea": "KP",
           "Democratic People's Republic of Korea": "KP", "Democratic People’s Republic of Korea": "KP", "Iran": "IR", "Türkiye": "TR", "Turkey": "TR",
           "Czech Republic": "CZ", "Czechia": "CZ", "Vietnam": "VN", "Viet Nam": "VN", "UAE": "AE", "United Arab Emirates": "AE", "Deutschland": "DE",
           "España": "ES", "Italia": "IT", "México": "MX", "Brasil": "BR"}
# 영토 코드가 잡히면 모국 코드는 제외(영토는 모국과 합치지 않는다는 TERRITORY_RULES 1번과 일관). 예: "Hong Kong, China" -> HK만
PARENT = {"HK": "CN", "MO": "CN", "TW": "CN", "PR": "US", "GU": "US", "VI": "US", "AS": "US", "MP": "US", "GI": "GB", "KY": "GB", "BM": "GB",
          "FO": "DK", "GL": "DK", "AW": "NL", "CW": "NL", "SX": "NL", "NC": "FR", "PF": "FR", "RE": "FR", "GP": "FR", "MQ": "FR", "GF": "FR", "YT": "FR"}
US_ABBR = "AL AK AZ AR CA CO CT DE FL GA HI ID IL IN IA KS KY LA ME MD MA MI MN MS MO MT NE NV NH NJ NM NY NC ND OH OK OR PA RI SC SD TN TX UT VT VA WA WV WI WY DC".split()
US_NAMES = ["Alabama", "Alaska", "Arizona", "Arkansas", "California", "Colorado", "Connecticut", "Delaware", "Florida", "Hawaii", "Idaho", "Illinois", "Indiana", "Iowa",
            "Kansas", "Kentucky", "Louisiana", "Maine", "Maryland", "Massachusetts", "Michigan", "Minnesota", "Mississippi", "Missouri", "Montana", "Nebraska", "Nevada",
            "New Hampshire", "New Jersey", "New York", "North Carolina", "North Dakota", "Ohio", "Oklahoma", "Oregon", "Pennsylvania", "Rhode Island", "South Carolina",
            "South Dakota", "Tennessee", "Texas", "Utah", "Vermont", "Virginia", "Washington", "West Virginia", "Wisconsin", "Wyoming"]
CA_ABBR = "ON QC BC AB NS MB NB NL PE SK YT NT NU".split()
CA_NAMES = ["Ontario", "Quebec", "Québec", "British Columbia", "Alberta", "Nova Scotia", "Manitoba", "Saskatchewan", "New Brunswick"]

def make_spec():
    try: import pycountry
    except Exception: raise Halt("pycountry가 없습니다: pip install pycountry (없으면 국가명 목록이 빠진 약한 규칙이 동결됨)")
    names = {}
    for c in pycountry.countries:
        for nm in {c.name, getattr(c, "common_name", c.name)}:
            if len(nm) >= 4 and nm not in AMB and "," not in nm: names[nm] = c.alpha_2
    spec = dict(version=1, created=now(), pycountry=getattr(pycountry, "__version__", "unknown"), country_names=names, aliases=ALIASES, parent_drop=PARENT,
                ambiguous=AMB, us_abbr=US_ABBR, us_names=US_NAMES, ca_abbr=CA_ABBR, ca_names=CA_NAMES,
                tiers={"1": "국가명·별칭(단어 경계), 영토가 잡히면 모국 제외", "2": "미국 주 약어·주 이름(쉼표 뒤, 문자열 끝 또는 || 앞, 우편번호 허용)", "3": "캐나다 주 이름 또는 주 약어+우편번호"})
    spec["sha256"] = spec_hash(spec); return spec

def spec_hash(spec):
    return sha_b(json.dumps({k: v for k, v in spec.items() if k not in ("sha256", "created")}, sort_keys=True, ensure_ascii=False).encode())

class Recoverer:
    def __init__(self, spec):
        al = {**spec["country_names"], **spec["aliases"]}; self.al = {k.lower(): v for k, v in al.items()}; self.parent = spec["parent_drop"]
        alt = "|".join(re.escape(a) for a in sorted(al, key=len, reverse=True))
        self.t1 = re.compile(r"(?<![A-Za-z])(" + alt + r")(?![A-Za-z])", re.I)
        end = r"(?:[\s,]+\d{5}(?:-\d{4})?)?\s*(?=\|\||;|\.|$)"
        self.us_abbr = re.compile(r",\s*(" + "|".join(spec["us_abbr"]) + r")\b" + end)
        self.us_name = re.compile(r",\s*(" + "|".join(sorted(spec["us_names"], key=len, reverse=True)) + r")\b" + end)
        self.ca_name = re.compile(r"(?<![A-Za-z])(" + "|".join(spec["ca_names"]) + r")(?![A-Za-z])")
        self.ca_post = re.compile(r",\s*(" + "|".join(spec["ca_abbr"]) + r")\b[\s,]+[A-Z]\d[A-Z]\s?\d[A-Z]\d")
    def __call__(self, aff):
        s = html.unescape(aff or "").replace("#TAB#", " ").strip()
        if not s: return [], 0
        cs = {self.al[m.lower()] for m in self.t1.findall(s)}
        if cs:
            for c in list(cs):
                if self.parent.get(c) in cs: cs.discard(self.parent[c])
            return sorted(cs), 1
        if self.us_abbr.search(s) or self.us_name.search(s): return ["US"], 2
        if self.ca_name.search(s) or self.ca_post.search(s): return ["CA"], 3
        return [], 0

# ============================================================
# preview: 디스크에 쓰지 않고 규칙 점검
# ============================================================
def preview(R, spec):
    hdr("[preview] 규칙 점검 (디스크에 쓰지 않음)")
    print(f"국가명 {len(spec['country_names'])}개, 별칭 {len(spec['aliases'])}개, 영토→모국 제외 {len(spec['parent_drop'])}개, 모호어 제외 {spec['ambiguous']}")
    print(f"규칙 해시(예정) {spec['sha256'][:12]} | pycountry {spec['pycountry']}")
    ok = True
    uc = S1 / "unknown_check.csv"
    if uc.exists():
        U = rd(uc); U = U[(U.missed == "1") & (U.true_country != "")]
        res = [R(a) for a in U.raw_aff]; rec = [bool(c) for c, _ in res]
        hit = [set(c) == set(t.split("|")) for (c, _), t in zip(res, U.true_country)]
        rc = [h for h, r in zip(hit, rec) if r]
        print(f"\n[A. unknown_check 표본] 판단 가능 {len(U)}건 중 규칙 복구 {sum(rec)}건 ({sum(rec) / max(len(U), 1):.1%}), 복구분 정답 {sum(rc)}/{len(rc)}")
        print("※ 이 표본은 규칙 설계에 참고했으므로 in-sample(낙관적)입니다. 판단은 아래 B를 보세요.")
        for (c, t), tc, a in zip(res, U.true_country, U.raw_aff):
            if c and set(c) != set(tc.split("|")): print("  오답:", c, "정답", tc, "|", a[:90])
    A = read_tbl("authorship_level"); A["countries"] = A.countries.fillna(""); A["raw_aff"] = A.raw_aff.fillna("")
    K = A[(A.countries != "") & (A.raw_aff != "")]
    K = K.sample(min(VALIDATE_N, len(K)), random_state=SEED).copy()
    out = [R(a) for a in K.raw_aff]; K["rec"] = ["|".join(c) for c, _ in out]; K["tier"] = [t for _, t in out]
    got = K[K.tier > 0].copy(); got["exact"] = [set(r.split("|")) == set(k.split("|")) for r, k in zip(got.rec, got.countries)]
    print(f"\n[B. 국가가 이미 있는 저자 {len(K):,}명으로 정밀도 시험] 규칙이 값을 낸 경우 {len(got):,}명({len(got) / len(K):.1%})")
    print("※ OpenAlex가 소속 2개 이상에서 국가를 만든 경우 불일치로 잡힐 수 있어 정밀도의 하한에 가깝습니다.")
    tot = got.exact.mean() if len(got) else float("nan"); print(f"전체 일치율 {tot:.3f} (기준 {CRIT_OVERALL})")
    if not (tot >= CRIT_OVERALL): ok = False
    for t, g in got.groupby("tier"):
        a = g.exact.mean(); judged = len(g) >= 30
        print(f"  티어 {t}: n={len(g):,}, 일치율 {a:.3f}" + (f" (기준 {CRIT_TIER})" if judged else " (표본<30, 기준 미적용)"))
        if judged and a < CRIT_TIER: ok = False
    bad = got[~got.exact]
    if len(bad):
        print("불일치 쌍 상위(규칙 -> OpenAlex):", Counter(zip(bad.rec, bad.countries)).most_common(10))
        print(bad[["raw_aff", "rec", "countries"]].head(8).to_string(index=False, max_colwidth=70))
    print("\n판정:", "기준 통과" if ok else "기준 미달 -> 규칙을 고친 뒤 다시 점검하세요(고쳤다면 CHANGELOG에 '점검 후 수정'으로 남길 것)")
    return ok

# ============================================================
# territory_doc
# ============================================================
def write_territory_doc():
    txt = ["# 영토·구 국가 표기 규칙 (S1)", f"- 작성: {now()}  (S7 결과 열람 전)", "",
           "## 규칙", "1. 국가는 OpenAlex가 부여한 `country_code`를 그대로 쓴다. 영토(HK, TW, PR 등)는 모국과 합치지 않고 별도 국가로 둔다.",
           "2. 영토 코드에는 `is_territory=True` 열을 붙여 민감도 분석에서 분리할 수 있게 한다.",
           "3. 구 국가 코드(SU, YU, CS, DD, AN)는 `ZZ_former`로 묶고 승계국에 배분하지 않는다.",
           "4. 다중 소속 저자는 소속 국가에 균등 분할한다. 소속이 없는 몫은 `UNK`로 유지한다.",
           "5. 소속 국가는 국적이 아니다.",
           "6. 국가 미상 복구(recovery) 규칙에서도 영토 코드가 잡히면 모국 코드는 제외한다(예: 'Hong Kong, China' -> HK). 복구는 민감도 분석에서만 쓰고 주 분석 테이블은 바꾸지 않는다.",
           "", "## 영토 코드 목록", ", ".join(sorted(TERRITORY)), "", "## 구 국가 코드 목록", ", ".join(sorted(FORMER)), ""]
    try:
        P = read_tbl("country_authorship"); P["ws"] = P.w * P.samp_w; tot = P.ws.sum()
        terr = P[P.is_territory].groupby("country").ws.sum().sort_values(ascending=False) / tot
        fm = P[P.country == "ZZ_former"].ws.sum() / tot
        txt += ["## 현재 자료의 규모 (기술 통계, 가설 검정 아님)", f"- 영토 합계 몫: {terr.sum():.4f}", f"- 구 국가(ZZ_former) 몫: {fm:.5f}",
                "- 영토별 몫: " + ", ".join(f"{c} {v:.4f}" for c, v in terr.head(8).items()), ""]
        print(f"영토 합계 {terr.sum():.4f} | ZZ_former {fm:.5f}")
    except Exception as e: print("※ country_authorship 읽기 실패, 규모 절 생략:", e)
    (S1 / "TERRITORY_RULES.md").write_text("\n".join(txt), encoding="utf-8"); print("저장: TERRITORY_RULES.md")

# ============================================================
# freeze
# ============================================================
def freeze(R, spec, ok):
    hdr("[freeze] 규칙 동결 + 문서 + 복구 적용 + 기록")
    if not ok and not OVERRIDE_CRITERIA: raise Halt("정밀도 시험 기준 미달: 규칙을 고치거나 OVERRIDE_CRITERIA=True(사유를 기록)로 동결하세요")
    p = S1 / SPEC_FILE
    if p.exists():
        old = json.loads(p.read_text(encoding="utf-8"))
        if spec_hash({k: v for k, v in old.items()}) != old.get("sha256"): raise Halt(f"{SPEC_FILE}이 동결 후 변경되었습니다")
        if old["sha256"] != spec["sha256"]: raise Halt(f"이미 동결된 규칙({old['sha256'][:12]})과 현재 규칙({spec['sha256'][:12]})이 다릅니다. 새 버전으로 분리하고 CHANGELOG에 사유를 남기세요")
        print(f"기존 동결 규칙과 동일({old['sha256'][:12]}): 재사용")
    else:
        p.write_text(json.dumps(spec, ensure_ascii=False, indent=1), encoding="utf-8"); print(f"규칙 동결 기록: {SPEC_FILE} (sha256 {spec['sha256'][:12]})")
    write_territory_doc()
    A = read_tbl("authorship_level"); A["countries"] = A.countries.fillna(""); A["raw_aff"] = A.raw_aff.fillna("")
    T = A[(A.countries == "") & (A.raw_aff != "")][["pid", "rank", "raw_aff"]].copy()
    out = [R(a) for a in T.raw_aff]; T["countries_rec"] = ["|".join(c) for c, _ in out]; T["rec_tier"] = [t for _, t in out]
    T.to_csv(S1 / "authorship_level_recovered.csv.gz", index=False, compression="gzip")
    n, k = len(T), int((T.rec_tier > 0).sum())
    print(f"소속 문자열은 있으나 국가 없음 {n:,}건 중 복구 {k:,}건 ({k / max(n, 1):.1%}) | 티어별 {T.rec_tier.value_counts().sort_index().to_dict()}")
    print("복구 국가 상위:", T[T.countries_rec != ""].countries_rec.value_counts().head(8).to_dict())
    changelog_once("territory_doc·recovery 동결",
        f"- {now()} | territory_doc·recovery 동결: 영토·구 국가 표기 규칙(TERRITORY_RULES.md)과 국가 미상 규칙 복구 규칙({SPEC_FILE}, sha256 {spec['sha256'][:12]})을 S7 결과 열람 전에 기록. 복구는 민감도용이며 주 분석 테이블은 변경하지 않음. 규칙은 unknown_check 100건(in-sample)을 참고해 설계했고, 동결 전에 국가가 이미 있는 저자 표본으로 정밀도를 점검함. 점검 중 발견한 오매칭 유형(New Jersey->JE, 'Hong Kong, China'의 CN 중복)을 별칭과 영토-모국 제외 규칙으로 반영함{' (기준 미달을 OVERRIDE로 동결)' if (not ok) else ''} | 규칙 사전 기록 | S2 연도별 미상 몫 점검, S7 민감도")
    keys = [SPEC_FILE, "TERRITORY_RULES.md", "authorship_level_recovered.csv.gz", "CHANGELOG_S1.md"]
    pd.DataFrame([dict(file=f"01_S1/{f}", sha256=sha_f(S1 / f)) for f in keys]).assign(at=now()).to_csv(S1 / "S1_FREEZE_MANIFEST_territory_recovery.csv", index=False)
    print("S1_FREEZE_MANIFEST_territory_recovery.csv 기록")
    if not COMMIT: print("COMMIT=False: 커밋·태그는 하지 않았습니다"); return
    if git("rev-parse", "--show-toplevel").returncode: raise Halt("git 저장소가 아닙니다")
    if not git("config", "user.name").stdout.strip() or not git("config", "user.email").stdout.strip(): raise Halt("git user.name/user.email을 먼저 설정하세요")
    tag = TAG_NAME or f"S1-territory-recovery-freeze-{pd.Timestamp.now().strftime('%Y-%m-%d')}"
    if git("tag", "-l", tag).stdout.strip(): raise Halt(f"태그 {tag}가 이미 있습니다(TAG_NAME을 바꾸세요)")
    spec_p = ["--", "paper1/01_S1", ":(exclude)paper1/01_S1/oa_raw", ":(exclude)paper1/01_S1/_backup_*"]
    print(git("status", "--short", *spec_p).stdout or "(변경 없음)")
    git("add", *spec_p, check=True)
    c = git("commit", "-m", "S1 후속 freeze: 영토·구 국가 표기 규칙, 국가 미상 복구 규칙 동결(민감도용)")
    print((c.stdout.strip() or c.stderr.strip())[:500])
    if c.returncode and "nothing to commit" not in (c.stdout + c.stderr): raise Halt("커밋 실패")
    git("tag", "-a", tag, "-m", "S1 territory/recovery freeze", check=True)
    print(f"태그 생성: {tag} | commit {git('rev-parse', '--short', 'HEAD').stdout.strip()} | 원격 반영: git push --follow-tags")

assert S0.exists() and S1.exists(), "ROOT를 확인하세요"
print(f"ROOT={ROOT} | FREEZE={FREEZE}")
try:
    spec = make_spec(); R = Recoverer(spec)
    ok = preview(R, spec)
    if FREEZE: freeze(R, spec, ok); print("\n✔ 동결 완료")
    else: print("\n(점검만 수행: 디스크에 아무것도 쓰지 않았습니다. 문제가 없으면 FREEZE=True로 바꿔 다시 실행하세요)")
except Halt as e:
    print(f"\n⛔ 중단: {e}")

ROOT=/home/yjlee/Research/Minnesota_Transport | FREEZE=False

[preview] 규칙 점검 (디스크에 쓰지 않음)
국가명 234개, 별칭 56개, 영토→모국 제외 23개, 모호어 제외 ['Georgia', 'Jordan', 'Chad', 'Niger', 'Guinea', 'Malta', 'Cuba', 'Peru', 'Oman', 'Togo', 'Samoa']
규칙 해시(예정) d4a87d357217 | pycountry 26.2.16

[A. unknown_check 표본] 판단 가능 97건 중 규칙 복구 97건 (100.0%), 복구분 정답 97/97
※ 이 표본은 규칙 설계에 참고했으므로 in-sample(낙관적)입니다. 판단은 아래 B를 보세요.

[B. 국가가 이미 있는 저자 50,000명으로 정밀도 시험] 규칙이 값을 낸 경우 49,465명(98.9%)
※ OpenAlex가 소속 2개 이상에서 국가를 만든 경우 불일치로 잡힐 수 있어 정밀도의 하한에 가깝습니다.
전체 일치율 0.984 (기준 0.95)
  티어 1: n=44,009, 일치율 0.982 (기준 0.9)
  티어 2: n=5,447, 일치율 0.996 (기준 0.9)
  티어 3: n=9, 일치율 0.889 (표본<30, 기준 미적용)
불일치 쌍 상위(규칙 -> OpenAlex): [(('HK', 'CN|HK'), 131), (('CN', 'CN|US'), 71), (('HK', 'CN'), 54), (('IT', 'BE|IT'), 33), (('MO', 'CN|MO'), 19), (('US', 'CN|US'), 14), (('GB|US', 'US'), 14), (('HK|MO', 'CN'), 10), (('CN', 'US'), 8), (('CN', 'CN|NL'), 7)]
                                                               raw_aff   rec countries
Current

In [6]:
#

In [7]:
# -*- coding: utf-8 -*-
# =====================================================================================
# S1 최종 마무리 단일 셀 (OpenAlex 호출 없음, 크레딧 소모 없음)
# 순서: 사전 점검 -> 규칙 동결(recovery, territory) -> 복구 적용 -> CHANGELOG -> oa_raw 보관본
#       -> 최종 manifest + S1_DONE.md -> commit + tag -> (선택) push -> 연도별 UNK 몫 참고 출력
# 안전장치: 규칙 해시가 점검에서 본 값(d4a87d357217)과 다르면 동결하지 않고 멈춘다.
# 여러 번 실행해도 안전하다(이미 한 단계는 건너뜀). 필요: pip install pycountry
# =====================================================================================
import re, json, html, hashlib, subprocess, tarfile, warnings
from pathlib import Path
from collections import Counter
import pandas as pd
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)

# ---------------- 설정 (여기만 수정) ----------------
ROOT = Path.home() / "Research/Minnesota_Transport"
EXPECTED_SPEC_PREFIX = "d4a87d357217"   # 두 번째 점검에서 확인한 규칙 해시(앞 12자리)
FIRST_PREVIEW_HASH = "6835d25194ae"     # 첫 점검 해시(CHANGELOG에 수정 이력으로 기록)
VALIDATE_N, SEED = 50000, 2026
CRIT_OVERALL, CRIT_TIER = 0.95, 0.90    # 결과를 보기 전에 정해 둔 기준
ARCHIVE_OA_RAW = True                   # oa_raw를 tar.gz로 보관하고 해시를 기록(커밋에는 해시만 들어감)
ARCHIVE_DIR = ROOT / "archive"          # git 추적 밖 폴더
COMMIT = True
PUSH = False                            # True면 git push --follow-tags 시도(원격이 설정돼 있어야 함)
TAG_NAME = None                         # None이면 S1-final-YYYY-MM-DD
# ----------------------------------------------------

S0, S1 = ROOT / "paper1/00_S0", ROOT / "paper1/01_S1"
DATA = ROOT / "analysis_data/final"
SPEC_FILE = "RECOVERY_RULES_PREREG.json"
hdr = lambda t: print("\n" + "=" * 90 + f"\n{t}\n" + "=" * 90)
now = lambda: pd.Timestamp.now().isoformat(timespec="seconds")
sha_b = lambda b: hashlib.sha256(b).hexdigest()
sha_f = lambda p: sha_b(Path(p).read_bytes())
rd = lambda p: pd.read_csv(p, dtype=str, encoding="utf-8-sig").fillna("")
class Halt(Exception): pass

def read_tbl(name):
    p = S1 / f"{name}.parquet"
    return pd.read_parquet(p) if p.exists() else pd.read_csv(S1 / f"{name}.csv.gz")

def changelog_once(key, line):
    cl = S1 / "CHANGELOG_S1.md"
    if not cl.exists(): cl.write_text("# CHANGELOG S1 (날짜 | 변경 | 사유 | 영향)\n\n", encoding="utf-8")
    if key in cl.read_text(encoding="utf-8"): return False
    with cl.open("a", encoding="utf-8") as f: f.write(line + "\n")
    return True

def git(*a, check=False):
    r = subprocess.run(["git", "-C", str(ROOT), *a], capture_output=True, text=True)
    if check and r.returncode: raise Halt(f"git {' '.join(a)} 실패: {r.stderr.strip()}")
    return r

# ============================================================
# 규칙 정의 (점검에서 본 것과 동일해야 함: 해시로 확인)
# ============================================================
TERRITORY = {"HK", "MO", "TW", "PR", "GU", "VI", "AS", "MP", "GI", "FO", "GL", "AW", "CW", "SX", "NC", "PF", "RE", "GP", "MQ", "GF", "YT", "KY", "BM", "XK", "PS", "EH"}
FORMER = {"SU", "YU", "CS", "DD", "AN"}
AMB = ["Georgia", "Jordan", "Chad", "Niger", "Guinea", "Malta", "Cuba", "Peru", "Oman", "Togo", "Samoa"]
ALIASES = {"USA": "US", "U.S.A.": "US", "U.S.A": "US", "United States": "US", "United States of America": "US", "New Mexico": "US",
           "New Jersey": "US", "Jersey City": "US",
           "Inner Mongolia": "CN",
           "UK": "GB", "U.K.": "GB", "U.K": "GB", "United Kingdom": "GB", "England": "GB", "Scotland": "GB", "Wales": "GB",
           "Northern Ireland": "GB", "Great Britain": "GB", "Britain": "GB", "New South Wales": "AU", "Papua New Guinea": "PG",
           "PR China": "CN", "P.R. China": "CN", "P. R. China": "CN", "People's Republic of China": "CN", "People’s Republic of China": "CN", "China": "CN",
           "Hong Kong": "HK", "Taiwan": "TW", "Macau": "MO", "Macao": "MO", "The Netherlands": "NL", "Netherlands": "NL", "Holland": "NL",
           "Russia": "RU", "Russian Federation": "RU", "South Korea": "KR", "Republic of Korea": "KR", "Korea": "KR", "North Korea": "KP",
           "Democratic People's Republic of Korea": "KP", "Democratic People’s Republic of Korea": "KP", "Iran": "IR", "Türkiye": "TR", "Turkey": "TR",
           "Czech Republic": "CZ", "Czechia": "CZ", "Vietnam": "VN", "Viet Nam": "VN", "UAE": "AE", "United Arab Emirates": "AE", "Deutschland": "DE",
           "España": "ES", "Italia": "IT", "México": "MX", "Brasil": "BR"}
PARENT = {"HK": "CN", "MO": "CN", "TW": "CN", "PR": "US", "GU": "US", "VI": "US", "AS": "US", "MP": "US", "GI": "GB", "KY": "GB", "BM": "GB",
          "FO": "DK", "GL": "DK", "AW": "NL", "CW": "NL", "SX": "NL", "NC": "FR", "PF": "FR", "RE": "FR", "GP": "FR", "MQ": "FR", "GF": "FR", "YT": "FR"}
US_ABBR = "AL AK AZ AR CA CO CT DE FL GA HI ID IL IN IA KS KY LA ME MD MA MI MN MS MO MT NE NV NH NJ NM NY NC ND OH OK OR PA RI SC SD TN TX UT VT VA WA WV WI WY DC".split()
US_NAMES = ["Alabama", "Alaska", "Arizona", "Arkansas", "California", "Colorado", "Connecticut", "Delaware", "Florida", "Hawaii", "Idaho", "Illinois", "Indiana", "Iowa",
            "Kansas", "Kentucky", "Louisiana", "Maine", "Maryland", "Massachusetts", "Michigan", "Minnesota", "Mississippi", "Missouri", "Montana", "Nebraska", "Nevada",
            "New Hampshire", "New Jersey", "New York", "North Carolina", "North Dakota", "Ohio", "Oklahoma", "Oregon", "Pennsylvania", "Rhode Island", "South Carolina",
            "South Dakota", "Tennessee", "Texas", "Utah", "Vermont", "Virginia", "Washington", "West Virginia", "Wisconsin", "Wyoming"]
CA_ABBR = "ON QC BC AB NS MB NB NL PE SK YT NT NU".split()
CA_NAMES = ["Ontario", "Quebec", "Québec", "British Columbia", "Alberta", "Nova Scotia", "Manitoba", "Saskatchewan", "New Brunswick"]

def spec_hash(spec):
    return sha_b(json.dumps({k: v for k, v in spec.items() if k not in ("sha256", "created")}, sort_keys=True, ensure_ascii=False).encode())

def make_spec():
    try: import pycountry
    except Exception: raise Halt("pycountry가 없습니다: pip install pycountry")
    names = {}
    for c in pycountry.countries:
        for nm in {c.name, getattr(c, "common_name", c.name)}:
            if len(nm) >= 4 and nm not in AMB and "," not in nm: names[nm] = c.alpha_2
    spec = dict(version=1, created=now(), pycountry=getattr(pycountry, "__version__", "unknown"), country_names=names, aliases=ALIASES, parent_drop=PARENT,
                ambiguous=AMB, us_abbr=US_ABBR, us_names=US_NAMES, ca_abbr=CA_ABBR, ca_names=CA_NAMES,
                tiers={"1": "국가명·별칭(단어 경계), 영토가 잡히면 모국 제외", "2": "미국 주 약어·주 이름(쉼표 뒤, 문자열 끝 또는 || 앞, 우편번호 허용)", "3": "캐나다 주 이름 또는 주 약어+우편번호"})
    spec["sha256"] = spec_hash(spec); return spec

class Recoverer:
    def __init__(self, spec):
        al = {**spec["country_names"], **spec["aliases"]}; self.al = {k.lower(): v for k, v in al.items()}; self.parent = spec["parent_drop"]
        alt = "|".join(re.escape(a) for a in sorted(al, key=len, reverse=True))
        self.t1 = re.compile(r"(?<![A-Za-z])(" + alt + r")(?![A-Za-z])", re.I)
        end = r"(?:[\s,]+\d{5}(?:-\d{4})?)?\s*(?=\|\||;|\.|$)"
        self.us_abbr = re.compile(r",\s*(" + "|".join(spec["us_abbr"]) + r")\b" + end)
        self.us_name = re.compile(r",\s*(" + "|".join(sorted(spec["us_names"], key=len, reverse=True)) + r")\b" + end)
        self.ca_name = re.compile(r"(?<![A-Za-z])(" + "|".join(spec["ca_names"]) + r")(?![A-Za-z])")
        self.ca_post = re.compile(r",\s*(" + "|".join(spec["ca_abbr"]) + r")\b[\s,]+[A-Z]\d[A-Z]\s?\d[A-Z]\d")
    def __call__(self, aff):
        s = html.unescape(aff or "").replace("#TAB#", " ").strip()
        if not s: return [], 0
        cs = {self.al[m.lower()] for m in self.t1.findall(s)}
        if cs:
            for c in list(cs):
                if self.parent.get(c) in cs: cs.discard(self.parent[c])
            return sorted(cs), 1
        if self.us_abbr.search(s) or self.us_name.search(s): return ["US"], 2
        if self.ca_name.search(s) or self.ca_post.search(s): return ["CA"], 3
        return [], 0

# ============================================================
# 0) 사전 점검 (디스크 쓰기 없음)
# ============================================================
def precheck():
    hdr("[0] 사전 점검")
    if not (S0.exists() and S1.exists()): raise Halt("ROOT를 확인하세요")
    if git("rev-parse", "--show-toplevel").returncode: raise Halt("git 저장소가 아닙니다")
    if not git("config", "user.name").stdout.strip() or not git("config", "user.email").stdout.strip(): raise Halt("git user.name/user.email을 먼저 설정하세요")
    spec = make_spec(); print(f"규칙 해시 {spec['sha256'][:12]} (기대 {EXPECTED_SPEC_PREFIX}) | pycountry {spec['pycountry']}")
    old_p = S1 / SPEC_FILE
    if old_p.exists():
        old = json.loads(old_p.read_text(encoding="utf-8"))
        if old["sha256"] != spec["sha256"]: raise Halt(f"이미 동결된 규칙({old['sha256'][:12]})과 현재 규칙({spec['sha256'][:12]})이 다릅니다")
        print("이미 동결된 규칙과 동일: 재사용")
    elif not spec["sha256"].startswith(EXPECTED_SPEC_PREFIX):
        raise Halt(f"규칙 해시가 점검 때 본 값과 다릅니다({spec['sha256'][:12]}). 복사 오류이거나 pycountry 버전 차이일 수 있습니다. 다르면 동결하지 말고 점검부터 다시 하세요")
    fl = S0 / "FREEZE_LOG.csv"
    if fl.exists():
        L = rd(fl); hit = L[L.sha256.str.startswith("f16e76b4")] if "sha256" in L.columns else []
        print("FREEZE_LOG에 보정 PREREG(f16e76b4) 기록:", "있음" if len(hit) else "없음(참고: git 커밋 90416c0에는 포함되어 있음. 논문 부록에 커밋 해시를 적으면 충분)")
    R = Recoverer(spec)
    A = read_tbl("authorship_level"); A["countries"] = A.countries.fillna(""); A["raw_aff"] = A.raw_aff.fillna("")
    K = A[(A.countries != "") & (A.raw_aff != "")]; K = K.sample(min(VALIDATE_N, len(K)), random_state=SEED).copy()
    out = [R(a) for a in K.raw_aff]; K["rec"] = ["|".join(c) for c, _ in out]; K["tier"] = [t for _, t in out]
    got = K[K.tier > 0].copy(); got["exact"] = [set(r.split("|")) == set(k.split("|")) for r, k in zip(got.rec, got.countries)]
    tot = got.exact.mean(); print(f"정밀도 시험 {len(got):,}명 | 전체 일치율 {tot:.3f} (기준 {CRIT_OVERALL})")
    ok = tot >= CRIT_OVERALL
    for t, g in got.groupby("tier"):
        a = g.exact.mean(); judged = len(g) >= 30; print(f"  티어 {t}: n={len(g):,}, {a:.3f}" + ("" if judged else " (n<30, 기준 미적용)"))
        if judged and a < CRIT_TIER: ok = False
    if not ok: raise Halt("정밀도 기준 미달: 동결하지 않습니다")
    return spec, R

# ============================================================
# 1) 동결 + territory 문서 + 복구 적용 + CHANGELOG
# ============================================================
def write_territory_doc():
    txt = ["# 영토·구 국가 표기 규칙 (S1)", f"- 작성: {now()}  (S7 결과 열람 전)", "",
           "## 규칙", "1. 국가는 OpenAlex가 부여한 `country_code`를 그대로 쓴다. 영토(HK, TW, PR 등)는 모국과 합치지 않고 별도 국가로 둔다.",
           "2. 영토 코드에는 `is_territory=True` 열을 붙여 민감도 분석에서 분리할 수 있게 한다.",
           "3. 구 국가 코드(SU, YU, CS, DD, AN)는 `ZZ_former`로 묶고 승계국에 배분하지 않는다.",
           "4. 다중 소속 저자는 소속 국가에 균등 분할한다. 소속이 없는 몫은 `UNK`로 유지한다.",
           "5. 소속 국가는 국적이 아니다.",
           "6. 국가 미상 복구(recovery) 규칙에서도 영토 코드가 잡히면 모국 코드는 제외한다(예: 'Hong Kong, China' -> HK). 복구는 민감도 분석에서만 쓰고 주 분석 테이블은 바꾸지 않는다.",
           "7. 소득군은 OGHIST 산정 달력연도 기준(income_tv)을 주로 쓰고, 회계연도 기준은 wb_income_history_long.csv로 민감도. 결측은 채우지 않는다(ET 2024 원본 빈칸, RE는 OGHIST에 없음).",
           "", "## 영토 코드 목록", ", ".join(sorted(TERRITORY)), "", "## 구 국가 코드 목록", ", ".join(sorted(FORMER)), ""]
    try:
        P = read_tbl("country_authorship"); P["ws"] = P.w * P.samp_w; tot = P.ws.sum()
        terr = P[P.is_territory].groupby("country").ws.sum().sort_values(ascending=False) / tot
        fm = P[P.country == "ZZ_former"].ws.sum() / tot
        txt += ["## 현재 자료의 규모 (기술 통계, 가설 검정 아님)", f"- 영토 합계 몫: {terr.sum():.4f}", f"- 구 국가(ZZ_former) 몫: {fm:.5f}",
                "- 영토별 몫: " + ", ".join(f"{c} {v:.4f}" for c, v in terr.head(8).items()), ""]
        print(f"영토 합계 {terr.sum():.4f} | ZZ_former {fm:.5f}")
    except Exception as e: print("※ 규모 절 생략:", e)
    (S1 / "TERRITORY_RULES.md").write_text("\n".join(txt), encoding="utf-8"); print("저장: TERRITORY_RULES.md")

def freeze_rules(spec, R):
    hdr("[1] 규칙 동결 + 복구 적용 + CHANGELOG")
    p = S1 / SPEC_FILE
    if not p.exists():
        p.write_text(json.dumps(spec, ensure_ascii=False, indent=1), encoding="utf-8"); print(f"규칙 동결 기록: {SPEC_FILE} (sha256 {spec['sha256'][:12]})")
    write_territory_doc()
    A = read_tbl("authorship_level"); A["countries"] = A.countries.fillna(""); A["raw_aff"] = A.raw_aff.fillna("")
    T = A[(A.countries == "") & (A.raw_aff != "")][["pid", "rank", "raw_aff"]].copy()
    out = [R(a) for a in T.raw_aff]; T["countries_rec"] = ["|".join(c) for c, _ in out]; T["rec_tier"] = [t for _, t in out]
    T.to_csv(S1 / "authorship_level_recovered.csv.gz", index=False, compression="gzip")
    n, k = len(T), int((T.rec_tier > 0).sum())
    print(f"소속 문자열은 있으나 국가 없음 {n:,}건 중 복구 {k:,}건 ({k / max(n, 1):.1%}) | 티어별 {T.rec_tier.value_counts().sort_index().to_dict()}")
    print("복구 국가 상위:", T[T.countries_rec != ""].countries_rec.value_counts().head(8).to_dict())
    if n and k / n < 0.80: print("⚠ 복구 비율이 unknown_check 표본(97%)보다 크게 낮습니다: 표본의 대표성이 낮았을 수 있으니 논문 한계 절에 이 비율을 쓰세요")
    changelog_once("territory_doc·recovery 동결",
        f"- {now()} | territory_doc·recovery 동결: 영토·구 국가 표기 규칙(TERRITORY_RULES.md)과 국가 미상 규칙 복구 규칙({SPEC_FILE}, sha256 {spec['sha256'][:12]})을 S7 결과 열람 전에 기록. "
        f"복구는 민감도용이며 주 분석 테이블은 변경하지 않음. 규칙은 unknown_check 100건(in-sample)을 참고해 설계했고, 동결 전에 국가가 이미 있는 저자 50,000명 표본으로 정밀도를 점검함(기준 전체 0.95, 티어 0.90은 결과 열람 전 설정). "
        f"점검 중 발견한 오매칭 유형(New Jersey->JE, Inner Mongolia->MN, 'Hong Kong, China'의 CN 중복)을 별칭과 영토-모국 제외 규칙으로 반영함: 첫 점검(해시 {FIRST_PREVIEW_HASH}, 일치율 0.983) 후 Inner Mongolia 별칭을 추가해 {spec['sha256'][:12]}(0.984)로 수정함(점검 후 수정). "
        f"점검 표본은 OpenAlex가 이미 국가를 붙인 저자라 실제 복구 대상(OpenAlex가 못 붙인 저자)의 정밀도와 다를 수 있음. raw_aff는 저자당 2개까지만 저장되어 일치율은 하한에 가까움. 복구 {k:,}/{n:,}건 | 규칙 사전 기록 | S2 연도별 미상 몫 점검, S7 민감도")

# ============================================================
# 2) oa_raw 보관본
# ============================================================
def archive_raw():
    hdr("[2] oa_raw 보관본")
    rec = S1 / "S1_OA_RAW_ARCHIVE.txt"
    if not ARCHIVE_OA_RAW: print("건너뜀(ARCHIVE_OA_RAW=False)"); return
    if rec.exists(): print("이미 기록됨:\n" + rec.read_text(encoding="utf-8")); return
    raw = S1 / "oa_raw"
    if not raw.exists(): raise Halt("oa_raw가 없습니다")
    ARCHIVE_DIR.mkdir(exist_ok=True)
    tgz = ARCHIVE_DIR / f"oa_raw_{pd.Timestamp.now().strftime('%Y%m%d')}.tar.gz"
    with tarfile.open(tgz, "w:gz") as t: t.add(raw, arcname="oa_raw")
    files = {f.name: sha_f(f) for f in sorted(raw.glob("*")) if f.is_file()}
    rec.write_text(f"archive: {tgz}\narchive_sha256: {sha_f(tgz)}\nsize_mb: {tgz.stat().st_size / 1e6:.1f}\ncreated: {now()}\n"
                   + "".join(f"{k}: {v}\n" for k, v in files.items()), encoding="utf-8")
    print(f"보관본 {tgz} ({tgz.stat().st_size / 1e6:.0f}MB), 해시를 S1_OA_RAW_ARCHIVE.txt에 기록(커밋됨). 이 파일은 외장 디스크나 클라우드에도 복사해 두세요")

# ============================================================
# 3) 최종 manifest + S1_DONE.md
# ============================================================
def final_manifest():
    hdr("[3] 최종 manifest + S1_DONE.md")
    keys = ["manual_verify.csv", "unknown_check.csv", "target_lens_review.csv", "manual_source_check.csv", "SOURCE_CHECK_SUMMARY.json", "target_lens_precision.json",
            "validation_report.md", "coverage_by_year.csv", "coverage_low_cells.csv", "WINDOW_DRAFT.json", "paper_status.csv", "wb_income_history.csv",
            "wb_income_history_long.csv", "S1_검수판정_기준_및_확인기록.md", "TERRITORY_RULES.md", SPEC_FILE, "authorship_level_recovered.csv.gz",
            "S1_OA_RAW_ARCHIVE.txt", "CHANGELOG_S1.md"]
    need = ["validation_report.md", "SOURCE_CHECK_SUMMARY.json", "target_lens_precision.json", "TERRITORY_RULES.md", SPEC_FILE, "CHANGELOG_S1.md"]
    miss = [f for f in need if not (S1 / f).exists()]
    if miss: raise Halt(f"핵심 산출물이 없습니다: {miss}")
    M = [dict(file=f"01_S1/{f}", sha256=sha_f(S1 / f)) for f in keys if (S1 / f).exists()]
    M += [dict(file=f"00_S0/{f}", sha256=sha_f(S0 / f)) for f in ["PREREG.json", "EVENTS_FROZEN.csv", "EXCLUDE_PIDS_FROZEN.csv"] if (S0 / f).exists()]
    pd.DataFrame(M).assign(at=now()).to_csv(S1 / "S1_FREEZE_MANIFEST_final.csv", index=False)
    print(f"S1_FREEZE_MANIFEST_final.csv {len(M)}개 해시 기록 | 없는 선택 산출물: {[f for f in keys if not (S1 / f).exists()]}")
    sc = json.loads((S1 / "SOURCE_CHECK_SUMMARY.json").read_text(encoding="utf-8")); lens = json.loads((S1 / "target_lens_precision.json").read_text(encoding="utf-8"))
    done = ["# S1 완료 기록", f"- 작성: {now()}", "",
            "## 통과 판정", "- 분석 창 2007(재계산 2007, 엄격 규칙 2009은 민감도): PASS",
            "- manual_verify 200건(텍스트 기준) 200/200, Wilson [0.981, 1.000]: PASS",
            f"- 원문 대조 {sc['n_checked']}건 중 일치 {sc['n_agree']}, Wilson {sc['wilson']}: 보강 증거(200건 대비 축소, 사유는 CHANGELOG)",
            f"- 대상 렌즈 정밀도 {lens['correct']}/{lens['n']}, Wilson {lens['wilson']} (재현율 미측정)", "",
            "## 동결", f"- 국가 미상 복구 규칙 해시 {json.loads((S1 / SPEC_FILE).read_text(encoding='utf-8'))['sha256'][:12]} (민감도용)",
            "- 상세 해시: S1_FREEZE_MANIFEST_final.csv, 변경 이력: CHANGELOG_S1.md", "",
            "## 한계(논문 한계 절에 반영)", "- 소속 국가는 국적이 아니며 단일 출처(OpenAlex), Lens 교차 확인 생략.",
            "- 저자 수 정렬 일치율 100%는 독립 검증이 아님(n_authors 출처 미확인).",
            "- 검수 사전 판정은 AI(Claude)가 파일 텍스트로 수행, 연구자가 검토. 기준 문서는 판정 후 evaluate 전에 작성.",
            "- 복구 규칙의 정밀도 시험 표본은 실제 복구 대상과 집단이 다름. raw_aff는 저자당 2개까지만 보관.",
            "", "## 남은 맥락 항목(본 분석을 막지 않음)", "- nauthors_check, 저널 창간연도 확인(AMAR, TR-E pending), JIF 수집, gazetteer(선택)", ""]
    (S1 / "S1_DONE.md").write_text("\n".join(done), encoding="utf-8"); print("저장: S1_DONE.md")

# ============================================================
# 4) commit + tag (+ push)
# ============================================================
def commit_tag():
    hdr("[4] commit + tag")
    if not COMMIT: print("COMMIT=False: 건너뜀"); return
    tag = TAG_NAME or f"S1-final-{pd.Timestamp.now().strftime('%Y-%m-%d')}"
    if git("tag", "-l", tag).stdout.strip(): raise Halt(f"태그 {tag}가 이미 있습니다(TAG_NAME을 바꾸세요)")
    sp = ["--", "paper1/00_S0", "paper1/01_S1", ":(exclude)paper1/01_S1/oa_raw", ":(exclude)paper1/01_S1/_backup_*"]
    print(git("status", "--short", *sp).stdout or "(변경 없음)")
    git("add", *sp, check=True)
    c = git("commit", "-m", "S1 final: 국가 미상 복구 규칙·영토 표기 규칙 동결, oa_raw 보관 해시, 최종 manifest")
    print((c.stdout.strip() or c.stderr.strip())[:500])
    if c.returncode and "nothing to commit" not in (c.stdout + c.stderr): raise Halt("커밋 실패")
    git("tag", "-a", tag, "-m", "S1 final", check=True)
    print(f"태그 생성: {tag} | commit {git('rev-parse', '--short', 'HEAD').stdout.strip()}")
    print("이전 태그:", git("tag", "-l", "S0S1-*", "S1-*").stdout.split())
    if PUSH:
        r = git("push", "--follow-tags"); print("push:", (r.stdout + r.stderr).strip()[:400])
    else: print("원격 반영은 터미널에서: git push --follow-tags")

# ============================================================
# 5) 참고: 연도별 국가 미상 몫(복구 전후) -> S2 입력
# ============================================================
def unk_by_year():
    hdr("[5] 참고: 연도별 저자 단위 국가 미상 몫(복구 전/후)")
    try:
        A = read_tbl("authorship_level"); A["countries"] = A.countries.fillna("")
        Rr = pd.read_csv(S1 / "authorship_level_recovered.csv.gz", dtype={"pid": str}); A["pid"] = A.pid.astype(str)
        F = pd.read_csv(DATA / "F1_papers.csv", low_memory=False, usecols=["pid", "year"]); F["pid"] = F.pid.astype(str)
        A = A.merge(F, on="pid").merge(Rr[["pid", "rank", "rec_tier"]], on=["pid", "rank"], how="left")
        A["unk0"] = A.countries == ""; A["unk1"] = A.unk0 & (A.rec_tier.fillna(0) == 0)
        T = A.groupby("year")[["unk0", "unk1"]].mean().round(3); T.columns = ["복구 전", "복구 후"]
        T.to_csv(S1 / "unk_share_by_year.csv"); print(T.to_string()); print("저장: unk_share_by_year.csv (다음 커밋에 포함)")
    except Exception as e: print("※ 생략:", e)

print(f"ROOT={ROOT}")
try:
    spec, R = precheck()
    freeze_rules(spec, R)
    archive_raw()
    final_manifest()
    commit_tag()
    unk_by_year()
    print("\n✔ S1 최종 마무리 완료")
except Halt as e:
    print(f"\n⛔ 중단: {e}\n(여기까지의 파일 변경은 유지됩니다. 문제를 고친 뒤 같은 셀을 다시 실행하세요)")

ROOT=/home/yjlee/Research/Minnesota_Transport

[0] 사전 점검
규칙 해시 d4a87d357217 (기대 d4a87d357217) | pycountry 26.2.16
FREEZE_LOG에 보정 PREREG(f16e76b4) 기록: 없음(참고: git 커밋 90416c0에는 포함되어 있음. 논문 부록에 커밋 해시를 적으면 충분)
정밀도 시험 49,465명 | 전체 일치율 0.984 (기준 0.95)
  티어 1: n=44,009, 0.982
  티어 2: n=5,447, 0.996
  티어 3: n=9, 0.889 (n<30, 기준 미적용)

[1] 규칙 동결 + 복구 적용 + CHANGELOG
규칙 동결 기록: RECOVERY_RULES_PREREG.json (sha256 d4a87d357217)
영토 합계 0.0332 | ZZ_former 0.00000
저장: TERRITORY_RULES.md
소속 문자열은 있으나 국가 없음 13,362건 중 복구 12,854건 (96.2%) | 티어별 {0: 508, 1: 8918, 2: 3936}
복구 국가 상위: {'US': 5226, 'CN': 2494, 'FR': 605, 'CA': 473, 'DE': 387, 'GB': 343, 'AU': 322, 'IN': 257}

[2] oa_raw 보관본
보관본 /home/yjlee/Research/Minnesota_Transport/archive/oa_raw_20261013.tar.gz (41MB), 해시를 S1_OA_RAW_ARCHIVE.txt에 기록(커밋됨). 이 파일은 외장 디스크나 클라우드에도 복사해 두세요

[3] 최종 manifest + S1_DONE.md
S1_FREEZE_MANIFEST_final.csv 22개 해시 기록 | 없는 선택 산출물: []
저장: S1_DONE.md

[4] commit + tag
 M paper1/01_S1/CHANGELOG_S1.md
?? paper1/01_S1/RECOVERY_RULES_P

In [8]:
#